# x50 データの再帰予測（目的変数のラグ・移動平均を使う Y_cat ごとのモデル）

- **作成者**: ImoZukushi
- **作成日**: 2026-11-22
- **目的**: `015_regression_x50.ipynb`（Y_cat ごとの個別モデル・特徴量の選択・予測値の補正）をもとに、
  **目的変数のラグと移動平均を特徴量に加え、再帰予測（1日ずつ予測し、予測値を次の日以降のラグに使う）** を行う。
- **015 との違い**
  - **ラグ・移動平均:** Y_cat ごとに、目的変数の `CONFIG.LAGS` 日前の値（`Y_lag_1` など）と、前日までの
    `CONFIG.ROLLING_WINDOWS` 日間の平均（`Y_ma_7` など）を特徴量に加える。
  - **再帰予測:** 学習では実測値のラグを使い、テスト期間（と CV の検証期間）では、実測値を使わずに、予測値をラグに使いながら
    1日ずつ予測する。
  - **`irregular_flg` が 1 の日はラグで欠損（NaN）にする:** 学習期間の実測値も、予測期間の予測値も、`irregular_flg` が
    1 の日の値は NaN に上書きしてからラグ・移動平均を作る（通常と違う値を、後の日の予測に使わない）。
  - **CV の既定を時系列 CV にした:** テストと同じく、学習期間より後の長い期間を再帰予測して評価するため
    （`CONFIG.CV_METHOD`。GroupKFold も選べる）。
- 015 と同じこと
  - Y_cat ごとに、その Y_cat の行だけで CV・学習・テスト予測を行う（Y_cat は特徴量に使わない）。メモリを節約した学習。
  - Y_cat ごとの特徴量の選択（ラグ・移動平均も `CONFIG.EXCLUDE_COLS_BY_CAT` で除ける）、モデルの選択、スケーリングと逆変換、
    `irregular_flg`（特徴量）・`eliminate_flg`（1 の行は学習に使わない。
    ただし、目的変数の実測値はラグ・移動平均に使う）の扱い、CV の結果・SHAP・残差解析。
  - 後処理（Y_cat ごとの線形回帰による補正、上下限のクリップ、`irregular_flg` が 1 の行を 0 にする）。後処理は再帰予測が
    すべて終わった後にかける（後処理の後の値はラグに使わない）。
- **入力**（ダミー。`scripts/generate_dummy_x50.py` で作成）
  - 学習: `data/synthetic/x50/dummy_x50_train.csv`（列 `date`, `Y_cat`, `irregular_flg`, `eliminate_flg`, `Y`, `X1`〜`X50`）
  - テスト: `data/synthetic/x50/dummy_x50_test.csv`（列 `date`, `Y_cat`, `irregular_flg`, `X1`〜`X50`。目的変数なし）
- **出力**
  - 前処理済みの CSV: `data/processed/x50_regression_016/`（テストのラグ・移動平均は、予測しながら作るので欠損のまま）
  - `outputs/experiments/x50_regression_016_{モデル名}/{実行日時}/`（CV スコア・予測・図・SHAP・残差解析。
    Y_cat ごとの CV スコア・OOF 予測などは `categories/{Y_cat}/`、保存したモデルは `models/{Y_cat}/`、
    Y_cat ごとの特徴量は `features_by_category.csv`、補正の式・MAE・補正後の OOF 予測は `correction/`、
    テストの再帰予測に使ったラグ・移動平均と、何日先かによる誤差は `recursion/`）
  - テスト予測: `outputs/predictions/x50_regression_016_{モデル名}_{実行日時}.csv`
    （列 `ID`, `date`, `Y_cat`, `pred`。テストデータの全行。`pred` は逆変換・補正・後処理の後の値）
  - MLflow（`mlruns/`）、ログファイル `outputs/logs/`
- **用語**: 標準化は `(x - 平均) / 標準偏差`、正規化は `(x - 最小値) / (最大値 - 最小値)`（min-max）。
  OOF 予測は、その行を学習に使っていない fold のモデルの予測（再帰予測）。1期先予測は、前日までの実測値が分かるとして
  ラグを作った予測（再帰予測と比べるためのもの）。

In [ ]:
class CONFIG:
    """このノートブックの設定（ファイル名・列名・定数・パラメータ）。ここだけを変えれば条件を切り替えられる。"""

    # --- 入力・出力（data/ と outputs/ からの相対パス） ---
    RAW_TRAIN_PATH = "synthetic/x50/dummy_x50_train.csv"  # 学習データ
    RAW_TEST_PATH = "synthetic/x50/dummy_x50_test.csv"  # テストデータ（目的変数の値なし）
    FEATURES_DIR = "processed/x50_regression_016"  # 前処理済みの CSV の出力先（data/ の下）
    PREDICTIONS_DIR = "predictions"  # テスト予測の CSV の出力先（outputs/ の下）

    # --- 列 ---
    DATE_COL = "date"  # 日付
    CAT_COL = "Y_cat"  # 目的変数のカテゴリ（Y1〜Y10）
    TARGET_COL = "Y"  # 目的変数の値
    IRREGULAR_COL = (
        "irregular_flg"  # 学習・テスト。int 型の 0/1。特徴量に使い、テストで1の行は予測を0にする
    )
    ELIMINATE_COL = "eliminate_flg"  # 学習だけ。int 型の 0/1。1の行は学習に使わない（ラグには使う）
    GROUP_COL = "year_month"  # GroupKFold のグループ（年月。例: 2022-01）
    ID_COL = "ID"  # テスト予測の CSV の連番の列
    PRED_COL = "pred"  # 予測値の列
    ACTUAL_COL = "actual"  # 正解の列

    # --- 使うデータ ---
    TARGET_CATS = None  # 使う目的変数のカテゴリ（None なら学習データの全カテゴリ）
    # 使う説明変数（None なら全部。例: ["X1", "X2", "X11"]）。
    # CSV を読み込むときに、この列だけを読む。
    # irregular_flg はこれとは別に、必ず特徴量に入れる
    FEATURE_COLS = None
    # Y_cat ごとの使う説明変数（Y_cat → 説明変数のリスト）。
    # ここに無い Y_cat は FEATURE_COLS を使う。
    # FEATURE_COLS に無い説明変数も指定できる。空のリストなら説明変数を使わない
    # （日付の特徴量と irregular_flg だけ）。例: {"Y1": ["X1", "X2", "X11"], "Y2": ["X5", "X8"]}
    FEATURE_COLS_BY_CAT = {}
    # Y_cat ごとの除く特徴量（Y_cat → 特徴量のリスト）。上で決めた特徴量から除く。
    # 説明変数のほか、日付の特徴量（date_year など）・ラグ（Y_lag_1 など）・移動平均（Y_ma_7 など）
    # も指定できる。
    # 例: {"Y1": ["X3"], "Y3": ["date_year", "date_elapsed_days"]}
    # irregular_flg は除けない（裾の重み付け・補正・後処理に使うため、必ず特徴量に入れる）
    EXCLUDE_COLS_BY_CAT = {}
    # カテゴリとして扱う説明変数（None なら自動: 学習・テストのどちらかで数値として読めない列）。
    # 数値のコード（例: 店舗番号）をカテゴリとして扱うときはリストで指定する（例: ["X3", "X7"]）。
    # どちらも CSV の文字のまま読み、モデルに渡す前に CAT_ENCODING（系列ID と同じ）でエンコードする
    CATEGORICAL_COLS = None

    # --- 目的変数のラグ・移動平均（再帰予測） ---
    # ラグ: 何日前の値を使うか（1以上。空のリストなら使わない）。列名は Y_lag_{日数}
    LAGS = [1, 2, 3, 7, 14, 28]
    # 移動平均: 前日までの何日間の平均を使うか（空のリストなら使わない）。列名は Y_ma_{日数}
    ROLLING_WINDOWS = [7, 14, 28]
    # 移動平均を計算するのに必要な、期間内の値のある日数（欠損の日を除いて平均する）。
    # 期間の日数より大きければ、期間の日数として扱う
    ROLLING_MIN_PERIODS = 1
    # eliminate_flg が 1 の行の目的変数の実測値を、ラグ・移動平均に使うか（行は学習に使わない）。
    # False なら欠損（NaN）にする
    # （irregular_flg が 1 の日は、この設定によらず、学習・予測期間とも必ず欠損にする）
    LAG_USE_ELIMINATED = True

    # --- 前処理 ---
    # 目的変数の変換: None（行わない）/ "standardize"（Y_cat ごとの標準化）/
    # "normalize"（Y_cat ごとの正規化）/ "log"（対数）。
    # 標準化・正規化の中心・分母は、各 fold の学習側の行から求める
    # （eliminate_flg が 1 の行は使わない。irregular_flg が 1 の行は使う）。
    # 予測・CV の結果・SHAP は元の尺度に戻して出す（対数の SHAP だけはモデルの尺度のまま）。
    # 対数は y + TARGET_LOG_OFFSET > 0 の値にしか使えない
    # （irregular_flg が 1 の行が負の値のデータでは使えない）
    TARGET_TRANSFORM = None
    TARGET_LOG_OFFSET = 0.0
    # 説明変数のスケーリング（Y_cat ごと）:
    # None（行わない）/ "standardize"（標準化）/ "normalize"（正規化）
    FEATURE_SCALING = None
    # スケーリングする列（例: ["X1", "X2", "date_elapsed_days"]）。
    # 説明変数・日付の特徴量・irregular_flg・ラグ・移動平均から選べる（カテゴリの説明変数は不可）。
    # None なら irregular_flg とカテゴリ以外の説明変数の全部（ラグ・移動平均は含まない）
    FEATURE_SCALING_COLS = None
    DATE_COMPONENTS = ["year", "month", "day", "weekday", "week"]  # 日付から取り出す成分

    # --- 後処理1: Y_cat ごとの線形回帰による予測値の補正 ---
    # 補正する Y_cat: None（全部）/ "improved"（補正で CV の MAE が小さくなる Y_cat だけ）/
    # Y_cat のリスト（例: ["Y1", "Y3"]。空のリストなら補正しない）。
    # 補正の式は、OOF 予測と実績値（irregular_flg が 1 の行は除く）から Y_cat ごとに求める
    CORRECTION_CATS = "improved"

    # --- 後処理2: 予測値の上下限（Y_cat ごと。None は制限なし） ---
    DEFAULT_TARGET_BOUNDS = (0.0, None)  # (下限, 上限)。TARGET_BOUNDS に無いカテゴリに使う
    TARGET_BOUNDS = {}  # カテゴリごとの (下限, 上限)。例: {"Y1": (0.0, 150.0), "Y2": (10.0, None)}

    # --- 予測が中央に寄るのを抑える対策（既定はどれも行わない） ---
    # 目的変数の分布の裾の行への重み付け: (下側の分位, 上側の分位)。None なら重みを付けない。
    # 例 (0.1, 0.9): カテゴリごとに、学習側の 10% 点以下と 90% 点以上の行の重みを TAIL_WEIGHT にする
    # （片側だけなら (None, 0.9) のように書く。irregular_flg が 1 の行は対象にしない）
    TAIL_QUANTILES = None
    TAIL_WEIGHT = 3.0  # 裾の行の重み（それ以外の行は 1）
    # 目的関数（損失）の共通の名前。各モデルの指定への変換は src/modeling/models/ が行う。
    # "l2"（二乗誤差。平均に寄りやすい）/ "l1"（絶対誤差。中央値に寄る）/ "huber"（Huber損失）/
    # "quantile"（分位点損失）/ "poisson"（ポアソン損失。負の目的変数には使えない）。
    # RandomForest・ExtraTrees は huber・quantile に、線形モデルは l2 以外に対応しない
    OBJECTIVE = "l2"
    # huber の閾値（誤差がこれを超えると絶対誤差になる）・quantile の分位。
    # None なら共通の既定（huber 1.0・quantile 0.5）。huber の閾値は目的変数の尺度に合わせる
    # （誤差の典型的な大きさが目安。小さすぎると絶対誤差に近くなり、学習が遅く予測が寄る）
    OBJECTIVE_ALPHA = None
    # 木の本数の既定（early stopping のあるモデルだけ。多めにして early stopping で決める）。
    # ここに無いモデル（RandomForest・ExtraTrees）は src の既定（300）。線形モデルには木が無い
    N_ESTIMATORS = {"lightgbm": 1000, "xgboost": 1000, "catboost": 1000}
    # モデルに追加で渡すパラメータ（例: {"max_depth": 12}）。上の設定より優先する
    MODEL_PARAMS = {}
    # Optuna によるパラメータの最適化（0 なら行わない）。目的関数は元の尺度の CV の MAE
    N_TRIALS = 0
    TUNING_TIMEOUT = None  # 探索を打ち切る秒数（None なら N_TRIALS 回まで）

    # --- 学習 ---
    # モデル: 木 "lightgbm" / "xgboost" / "random_forest" / "extra_trees" / "catboost"、
    #        線形 "ols" / "lasso" / "ridge" / "elastic_net"
    MODEL = "lightgbm"
    # カテゴリの説明変数のエンコーディング（Y_cat は Y_cat ごとのモデルなので特徴量に使わない）:
    # "auto"（モデルの推奨。線形はワンホット、木は整数化）/
    # "onehot"（ワンホット）/ "ordinal"（整数化）
    CAT_ENCODING = "auto"
    # ワンホットにするカテゴリの説明変数の、列ごとの最大の列数（None なら全カテゴリ）。超えた分は
    # 少ないカテゴリから "{列名}_infrequent_sklearn" の1列にまとめる。系列ID には使わない
    ONEHOT_MAX_CATEGORIES = None
    DETAIL_TARGET = "Y1"  # 詳しい図（誤差曲線・係数など）を表示するカテゴリ
    # CV: "timeseries"（日付で区切る時系列CV。テストと同じく、学習期間より後を再帰予測する）または
    # "group"（GroupKFold。検証の月の分だけ再帰予測する）
    CV_METHOD = "timeseries"
    N_SPLITS = 5  # GroupKFold の分割数（グループ = 年月）
    CUTOFFS = ["2023-07-01", "2024-01-01"]  # 時系列CVの検証期間の開始日
    TEST_PREDICTION = "refit_full"  # "refit_full"（全学習データで学習し直す）または "fold_mean"
    USE_TRACKING = True  # MLflow に記録するか
    OUTPUT_ROOT = None  # 実験の出力のルート（None なら outputs/experiments）
    LOG_LEVEL = "WARNING"  # 画面に出すログのレベル（ファイル outputs/logs/ には INFO 以上を残す）

    # --- メモリの節約 ---
    # 説明変数（浮動小数の列）の型。"float32" にすると説明変数の表のメモリが float64 の半分になる
    # （値の有効桁は約7桁）。None なら読み込んだ型のまま。目的変数は float64 のまま
    FEATURE_DTYPE = "float32"
    # SHAP を計算する行数の上限（Y_cat ごと）。
    # 計算が重いモデル（RandomForest など）は src の上限も使う
    SHAP_MAX_SAMPLES_PER_CAT = 300
    # テスト予測に使ったモデルを joblib のファイルに保存するか（メモリにはモデルを残さないので、
    # 後でモデルを使うときは True にする）。保存先は実験の出力の models/{Y_cat}/
    SAVE_MODELS = False

    # --- 選べる値（変えない） ---
    MODELS = (
        "lightgbm",
        "xgboost",
        "random_forest",
        "extra_trees",
        "catboost",
        "ols",
        "lasso",
        "ridge",
        "elastic_net",
    )
    CAT_ENCODINGS = ("auto", "onehot", "ordinal")
    TARGET_TRANSFORMS = (None, "standardize", "normalize", "log")
    OBJECTIVES = ("l2", "l1", "huber", "quantile", "poisson")  # src の modeling.models.base と同じ
    FEATURE_SCALINGS = ("standardize", "normalize")  # src の GROUP_SCALING_METHODS と同じ
    MLFLOW_EXPERIMENT = "x50_forecast"

In [ ]:
# papermill で CONFIG の一部を変えるときに使う（このセルに parameters タグを付けている）。
# 例: uv run papermill ... \
#     -y "{CONFIG_OVERRIDES: {MODEL: ridge, TARGET_TRANSFORM: standardize, FEATURE_DTYPE: null}}"
CONFIG_OVERRIDES = {}

In [ ]:
unknown = [name for name in CONFIG_OVERRIDES if not hasattr(CONFIG, name)]
if unknown:
    raise ValueError(f"CONFIG に無い名前です: {unknown}")
for name, value in CONFIG_OVERRIDES.items():
    setattr(CONFIG, name, value)
if CONFIG_OVERRIDES:
    print(f"CONFIG を変更: {CONFIG_OVERRIDES}")

## 設定（`CONFIG`）の使い方

2セル目の `CONFIG` の値を書き換えて実行する。papermill で実行するときは、`CONFIG_OVERRIDES`（3セル目。parameters
タグ付き）に変えたい名前と値を渡す。

```bash
uv run papermill notebook/016_regression_x50.ipynb out.ipynb --cwd notebook \
    -y "{CONFIG_OVERRIDES: {MODEL: lasso, N_TRIALS: 30, LAGS: [7, 14]}}"
```

| 名前 | 選べる値 | 既定 |
|---|---|---|
| `MODEL` | 木: `"lightgbm"`、`"xgboost"`、`"random_forest"`、`"extra_trees"`、`"catboost"`。線形: `"ols"`、`"lasso"`、`"ridge"`、`"elastic_net"` | `"lightgbm"` |
| `CAT_ENCODING` | カテゴリの説明変数のエンコーディング。`"auto"`（線形はワンホット、木は整数化）、`"onehot"`、`"ordinal"`。Y_cat は特徴量に使わない | `"auto"` |
| `ONEHOT_MAX_CATEGORIES` | カテゴリの説明変数のワンホットの列ごとの最大の列数。`None` なら全カテゴリ | `None` |
| `TARGET_TRANSFORM` | `None`（行わない）、`"standardize"`（Y_cat ごとの標準化）、`"normalize"`（Y_cat ごとの正規化）、`"log"`（対数） | `None` |
| `FEATURE_COLS` | 使う説明変数のリスト。`None` なら全部（`irregular_flg` は必ず使う） | `None` |
| `FEATURE_COLS_BY_CAT` | Y_cat ごとの使う説明変数（Y_cat → リスト）。無い Y_cat は `FEATURE_COLS`。例 `{"Y1": ["X1", "X2"]}` | `{}` |
| `EXCLUDE_COLS_BY_CAT` | Y_cat ごとの除く特徴量（Y_cat → リスト）。日付の特徴量・ラグ・移動平均も指定できる。例 `{"Y3": ["date_year"]}` | `{}` |
| `LAGS` | 目的変数のラグ（何日前の値か。1以上の整数のリスト）。`[]` なら使わない | `[1, 2, 3, 7, 14, 28]` |
| `ROLLING_WINDOWS` | 目的変数の移動平均（前日までの何日間か）。`[]` なら使わない | `[7, 14, 28]` |
| `ROLLING_MIN_PERIODS` | 移動平均の計算に必要な、値のある日数 | `1` |
| `LAG_USE_ELIMINATED` | `eliminate_flg` が 1 の行の目的変数の実測値をラグ・移動平均に使うか（行そのものは学習に使わない） | `True` |
| `FEATURE_SCALING` | `None`（行わない）、`"standardize"`（Y_cat ごとの標準化）、`"normalize"`（Y_cat ごとの正規化） | `None` |
| `CATEGORICAL_COLS` | カテゴリとして扱う説明変数のリスト。`None` なら自動（数値として読めない列） | `None` |
| `FEATURE_SCALING_COLS` | スケーリングする列のリスト（説明変数・日付の特徴量・`irregular_flg`・ラグ・移動平均から選ぶ。カテゴリは不可）。`None` なら `irregular_flg`・カテゴリ以外の説明変数の全部 | `None` |
| `CV_METHOD` | `"timeseries"`（日付で区切る時系列CV）、`"group"`（GroupKFold） | `"timeseries"` |
| `CORRECTION_CATS` | 線形回帰で予測値を補正する Y_cat。`None`（全部）、`"improved"`（CV の MAE が小さくなる Y_cat だけ）、リスト（`[]` なら補正しない） | `"improved"` |
| `DEFAULT_TARGET_BOUNDS` | 後処理の (下限, 上限)。`None` は制限なし | `(0.0, None)` |
| `TARGET_BOUNDS` | カテゴリごとの (下限, 上限)。例 `{"Y1": (0.0, 150.0)}` | `{}` |
| `TAIL_QUANTILES` | 裾の行に重みを付ける分位 (下側, 上側)。例 `(0.1, 0.9)`。`None` なら付けない | `None` |
| `TAIL_WEIGHT` | 裾の行の重み | `3.0` |
| `OBJECTIVE` | 目的関数の共通の名前（`"l2"`、`"l1"`、`"huber"`、`"quantile"`、`"poisson"`） | `"l2"` |
| `OBJECTIVE_ALPHA` | huber の閾値・quantile の分位。`None` なら huber 1.0・quantile 0.5 | `None` |
| `N_ESTIMATORS` | 早期終了のあるモデルの木の本数（モデル名 → 本数） | 3モデルとも `1000` |
| `MODEL_PARAMS` | モデルに追加で渡すパラメータ（例 `{"max_depth": 12}`）。他の設定より優先する | `{}` |
| `N_TRIALS` | Optuna の試行回数（Y_cat ごと。0 なら最適化しない） | `0` |
| `FEATURE_DTYPE` | 説明変数（浮動小数の列）の型。`"float32"` でメモリが半分。`None` なら読み込んだ型のまま | `"float32"` |
| `SHAP_MAX_SAMPLES_PER_CAT` | SHAP を計算する行数の上限（Y_cat ごと） | `300` |
| `SAVE_MODELS` | テスト予測に使ったモデルをファイルに保存するか | `False` |

**ラグ・移動平均と再帰予測（016）**

- **日付で数える:** ラグ・移動平均は、Y_cat ごとに日付で数える（`Y_lag_7` はちょうど7日前の値）。行の無い日・目的変数が
  欠損の日は NaN。移動平均は前日までの値だけを使い（当日の値は含まない）、NaN の日を除いて平均する。
- **NaN にする日:** `irregular_flg` が 1 の日は、学習期間の実測値も予測期間の予測値も NaN に上書きしてからラグ・移動平均を
  作る。`irregular_flg` が欠損の日は、1 と分からないので上書きしない。`eliminate_flg` が 1 の行は、
  学習には使わないが、目的変数の実測値はラグ・移動平均に使う（`LAG_USE_ELIMINATED = False` なら NaN にする）。
- **学習:** 前日までの実測値から作ったラグ・移動平均で学習する。`irregular_flg` が 1 の行も、015 と同じく学習に使う
  （NaN にするのは、後の日のラグに使うときだけ）。
- **予測:** 日付の順に、前日までの履歴（実測値と、すでに予測した日の予測値）からラグ・移動平均を作って予測し、予測値を履歴に
  加える（`irregular_flg` が 1 の日は NaN を加える）。履歴に加えるのは、元の尺度に逆変換したモデルの予測で、補正・クリップ・
  0 にする処理の前の値。
- **CV:** 検証期間の実測値は、学習のラグにも検証のラグにも使わない。検証期間は再帰予測する。early stopping だけは、
  学習の途中では再帰予測ができないので、検証期間の実測値のラグで作った1期先予測の損失で止める。
- **まとめて予測する日数:** 使うラグの最小が 7 で移動平均を使わなければ、予測値を待たずに 7 日分をまとめて予測できるので速い
  （結果は1日ずつの予測と同じ）。ラグ 1 か移動平均を使うときは1日ずつ予測する。
- `LAGS = []`・`ROLLING_WINDOWS = []` にすると、015 と同じ（再帰しない）予測になる。

**Y_cat ごとの特徴量の選び方（015 と同じ）**

```python
FEATURE_COLS = ["X1", "X2", "X3"]  # 指定の無い Y_cat が使う説明変数
FEATURE_COLS_BY_CAT = {"Y1": ["X1", "X11", "X12"], "Y2": []}  # Y2 は日付と irregular_flg だけ
EXCLUDE_COLS_BY_CAT = {"Y1": ["date_year"], "Y4": ["X3"]}  # 除く特徴量（日付の特徴量も可）
```

- 各 Y_cat の特徴量は「日付の特徴量 + `irregular_flg` + 説明変数（`FEATURE_COLS_BY_CAT`、無ければ `FEATURE_COLS`）
  + ラグ・移動平均」から `EXCLUDE_COLS_BY_CAT` を除いたもの。例えば `EXCLUDE_COLS_BY_CAT = {"Y1": ["Y_lag_1"]}`。使う特徴量は「6. 実験設定と Y_cat ごとの特徴量」で確かめられる。
- データに無い Y_cat・列の名前を書くと、エラーで止まる（書き間違いに気付けるように）。
- `FEATURE_SCALING_COLS`・`CATEGORICAL_COLS` は全 Y_cat 共通で指定し、各 Y_cat では、その Y_cat が使う列だけに適用する。
- CSV から読むのは、どれかの Y_cat が使う説明変数だけ。

**予測が中央に寄るとき:** 二乗誤差で学習すると、予測は平均に寄りやすい。裾の行に重みを付ける（`TAIL_QUANTILES`）、
目的関数を変える（`OBJECTIVE`）、パラメータを最適化する（`N_TRIALS`）を試し、「8. CV の結果」の
「予測の広がり」で効果を確かめる。

**目的関数の共通の名前と、各モデルでの指定**（`src/modeling/models/` の `objective_params`）

| `OBJECTIVE` | LightGBM | XGBoost | CatBoost | RandomForest・ExtraTrees | 線形モデル |
|---|---|---|---|---|---|
| `"l2"` | `regression` | `reg:squarederror` | `RMSE` | `criterion="squared_error"` | 二乗誤差（追加の指定なし） |
| `"l1"` | `regression_l1` | `reg:absoluteerror` | `MAE` | `criterion="absolute_error"`（とても遅い） | 対応しない |
| `"huber"` | `huber`（`alpha`） | `reg:pseudohubererror`（`huber_slope`） | `Huber:delta=…` | 対応しない | 対応しない |
| `"quantile"` | `quantile`（`alpha`） | `reg:quantileerror`（`quantile_alpha`） | `Quantile:alpha=…` | 対応しない | 対応しない |
| `"poisson"` | `poisson` | `count:poisson` | `Poisson` | `criterion="poisson"` | 対応しない |

**モデルごとの違い**

| モデル | early stopping・学習の推移 | 木の本数 | Optuna の探索空間 |
|---|---|---|---|
| LightGBM | あり | `N_ESTIMATORS`（1000）から early stopping | 学習率・葉の数・行と列の抽出・正則化 |
| XGBoost | あり | 同上 | 学習率・深さ・行と列の抽出・正則化 |
| CatBoost | あり | 同上（CatBoost の `iterations`） | 学習率・深さ・L2 正則化・`random_strength`・`bagging_temperature` |
| RandomForest | なし | 300（`MODEL_PARAMS` の `n_estimators` で変える） | 深さ・葉の最小行数・列の割合・行の割合 |
| ExtraTrees | なし | 同上 | 深さ・葉の最小行数・列の割合 |
| OLS | なし | （木は無い） | なし（最適化しない） |
| LASSO | なし | （木は無い） | 正則化の強さ `alpha` |
| Ridge | なし | （木は無い） | 正則化の強さ `alpha` |
| ElasticNet | なし | （木は無い） | 正則化の強さ `alpha`・L1 の割合 `l1_ratio` |

- huber の閾値（`OBJECTIVE_ALPHA`）は目的変数の尺度に合わせて指定する。既定の 1.0 は、値が 100 前後のこのダミーでは
  小さすぎて絶対誤差に近くなり、CV の MAE が l2 より悪くなった（LightGBM 7.6・CatBoost 8.5。l2 は 5.4・5.6）。
- XGBoost の huber は擬似Huber損失（滑らかな近似）なので、他のモデルの huber と値が少し違う。
- RandomForest・ExtraTrees の SHAP は計算が重いため、300 行に抑えて計算する（`ModelSpec.shap_max_samples`）。

**Y_cat ごとのモデル（014 と同じ）**

- Y_cat ごとに、その Y_cat の行だけで CV の分割を作り、学習・テスト予測を行う（GroupKFold の年月・時系列 CV の
  境目の日付の決まりは 013 と同じ）。Optuna を使うときは、Y_cat ごとに別のパラメータを探す（時間は Y_cat の数倍）。
- Y_cat ごとのスケーリングは、そのモデルの学習側（その Y_cat の行だけ）で求めることになる（013 と同じ計算）。
- 1つのモデルが学習する行は全体の 1/(Y_cat の数) なので、行が少ない Y_cat では、モデルが不安定になりやすい。
  全 Y_cat で共通の傾向（013 の1つのモデルが Y_cat の間で共有できた情報）は使えない。

**スケーリングと逆変換（013 と同じ）**

| 対象 | 変換（Y_cat ごと） | 中心・分母を求める行 | 逆変換 |
|---|---|---|---|
| 目的変数 | 標準化 `(y - 平均) / 標準偏差`、正規化 `(y - 最小値) / (最大値 - 最小値)` | 各 fold の学習側の行（`eliminate_flg` が 1 の行は除く。`irregular_flg` が 1 の行は含む） | 予測 `z` を `z × 分母 + 中心` で元の尺度に戻す（`TargetScaler.inverse`） |
| 説明変数 | 同上（`GroupScaler`。Pipeline の先頭） | 各 fold の学習側の全行（欠損・±inf は除いて計算） | SHAP の図・表に出す値を、スケーリング前の値に戻す |
| SHAP 値 | — | — | `SHAP 値 × 分母`、期待値は `期待値 × 分母 + 中心`（行の Y_cat と fold の値） |

- 分母が 0（定数）の Y_cat の列は分母を1にする。学習側に値が無い Y_cat の列は、学習側の全行の値を使う。
- 元の尺度の SHAP 値は、`期待値 + SHAP 値の和 = 元の尺度の予測` を満たす（「スケーリングと逆変換の確認」で確かめる）。
  ただし Y_cat ごとに尺度が違うので、全体の SHAP 重要度は尺度の大きい Y_cat の影響が大きくなる。Y_cat ごとの
  重要度の割合（系列ごとの表・図）は、尺度に左右されない。
- `TARGET_TRANSFORM = "log"` は一次式でないので、SHAP 値は元の尺度に戻せない（モデルの尺度のまま出す）。
  `OBJECTIVE = "poisson"` の木のモデルの SHAP 値も、もともと対数の尺度（予測の対数の分解）。
- 誤差曲線（学習の推移）と線形モデルの係数は、モデルの尺度（スケーリングの後）のまま。

**線形モデルについて**

- 欠損値補完（中央値）→ 標準化 → モデル、の順の Pipeline で、どれも各 fold の学習側だけで fit する。
  `FEATURE_SCALING` を指定すると、Y_cat ごとのスケーリングの後に、さらに全行で1つの標準化がかかる。
  Y_cat ごとのスケーリングで、説明変数の意味は「その系列の中で、平均より何標準偏差大きいか」に変わる。
- 線形モデルが表せるのは、説明変数の足し算（と系列ごとの水準の違い）だけ。説明変数どうしの掛け合わせ・曲線の
  関係は表せない。日付は、年・月などの数値と、月・曜日の sin・cos（周期特徴量）として入る。
- LASSO・ElasticNet は、正則化（`alpha`）が強いと係数が 0 になり、その説明変数を使わなくなる。標準化した後の
  値で正則化するので、`alpha` は説明変数の単位によらない。既定の `alpha=1.0` は目的変数の尺度によっては強すぎるので、
  `N_TRIALS` で探すとよい。
- OLS はワンホットの全列と切片が一次従属になるため、係数は一意に決まらない（予測値は一意）。
- `"poisson"` は負の目的変数に使えない（`irregular_flg` が 1 の行が負の値のデータでは、学習でエラーになる）。

## リークを防ぐための決まり

| 段階 | 決まり | どこで確かめるか |
|---|---|---|
| 行の選択 | 学習データの `eliminate_flg` が 1 の行は、読み込みの直後に除く。前処理の fit・学習のどちらにも入らない（目的変数の実測値だけは、後の日のラグ・移動平均に使う） | 4. 前処理のリークの確認 / 予測のリークの確認 |
| `irregular_flg` | 特徴量として使う（学習・テストの両方にある、予測時点で分かる値）。欠損は欠損のままモデルに渡す。テストで 1 の行も予測し、後処理で 0 にする（欠損の行は 0 にしない） | 13. 後処理2 |
| 目的変数のラグ・移動平均 | 前日までの値だけを使う。`irregular_flg` が 1 の日（実測値・予測値とも）は NaN にする。`eliminate_flg` が 1 の行の実測値は使う（CV の検証期間の分は、他の実測値と同じく使わない） | 4. 前処理のリークの確認 |
| ラグ（CV） | 検証期間の実測値は、学習側のラグにも、検証側の再帰予測のラグにも使わない（その fold の履歴から消す）。early stopping と、比較用の1期先予測だけは、検証期間の実測値のラグを使う | `fold_hidden` / 再帰予測の確認 |
| ラグ（テスト） | 学習データの最後の日までの実測値と、それより後の予測値だけから作る | `recursive_predict` / 再帰予測の確認 |
| 日付の前処理 | 経過日数の基準日は学習データだけで決める | 4. 前処理のリークの確認 |
| 説明変数のスケーリング | Y_cat ごとの平均・標準偏差（最小値・最大値）は、`modeling` の Pipeline の先頭の `GroupScaler` が、CV の各 fold の学習側だけで求める。学習に無い Y_cat は学習側の全行の値を使う | スケーリングと逆変換の確認 |
| 系列ID | Y_cat ごとに別のモデルを学習するので、Y_cat は特徴量に使わない（Pipeline の中で除く）。各 Y_cat のモデルは、その Y_cat の行だけで CV・学習・予測する | `make_config` / `run_category` |
| カテゴリの説明変数 | 整数化・ワンホットは、各 fold の学習側だけで fit する。学習側に無いカテゴリは、整数化では欠損、ワンホットでは全列 0 になる | `make_config` |
| 線形モデルの前処理 | 欠損値補完の中央値・標準化の平均と標準偏差は、各 fold の学習側だけで求める（モデルの Pipeline の中） | `src/modeling/models/linear.py` |
| 目的変数の変換 | 標準化・正規化の中心・分母は、Y_cat ごとに、各 fold の学習側（`eliminate_flg` が 1 の行は除く）だけで求める。予測は元の尺度に戻してから評価・出力する | `train_cv` / スケーリングと逆変換の確認 |
| 裾の重み付け | 重みを決める分位点は、カテゴリごとに、各 fold の学習側だけで求める | `tail_weights` |
| パラメータの最適化 | Optuna は学習データの CV だけで評価する（テストデータは使わない）。同じ CV で最終の評価もするので、CV の誤差は楽観的になりやすい | `tune_params` |
| 特徴量 | 系列ID・日付の特徴量・`irregular_flg`・説明変数・目的変数のラグと移動平均（`CONFIG` で指定したもの）だけ。目的変数・日付・グループ・`eliminate_flg` の列が入っていないこと。Y_cat ごとに選んでも、選べるのはこれらの列だけ | `category_features_map` / `prepare` |
| CV（GroupKFold） | グループは年月。同じ月の行は学習と検証に分かれない。検証の月より後の月も学習に使う | `prepare` |
| CV（時系列） | `CONFIG.CUTOFFS` の日付で区切る。学習は境目より前だけ | `prepare` |
| 予測 | 各行の予測は、その行の特徴量（と、その行の Y_cat のモデル）と、前の日までの履歴だけから行う（後の日の影響を受けない）。モデルを残さないので、確認は学習の途中で行う | 予測のリークの確認 |
| 予測値の補正 | 補正の式は、学習データの OOF 予測と実績値だけから求める（テストデータは使わない）。補正後の MAE は、評価する fold 以外の行で求めた式を使って計算する | 12. 後処理1 |
| 後処理 | 上下限のクリップと 0 にする処理は、モデルの予測の後に行う（モデル・CV のスコアには含めない） | 13. 後処理2 |
| 欠損 | 欠損は埋めない。目的変数が欠損の行は学習・評価から除く。特徴量（説明変数・`irregular_flg`）の欠損はモデルに任せる（スケーリングも欠損を残したまま行う） | `model_frames` |

**注意:** GroupKFold の CV は、未来の月も学習に使い、再帰予測も検証の月の分（続いた月ならその長さ）しか続かないため、
1年先まで再帰するテストより楽観的になりやすい。既定の時系列 CV では、テストと同じく学習期間より後だけを再帰予測する。

In [ ]:
import contextlib
import ctypes
import dataclasses
import datetime as dt
import gc
import hashlib
import logging
import shutil
import sys
import time
import warnings
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

# pyproject.toml を目印にリポジトリルートを探し、src/ をimportパスに追加する
# （scripts/ の関数は import せず、このノートブック内で定義する）
REPO_ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

# ipywidgets が無い環境で tqdm が出す警告（進捗バーの表示だけの問題）は出さない
warnings.filterwarnings("ignore", message="IProgress not found")

# src/ 配下のモジュール（上でパスを追加した後にimportする必要がある）
from evaluation.curves import TrainingHistoryDisplay  # noqa: E402
from evaluation.influence import InfluenceDisplay, compute_influence  # noqa: E402
from evaluation.residuals import (  # noqa: E402
    QQPlotDisplay,
    ResidualCorrelogramDisplay,
    ResidualDistributionDisplay,
    ResidualPlotDisplay,
    plot_residuals_by_group,
    residual_summary,
)
from feature_engineering.datetime_features import (  # noqa: E402
    CyclicalFeaturesEncoder,
    DatetimeFeaturesExtractor,
    ElapsedTimeTransformer,
)
from feature_engineering.scaling import GroupScaler  # noqa: E402
from modeling.config import ExperimentConfig  # noqa: E402
from modeling.cv import folds_to_ids  # noqa: E402
from modeling.dataset import Dataset  # noqa: E402
from modeling.evaluation import EVALUATION_DIR  # noqa: E402
from modeling.experiment import (  # noqa: E402
    make_output_dir,
    prepare_dataset,
    save_cv_outputs,
)
from modeling.explain import (  # noqa: E402
    ShapResult,
    compute_oof_shap,
    sample_oof_rows,
    save_shap_outputs,
)
from modeling.metrics import get_metric  # noqa: E402
from modeling.models import get_model_spec  # noqa: E402
from modeling.pipeline import MODEL_STEP  # noqa: E402
from modeling.tasks import Task, predict  # noqa: E402
from modeling.tracking import MLflowTracker, NullTracker, Tracker  # noqa: E402
from modeling.trainer import CVResult, fit_pipeline  # noqa: E402
from util.csv_io import detect_encoding  # noqa: E402
from util.paths import data_dir, ensure_parent_dir, get_repo_root, outputs_dir  # noqa: E402
from util.plotting import add_caption, ensure_japanese_font  # noqa: E402

# pandas の表示で列を省略しすぎないようにする
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

## 関数の定義

このノートブックは `scripts/` を import せず、使う関数をここで定義する。ログ・図の部品は 004、要約は 008、
学習（`run_training` など）は 015 と同じ流れの関数に、ラグ・移動平均と再帰予測を加えたもの。列名・定数はすべて `CONFIG` から読む。

### 定数と結果の入れ物

In [ ]:
logger = logging.getLogger("regression_x50_016")


class LeakageError(ValueError):
    """入力データにリークのおそれがある場合のエラー。"""


class TargetScaler:
    """目的変数の変換（行わない・Y_cat ごとの標準化・正規化・対数）と、元の尺度に戻す処理。

    標準化は `(y - 平均) / 標準偏差`（ddof=0）、正規化は `(y - 最小値) / (最大値 - 最小値)`。
    中心（平均・最小値）と分母（標準偏差・最大値 − 最小値）は、`fit` に渡した行（CV の学習側。
    `eliminate_flg` が 1 の行は読み込みの直後に除いてある）から Y_cat ごとに求める。
    欠損・±inf は計算に使わない。分母が 0 の Y_cat は分母を1にする。

    Args:
        kind: `None` / `"standardize"` / `"normalize"` / `"log"`。
        offset: 対数の変換で `log(y + offset)` にする値。

    Raises:
        ValueError: 未対応の変換の場合。
    """

    AFFINE = ("standardize", "normalize")  # 一次式の変換（SHAP 値も元の尺度に戻せる）

    def __init__(self, kind: str | None, offset: float = 0.0) -> None:
        if kind not in CONFIG.TARGET_TRANSFORMS:
            raise ValueError(
                f"CONFIG.TARGET_TRANSFORM は {CONFIG.TARGET_TRANSFORMS} から選んでください: {kind}"
            )
        self.kind = kind
        self.offset = offset

    def fit(self, y: np.ndarray, categories: np.ndarray) -> "TargetScaler":
        """変換に必要な値を学習する。

        Args:
            y: 目的変数（元の尺度）。
            categories: 各行の Y_cat。

        Raises:
            ValueError: 対数を取れない値（`y + offset <= 0`）がある場合。
        """
        if self.kind == "log" and (y + self.offset <= 0).any():
            raise ValueError(
                "対数を取れない目的変数があります（irregular_flg が 1 の行が負の値など）。"
                "CONFIG.TARGET_TRANSFORM を None・standardize・normalize のどれかにするか、"
                "CONFIG.TARGET_LOG_OFFSET を指定してください"
            )
        if self.kind in self.AFFINE:
            finite = np.isfinite(y)
            center, scale = self._statistics(y[finite], categories[finite])
            self.center_ = center
            # 値が1種類しかない Y_cat は、0除算にならないよう分母を1にする
            self.scale_ = scale.where(scale > 0, 1.0)
        return self

    def _statistics(self, y: np.ndarray, categories: np.ndarray) -> tuple[pd.Series, pd.Series]:
        """Y_cat ごとの (中心, 分母)。"""
        grouped = pd.Series(y).groupby(categories)
        if self.kind == "standardize":
            return grouped.mean(), grouped.std(ddof=0)
        return grouped.min(), grouped.max() - grouped.min()

    def affine(self, categories: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """各行の (中心, 分母)。元の尺度 = モデルの尺度 × 分母 + 中心。

        変換しない場合は (0, 1)。

        Raises:
            ValueError: 一次式でない変換（対数）の場合。
        """
        if self.kind is None:
            return np.zeros(len(categories)), np.ones(len(categories))
        if self.kind not in self.AFFINE:
            raise ValueError(f"{self.kind} は一次式の変換ではありません")
        cats = pd.Series(categories)
        return cats.map(self.center_).to_numpy(), cats.map(self.scale_).to_numpy()

    def transform(self, y: np.ndarray, categories: np.ndarray) -> np.ndarray:
        """モデルが学習する尺度に変換する。"""
        if self.kind == "log":
            return np.log(y + self.offset)
        center, scale = self.affine(categories)
        return (y - center) / scale

    def inverse(self, z: np.ndarray, categories: np.ndarray) -> np.ndarray:
        """モデルの尺度の値を、元の尺度に戻す（後処理の逆変換）。"""
        if self.kind == "log":
            return np.exp(z) - self.offset
        center, scale = self.affine(categories)
        return z * scale + center


@dataclass(frozen=True)
class LagSpec:
    """目的変数のラグ・移動平均の定義と計算。

    系列は「ある日から1日刻みで並んだ値（無い日・使わない日は NaN）」の配列で持ち、
    行の日付を配列の位置に直して計算する。どの特徴量も、前日までの値だけを使う。

    Attributes:
        lags: ラグ（何日前の値か）。
        windows: 移動平均（前日までの何日間か）。
        min_periods: 移動平均の計算に必要な、値のある日数。
    """

    lags: tuple[int, ...]
    windows: tuple[int, ...]
    min_periods: int

    @classmethod
    def from_config(cls) -> "LagSpec":
        """`CONFIG` の指定から作る。

        Raises:
            ValueError: ラグ・移動平均の日数が 1 以上の整数でない場合や、重複がある場合。
        """
        for name in ("LAGS", "ROLLING_WINDOWS"):
            days = getattr(CONFIG, name)
            if not isinstance(days, (list, tuple)) or any(
                not isinstance(d, int) or isinstance(d, bool) or d < 1 for d in days
            ):
                raise ValueError(f"CONFIG.{name} は 1 以上の整数のリストにしてください: {days}")
            if len(set(days)) != len(days):
                raise ValueError(f"CONFIG.{name} に重複があります: {days}")
        if CONFIG.ROLLING_MIN_PERIODS < 1:
            raise ValueError("CONFIG.ROLLING_MIN_PERIODS は 1 以上にしてください")
        return cls(tuple(CONFIG.LAGS), tuple(CONFIG.ROLLING_WINDOWS), CONFIG.ROLLING_MIN_PERIODS)

    def select(self, columns: list[str]) -> "LagSpec":
        """`columns` にある特徴量だけにする（Y_cat ごとに特徴量を除いたとき）。"""
        return dataclasses.replace(
            self,
            lags=tuple(k for k in self.lags if f"{CONFIG.TARGET_COL}_lag_{k}" in columns),
            windows=tuple(w for w in self.windows if f"{CONFIG.TARGET_COL}_ma_{w}" in columns),
        )

    @property
    def names(self) -> list[str]:
        """特徴量の列名。"""
        target = CONFIG.TARGET_COL
        return [f"{target}_lag_{k}" for k in self.lags] + [f"{target}_ma_{w}" for w in self.windows]

    @property
    def block(self) -> int:
        """予測値を待たずにまとめて予測できる日数（使うラグの最小。移動平均は前日から使うので1）。"""
        return min([*self.lags, *([1] if self.windows else [])], default=sys.maxsize // 2)

    def compute(self, values: np.ndarray, positions: np.ndarray) -> np.ndarray:
        """`positions` の日のラグ・移動平均を計算する（列の順は `names`）。

        Args:
            values: 1日刻みの系列（NaN を含む）。
            positions: 計算する日の、`values` の中の位置。
        """
        columns = []
        for k in self.lags:
            columns.append(self._take(values, positions - k))
        for w in self.windows:
            window = self._take(values, positions[:, None] - np.arange(1, w + 1))
            observed = np.isfinite(window)
            count = observed.sum(axis=1)
            total = np.where(observed, window, 0.0).sum(axis=1)
            enough = count >= min(self.min_periods, w)
            columns.append(np.where(enough, total / np.maximum(count, 1), np.nan))
        if not columns:
            return np.empty((len(positions), 0))
        return np.column_stack(columns)

    @staticmethod
    def _take(values: np.ndarray, positions: np.ndarray) -> np.ndarray:
        """`positions` の値（系列の最初より前は NaN）。"""
        return np.where(positions >= 0, values[np.maximum(positions, 0)], np.nan)


@dataclass
class CategoryResult:
    """1つの Y_cat のモデルの学習・予測の結果。

    メモリを節約するため、モデル・学習用データそのものは持たず、表・図に使う小さい結果だけを持つ。

    Attributes:
        category: Y_cat。
        rows: 学習の表（`RegressionRun.frame`）の中の、この Y_cat の行の位置。
        test_rows: テストの表の中の、この Y_cat の行の位置。
        oof_pred: OOF 予測（再帰予測。元の尺度。`rows` の順）。
        onestep_pred: 1期先予測の OOF（前日までの実測値からラグを作った、比較用の予測）。
        horizon: 再帰予測で何日先の予測か（履歴の実測値の最後の日から数える）。
            検証に入らない行は 0。
        fold_ids: 各行を検証した fold（`rows` の順。どの fold にも入らない行は -1）。
        test_pred: テスト予測（再帰予測。元の尺度。`test_rows` の順）。
        test_lags: テストの再帰予測に使ったラグ・移動平均（`fold_mean` なら fold の平均）。
        oof_scores: OOF 全体の MAE・RMSE（元の尺度）。
        params: 学習に使ったモデルのパラメータ。
        fold_table: fold ごとの行数・期間・誤差の表。
        shap: OOF SHAP（元の尺度。`rows` は学習の表の中の位置）。計算できなければ None。
        influence: Leverage・Cook の距離（全行）。`row` は学習の表の中の位置。
        influence_irregular0: 同じく `irregular_flg` が 0 の行だけで計算したもの。
        n_params: Leverage の計算に使った特徴量のランク（fold の平均）。
        histories: fold ごとの学習の推移（記録の無いモデルは空）。
        coefficients: 線形モデルの係数（線形モデルでなければ None）。
        checks: 学習の途中で行った確認の結果（確認の名前 → 説明）。
        tuning_trials: Optuna の試行の一覧（最適化しなければ None）。
        seconds: 学習にかかった秒数。
        memory_mb: この Y_cat を終えてモデルを捨てた後のプロセスのメモリ
            （MB。psutil が無ければ NaN）。
        peak_memory_mb: この Y_cat の学習中に測ったプロセスのメモリの最大（MB）。
    """

    category: str
    rows: np.ndarray
    test_rows: np.ndarray
    oof_pred: np.ndarray
    onestep_pred: np.ndarray
    horizon: np.ndarray
    fold_ids: np.ndarray
    test_pred: np.ndarray
    test_lags: pd.DataFrame
    oof_scores: dict[str, float]
    params: dict
    fold_table: pd.DataFrame
    shap: ShapResult | None
    influence: pd.DataFrame | None
    influence_irregular0: pd.DataFrame | None
    n_params: int
    histories: list
    coefficients: pd.DataFrame | None
    checks: dict[str, str]
    tuning_trials: pd.DataFrame | None
    seconds: float
    memory_mb: float
    peak_memory_mb: float


@dataclass
class RegressionRun:
    """全 Y_cat の学習・予測の結果（Y_cat ごとの結果をまとめたもの）。

    Attributes:
        categories: 目的変数の種類（Y_cat）の並び。
        frame: 学習に使った行の表（全 Y_cat。行の位置が `CategoryResult.rows` に対応する）。
        output_dir: 実験の出力ディレクトリ。
        run_id: MLflow の run ID（記録しなければ None）。
        predictions: テスト予測（long 型。列 `date`, `Y_cat`, `pred`）。
        oof: 学習期間の OOF 予測（long 型。列 `date`, `Y_cat`, `actual`, `pred`, `fold`,
            `pred_onestep`, `horizon`）。`fold` は、その行の Y_cat のモデルの CV の fold。
            `pred` は再帰予測、`pred_onestep` は1期先予測、`horizon` は何日先の予測か。
        test_lags: テストの再帰予測に使ったラグ・移動平均（列 `date`, `Y_cat`, `irregular_flg`,
            `pred`（モデルの予測）, ラグ・移動平均。行の順は `predictions` と同じ）。
        results: Y_cat → その Y_cat の結果。
        categorical_cols: カテゴリの説明変数。
        features: Y_cat → その Y_cat のモデルの特徴量（エンコードの前の列）。
        shap: 全 Y_cat の OOF SHAP（元の尺度）。
        tuning_trials: Optuna の試行の一覧（列 `Y_cat` 付き。最適化しなければ None）。
    """

    categories: list[str]
    frame: pd.DataFrame
    output_dir: Path
    run_id: str | None
    predictions: pd.DataFrame
    oof: pd.DataFrame
    test_lags: pd.DataFrame
    results: dict[str, CategoryResult]
    categorical_cols: list[str]
    features: dict[str, list[str]]
    shap: ShapResult | None = None
    tuning_trials: pd.DataFrame | None = None

### ログ（004 と同じ）

In [ ]:
def setup_logging(
    log_path: Path, level: int = logging.INFO, console_level: int | None = None
) -> Path:
    """画面とファイルの両方にログを出す設定をする（同じ設定を2回呼んでも重複しない）。

    Args:
        log_path: ログファイルのパス（UTF-8）。
        level: ログの出力レベル（ファイル）。
        console_level: 画面に出すログのレベル（Noneなら `level` と同じ）。画面への出力量を減らしたい
            とき（小さいインスタンスの Jupyter など）は `logging.WARNING` にする。ファイルには
            `level` 以上のログが残る。

    Returns:
        ログファイルのパス。
    """
    ensure_parent_dir(log_path)
    console_level = level if console_level is None else console_level
    root = logging.getLogger()
    root.setLevel(min(level, console_level))
    for handler in list(root.handlers):
        if getattr(handler, "_x50", False):
            root.removeHandler(handler)
            handler.close()
    formatter = logging.Formatter("%(asctime)s %(levelname)s %(name)s: %(message)s")
    file_handler = logging.FileHandler(log_path, encoding="utf-8")
    file_handler.setLevel(level)
    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(console_level)
    handlers: list[logging.Handler] = [file_handler, console_handler]
    for handler in handlers:
        handler.setFormatter(formatter)
        handler._x50 = True  # type: ignore[attr-defined]
        root.addHandler(handler)
    # ライブラリの警告もログに残す（細かい情報ログは出さない）
    logging.captureWarnings(True)
    for name in ("matplotlib", "PIL", "shap", "urllib3", "alembic", "mlflow"):
        logging.getLogger(name).setLevel(logging.WARNING)
    return log_path


def shown(path: Path) -> str:
    """ログ・画面に出すパス（リポジトリ内ならルートからの相対パス。個人のフォルダ名を残さない）。"""
    resolved = Path(path).resolve()
    root = get_repo_root()
    return resolved.relative_to(root).as_posix() if resolved.is_relative_to(root) else str(path)


def file_digest(path: Path) -> str:
    """ファイルの SHA-256（先頭12文字）。どのデータで学習したかをログに残すために使う。"""
    return hashlib.sha256(path.read_bytes()).hexdigest()[:12]


def make_tracker(enabled: bool) -> Tracker:
    """MLflow に記録する（`enabled=False` なら何も記録しない）。"""
    return MLflowTracker(CONFIG.MLFLOW_EXPERIMENT) if enabled else NullTracker()

### 読み込みとモデルに渡す表

In [ ]:
def check_flag(frame: pd.DataFrame, column: str, allow_missing: bool = False) -> None:
    """フラグの列が 0 か 1 だけをとることを確かめる。

    Args:
        frame: 表。
        column: フラグの列。
        allow_missing: 欠損（NaN）を許すか。特徴量として使うフラグ（`irregular_flg`）は欠損を許し、
            欠損のままモデルに渡す（欠損があると CSV からは浮動小数の型で読まれる）。

    Raises:
        ValueError: 数値でない、0・1 以外の値がある、または欠損を許さない列に欠損がある場合。
    """
    values = frame[column]
    observed = values.dropna() if allow_missing else values
    if not pd.api.types.is_numeric_dtype(values) or not observed.isin([0, 1]).all():
        kind = "0・1・欠損" if allow_missing else "int 型の 0 か 1"
        raise ValueError(f"{column} は {kind} にしてください（{values.dtype}）")


def csv_encoding(path: Path) -> str:
    """CSV の文字コード（UTF-8 は BOM 付きにも対応する utf-8-sig にする）。"""
    encoding = detect_encoding(path)
    return "utf-8-sig" if encoding == "utf-8" else encoding


def read_columns(path: Path, feature_cols: list[str] | None, train: bool) -> pd.DataFrame:
    """CSV から、使う列だけを読み込む（メモリを節約し、使う説明変数をここで決める）。

    学習データは `irregular_flg`・`eliminate_flg`・目的変数を、
    テストデータは `irregular_flg` を読む。

    Args:
        path: CSV のパス。
        feature_cols: 使う説明変数（None なら全部）。
        train: 学習データか。

    Raises:
        ValueError: 必要な列が CSV に無い、またはフラグが int 型の 0/1 でない場合。
    """
    encoding = csv_encoding(path)
    header = list(pd.read_csv(path, nrows=0, encoding=encoding).columns)
    flags = [CONFIG.IRREGULAR_COL, CONFIG.ELIMINATE_COL] if train else [CONFIG.IRREGULAR_COL]
    meta = [CONFIG.DATE_COL, CONFIG.CAT_COL, *flags, *([CONFIG.TARGET_COL] if train else [])]
    excluded = {
        CONFIG.DATE_COL,
        CONFIG.CAT_COL,
        CONFIG.IRREGULAR_COL,
        CONFIG.ELIMINATE_COL,
        CONFIG.TARGET_COL,
    }
    features = [c for c in header if c not in excluded] if feature_cols is None else feature_cols
    missing = [c for c in [*meta, *features] if c not in header]
    if missing:
        raise ValueError(f"{path.name} に列がありません: {missing}")
    frame = pd.read_csv(path, usecols=[*meta, *features], encoding=encoding)
    frame[CONFIG.DATE_COL] = pd.to_datetime(frame[CONFIG.DATE_COL], format="%Y-%m-%d")
    # irregular_flg は特徴量なので欠損を許す
    # （欠損のままモデルに渡す。後処理で 0 にするのは 1 の行だけ）
    check_flag(frame, CONFIG.IRREGULAR_COL, allow_missing=True)
    frame[CONFIG.IRREGULAR_COL] = frame[CONFIG.IRREGULAR_COL].astype("float64")
    if train:
        check_flag(frame, CONFIG.ELIMINATE_COL)
    return frame[[*meta, *features]]


def check_category_lists(name: str, categories: list[str]) -> dict[str, list[str]]:
    """Y_cat ごとのリストの設定（`CONFIG` の名前）を取り出し、書き方を確かめる。

    値が None の Y_cat は、指定が無いものとして除く。

    Args:
        name: `CONFIG` の名前（`FEATURE_COLS_BY_CAT` など）。
        categories: 学習データにある全部の Y_cat（`CONFIG.TARGET_CATS` で絞り込む前）。

    Raises:
        ValueError: データに無い Y_cat がある場合や、値が列の名前のリストでない場合。
    """
    setting = {c: v for c, v in (getattr(CONFIG, name) or {}).items() if v is not None}
    unknown = sorted(set(setting) - set(categories))
    if unknown:
        raise ValueError(f"CONFIG.{name} に、学習データに無い Y_cat があります: {unknown}")
    for category, columns in setting.items():
        # 文字列を書くと1文字ずつの列の名前になってしまうので、リストだけを受け付ける
        if not isinstance(columns, (list, tuple)):
            raise ValueError(f"CONFIG.{name} の {category} は列の名前のリストにしてください")
    return {category: list(columns) for category, columns in setting.items()}


def columns_to_read(all_categories: list[str], used_categories: list[str]) -> list[str] | None:
    """CSV から読む説明変数（どれかの Y_cat が使う列）を決める。None なら全部。

    Args:
        all_categories: 学習データにある全部の Y_cat。
        used_categories: 学習・予測する Y_cat（`CONFIG.TARGET_CATS`）。

    Raises:
        ValueError: 説明変数でない列（`irregular_flg`・目的変数など）を指定した場合。
    """
    by_cat = check_category_lists("FEATURE_COLS_BY_CAT", all_categories)
    reserved = {
        CONFIG.DATE_COL,
        CONFIG.CAT_COL,
        CONFIG.IRREGULAR_COL,
        CONFIG.ELIMINATE_COL,
        CONFIG.TARGET_COL,
    }
    columns: list[str] = []
    read_all = False
    for category in used_categories:
        selected = by_cat.get(category, CONFIG.FEATURE_COLS)
        if selected is None:
            read_all = True  # 全部の説明変数を使う Y_cat がある
        else:
            columns += selected
    invalid = sorted(set(columns) & reserved)
    if invalid:
        raise ValueError(
            f"説明変数でない列は指定できません（irregular_flg は必ず特徴量に入る）: {invalid}"
        )
    # 重複を除く（初めて出た順）
    return None if read_all else list(dict.fromkeys(columns))


def category_features_map(
    all_categories: list[str],
    used_categories: list[str],
    date_cols: list[str],
    x_cols: list[str],
    lag_cols: list[str],
) -> dict[str, list[str]]:
    """Y_cat ごとの、モデルの特徴量（日付・`irregular_flg`・説明変数・ラグと移動平均）を決める。

    説明変数は `CONFIG.FEATURE_COLS_BY_CAT`（無い Y_cat は `CONFIG.FEATURE_COLS`。
    None なら全部）から、`CONFIG.EXCLUDE_COLS_BY_CAT` の列を除く。
    並びは表の列の順（指定した順によらない）。

    Args:
        all_categories: 学習データにある全部の Y_cat。
        used_categories: 学習・予測する Y_cat。
        date_cols: 日付の特徴量。
        x_cols: 読み込んだ説明変数（`irregular_flg` 以外）。
        lag_cols: 目的変数のラグ・移動平均（どの Y_cat でも使う。除くときは
            `CONFIG.EXCLUDE_COLS_BY_CAT`）。

    Returns:
        Y_cat → 特徴量のリスト。

    Raises:
        ValueError: 使う説明変数・除く特徴量に、無い列や `irregular_flg` がある場合や、
            説明変数とラグ・移動平均の列名が重なる場合。
    """
    duplicated = sorted(set(lag_cols) & {*date_cols, *x_cols})
    if duplicated:
        raise ValueError(f"ラグ・移動平均と同じ名前の列がデータにあります: {duplicated}")
    by_cat = check_category_lists("FEATURE_COLS_BY_CAT", all_categories)
    excluded = check_category_lists("EXCLUDE_COLS_BY_CAT", all_categories)
    every = [*date_cols, CONFIG.IRREGULAR_COL, *x_cols, *lag_cols]
    features = {}
    for category in used_categories:
        selected = by_cat.get(category, CONFIG.FEATURE_COLS)
        selected = x_cols if selected is None else selected
        drop = set(excluded.get(category, []))
        if CONFIG.IRREGULAR_COL in drop:
            raise ValueError(f"{category}: {CONFIG.IRREGULAR_COL} は除けません（必ず使う特徴量）")
        for name, unknown in (
            ("使う説明変数", sorted(set(selected) - set(x_cols))),
            ("除く特徴量（CONFIG.EXCLUDE_COLS_BY_CAT）", sorted(drop - set(every))),
        ):
            if unknown:
                raise ValueError(f"{category} の{name}に、特徴量でない列があります: {unknown}")
        usable = {*date_cols, CONFIG.IRREGULAR_COL, *selected, *lag_cols} - drop
        features[category] = [c for c in every if c in usable]
    return features


def categorical_columns(train: pd.DataFrame, test: pd.DataFrame, x_cols: list[str]) -> list[str]:
    """カテゴリとして扱う説明変数を決める（`CONFIG.CATEGORICAL_COLS`。None なら自動）。

    自動では、学習・テストのどちらかで数値として読めなかった列（文字列が混ざる列）をカテゴリにする。
    片方でだけ数値として読める列（例: テストにある値が数字だけ）も、両方カテゴリにそろえる。

    Args:
        train: 学習データ（`read_columns` の結果）。
        test: テストデータ（`read_columns` の結果）。
        x_cols: 説明変数（`irregular_flg` 以外）。

    Returns:
        カテゴリとして扱う説明変数（`x_cols` の順）。

    Raises:
        ValueError: `CONFIG.CATEGORICAL_COLS` に、使う説明変数でない列がある場合。
    """
    if CONFIG.CATEGORICAL_COLS is not None:
        unknown = sorted(set(CONFIG.CATEGORICAL_COLS) - set(x_cols))
        if unknown:
            raise ValueError(f"CONFIG.CATEGORICAL_COLS に使う説明変数でない列があります: {unknown}")
        return [c for c in x_cols if c in CONFIG.CATEGORICAL_COLS]
    numeric = pd.api.types.is_numeric_dtype
    return [c for c in x_cols if not (numeric(train[c]) and numeric(test[c]))]


def read_as_text(path: Path, frame: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """カテゴリの列を CSV から文字列として読み直して置き換える。

    数値として読むと、同じカテゴリが学習とテストで別の値になることがある（欠損のある列は `1.0`、
    無い列は `1` など）。CSV の文字のまま読み、学習とテストで同じカテゴリを同じ値にする。
    欠損は欠損のまま（整数化では欠損、ワンホットでは `{列名}_nan` の列になる）。

    Args:
        path: CSV のパス（`frame` を読んだファイル）。
        frame: `read_columns` の結果（行を絞り込む前）。
        columns: カテゴリの列。
    """
    if not columns:
        return frame
    text = pd.read_csv(
        path, usecols=columns, dtype=dict.fromkeys(columns, "str"), encoding=csv_encoding(path)
    )
    return frame.assign(**{c: text[c] for c in columns})


def category_summary(train: pd.DataFrame, test: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    """カテゴリの説明変数ごとの、カテゴリ数・欠損の行数・学習に無いカテゴリのテストの行数など。

    `数値の割合` は、欠損でない値のうち数値として読める値の割合（学習・テストを合わせる）。
    1 に近い列は、数値の列に文字（`unknown` など）が混ざってカテゴリと判定された疑いがある。
    """
    rows = []
    for c in columns:
        unseen = test[c].notna() & ~test[c].isin(train[c].dropna().unique())
        observed = pd.concat([train[c], test[c]]).dropna()
        numeric_share = pd.to_numeric(observed, errors="coerce").notna().mean()
        rows.append(
            {
                "列": c,
                "学習のカテゴリ数": train[c].nunique(),
                "テストのカテゴリ数": test[c].nunique(),
                "学習の欠損の行数": int(train[c].isna().sum()),
                "テストの欠損の行数": int(test[c].isna().sum()),
                "学習に無いカテゴリのテストの行数": int(unseen.sum()),
                "数値の割合": float(numeric_share) if len(observed) else float("nan"),
            }
        )
    return pd.DataFrame(rows).set_index("列")


def drop_eliminated(frame: pd.DataFrame) -> pd.DataFrame:
    """学習データから `eliminate_flg` が 1 の行を除き、その列も外す（学習に使わない）。"""
    return (
        frame.loc[frame[CONFIG.ELIMINATE_COL] == 0]
        .drop(columns=CONFIG.ELIMINATE_COL)
        .sort_values(CONFIG.DATE_COL, kind="stable")
        .reset_index(drop=True)
    )


def model_frames(
    train: pd.DataFrame, test: pd.DataFrame, exog_cols: list[str]
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """`prepare_dataset` に渡す学習・テストの表を作る。

    学習の表は「日付・系列ID・年月（グループ）・日付の特徴量・説明変数・ラグと移動平均・目的変数」。
    目的変数が欠損の行は学習できないため除く。テストの表のラグ・移動平均は、予測しながら作るので
    欠損のまま。
    `CONFIG.FEATURE_DTYPE` が指定されていれば、浮動小数の説明変数をその型にする（メモリの節約。
    目的変数は float64 のまま）。

    Returns:
        (学習の表, テストの表（目的変数なし）)。
    """
    columns = [CONFIG.DATE_COL, CONFIG.CAT_COL, *exog_cols]
    used = train[train[CONFIG.TARGET_COL].notna()]
    frame = used[columns].assign(
        **{
            CONFIG.GROUP_COL: used[CONFIG.DATE_COL].dt.strftime("%Y-%m"),
            CONFIG.TARGET_COL: used[CONFIG.TARGET_COL],
        }
    )
    frames = [frame.reset_index(drop=True), test[columns].reset_index(drop=True)]
    if CONFIG.FEATURE_DTYPE is not None:
        frames = [
            f.astype(
                {c: CONFIG.FEATURE_DTYPE for c in exog_cols if pd.api.types.is_float_dtype(f[c])}
            )
            for f in frames
        ]
    return frames[0], frames[1]


def target_histories(train: pd.DataFrame) -> dict[str, pd.Series]:
    """ラグ・移動平均の元になる、Y_cat ごとの目的変数の系列（1日刻み）を作る。

    次の日は NaN にする（ラグ・移動平均に使わない）。

    - `irregular_flg` が 1 の日（欠損の日は 1 と分からないので使う）。
    - `eliminate_flg` が 1 の日（`CONFIG.LAG_USE_ELIMINATED` が True なら使う）。
    - 行の無い日、目的変数が欠損・±inf の日。

    Args:
        train: 読み込んだ学習データ（`eliminate_flg` が 1 の行を除く前）。

    Returns:
        Y_cat → 系列（インデックスは、その Y_cat の最初の日から最後の日までの全部の日）。

    Raises:
        ValueError: 同じ Y_cat・日付の行が複数ある場合。
    """
    if train.duplicated([CONFIG.DATE_COL, CONFIG.CAT_COL]).any():
        raise ValueError("同じ Y_cat・日付の行が複数あるため、ラグを作れません")
    excluded = train[CONFIG.IRREGULAR_COL] == 1
    if not CONFIG.LAG_USE_ELIMINATED:
        excluded |= train[CONFIG.ELIMINATE_COL] == 1
    values = train[CONFIG.TARGET_COL].astype("float64").replace([np.inf, -np.inf], np.nan)
    table = train[[CONFIG.DATE_COL, CONFIG.CAT_COL]].assign(value=values.mask(excluded))
    histories = {}
    for category, part in table.groupby(CONFIG.CAT_COL, sort=False):
        dates = part[CONFIG.DATE_COL]
        days = pd.date_range(dates.min(), dates.max(), freq="D", unit=dates.dt.unit)
        histories[category] = part.set_index(CONFIG.DATE_COL)["value"].reindex(days)
    return histories


def day_positions(dates: pd.Series, history: pd.Series) -> np.ndarray:
    """日付を、系列の最初の日から数えた位置（0始まり）に直す。"""
    return (dates - history.index[0]).dt.days.to_numpy()


def add_lag_features(
    frame: pd.DataFrame, histories: dict[str, pd.Series], spec: LagSpec
) -> pd.DataFrame:
    """学習データに、前日までの実測値から作ったラグ・移動平均を加える。"""
    values = np.full((len(frame), len(spec.names)), np.nan)
    for category, history in histories.items():
        rows = np.flatnonzero((frame[CONFIG.CAT_COL] == category).to_numpy())
        positions = day_positions(frame[CONFIG.DATE_COL].iloc[rows], history)
        values[rows] = spec.compute(history.to_numpy(), positions)
    return with_lags(frame, values, spec.names)


def with_lags(frame: pd.DataFrame, values: np.ndarray, names: list[str]) -> pd.DataFrame:
    """ラグ・移動平均の列を `values` に置き換えた表（型は元の列に合わせる。無い列は加える）。"""
    return frame.assign(
        **{
            name: values[:, i].astype(frame[name].dtype if name in frame.columns else np.float64)
            for i, name in enumerate(names)
        }
    )

### 実験設定

In [ ]:
def category_encoding(model: str) -> str:
    """カテゴリの説明変数のエンコーディング（`"onehot"` / `"ordinal"`）を決める。

    `CONFIG.CAT_ENCODING` が `"auto"` なら、モデルの推奨（`ModelSpec.categorical_encoding`。
    線形モデルはワンホット、木のモデルは整数化）を使う。

    Raises:
        ValueError: `CONFIG.CAT_ENCODING` が未対応の値の場合。
    """
    if CONFIG.CAT_ENCODING not in CONFIG.CAT_ENCODINGS:
        raise ValueError(
            f"CONFIG.CAT_ENCODING は {CONFIG.CAT_ENCODINGS} のどれかです: {CONFIG.CAT_ENCODING}"
        )
    if CONFIG.CAT_ENCODING == "auto":
        return get_model_spec(model).categorical_encoding
    return CONFIG.CAT_ENCODING


def category_features(model: str, categorical_cols: list[str]) -> list[dict]:
    """カテゴリの説明変数をエンコードし、Y_cat の列を除く特徴量のステップ。

    Y_cat ごとに別のモデルを学習するので、Y_cat（1つのモデルの中では一定）は特徴量に使わない。
    Y_cat の列は、Y_cat ごとのスケーリング（`GroupScaler`）のグループに使うので、最後に除く。
    カテゴリの説明変数のワンホットは `X3_a` などの 0/1 の列を加え、元の列を除く
    （列数は `CONFIG.ONEHOT_MAX_CATEGORIES` で抑えられる）。
    整数化は、学習側に無いカテゴリ・欠損を欠損にする。

    Args:
        model: モデル名。
        categorical_cols: カテゴリの説明変数。
    """
    steps: list[dict] = []
    if categorical_cols and category_encoding(model) == "onehot":
        steps += [
            {
                "class": "feature_engineering.categorical.PandasOneHotEncoder",
                "params": {
                    "variables": list(categorical_cols),
                    "max_categories": CONFIG.ONEHOT_MAX_CATEGORIES,
                },
            },
            {
                "class": "modeling.pipeline.DropColumns",
                "params": {"columns": list(categorical_cols)},
            },
        ]
    elif categorical_cols:
        steps.append(
            {
                "class": "feature_engineering.categorical.PandasOrdinalEncoder",
                "params": {"variables": list(categorical_cols)},
            }
        )
    steps.append(
        {"class": "modeling.pipeline.DropColumns", "params": {"columns": [CONFIG.CAT_COL]}}
    )
    return steps


def make_config(
    model: str,
    feature_cols: list[str],
    cv_method: str = "group",
    n_splits: int = 5,
    cutoffs: list[str] | None = None,
    test_prediction: str = "refit_full",
    feature_scaling: str | None = None,
    scaling_cols: list[str] | None = None,
    objective: str = "l2",
    objective_alpha: float | None = None,
    model_params: dict | None = None,
    categorical_cols: list[str] | None = None,
) -> ExperimentConfig:
    """回帰の実験設定を作る（`ExperimentConfig` で検証する）。

    再帰予測はこのノートブックの `recursive_predict` で行うので、モデル自体は1期先を予測する通常の
    回帰として設定する（`modeling` の再帰予測は `irregular_flg` の日を NaN にできず、使わない）。

    Y_cat ごとのモデルの設定。特徴量のほかは、どの Y_cat のモデルも同じ（`category_configs`）。
    説明変数の Y_cat ごとのスケーリング（`GroupScaler`）は Pipeline の先頭に入れるので、
    CV の各 fold の学習側だけで fit される。系列ID をエンコードする前に置く
    （Y_cat の文字のままグループに使うため）。

    Args:
        model: モデル名（`CONFIG.MODELS` のどれか）。
        feature_cols: モデルの特徴量（日付の特徴量・`irregular_flg`・説明変数・ラグと移動平均）。
            スケーリングできるのも、この列だけ。
        cv_method: `group`（GroupKFold）/ `timeseries`（日付で区切る時系列CV）。
        n_splits: GroupKFold の分割数。
        cutoffs: 時系列CVの検証期間の開始日。
        test_prediction: テスト予測の作り方（`refit_full` / `fold_mean`）。
        feature_scaling: `None` / `standardize` / `normalize`（Y_cat ごと）。
        scaling_cols: スケーリングする列（None なら `feature_cols` の全部）。
        objective: 目的関数の共通の名前（`CONFIG.OBJECTIVES` のどれか）。
        objective_alpha: huber の閾値・quantile の分位（None なら共通の既定）。
        model_params: モデルに追加で渡すパラメータ（木の本数・目的関数の指定より優先する）。
        categorical_cols: カテゴリの説明変数（文字列の列）。系列ID と同じようにエンコードする。

    Raises:
        ValueError: 未対応の値や、モデルが対応しない目的関数、
            使わない説明変数・カテゴリの説明変数をスケーリングの対象に指定した場合。
    """
    if model not in CONFIG.MODELS:
        raise ValueError(f"モデルは {CONFIG.MODELS} から選んでください: {model}")
    if objective not in CONFIG.OBJECTIVES:
        raise ValueError(f"CONFIG.OBJECTIVE は {CONFIG.OBJECTIVES} から選んでください: {objective}")
    # 木の本数（early stopping のあるモデルだけ）→ 目的関数 → 追加のパラメータ の順に上書きする。
    # 目的関数の名前はモデルごとの指定に src で変換する（対応しない組み合わせは ValueError）
    params = {}
    if model in CONFIG.N_ESTIMATORS:
        params["n_estimators"] = CONFIG.N_ESTIMATORS[model]
    params |= get_model_spec(model).objective_params(objective, objective_alpha)
    params |= model_params or {}
    if feature_scaling is not None and feature_scaling not in CONFIG.FEATURE_SCALINGS:
        raise ValueError(
            f"CONFIG.FEATURE_SCALING は None か {CONFIG.FEATURE_SCALINGS} です: {feature_scaling}"
        )
    features = category_features(model, categorical_cols or [])
    if feature_scaling is not None:
        # 重複を除く（順は保つ）
        columns = list(dict.fromkeys(feature_cols if scaling_cols is None else scaling_cols))
        if not columns:
            raise ValueError("スケーリングする列がありません（CONFIG.FEATURE_SCALING_COLS）")
        unknown = sorted(set(columns) - set(feature_cols))
        if unknown:
            raise ValueError(f"使わない列はスケーリングできません: {unknown}")
        categorical = sorted(set(columns) & {CONFIG.CAT_COL, *(categorical_cols or [])})
        if categorical:
            raise ValueError(f"カテゴリの列はスケーリングできません: {categorical}")
        # Y_cat ごとのスケーリングは、系列ID をエンコードする前（Pipeline の先頭）に置く
        features.insert(
            0,
            {
                "class": "feature_engineering.scaling.GroupScaler",
                "params": {
                    "variables": columns,
                    "group_col": CONFIG.CAT_COL,
                    "method": feature_scaling,
                },
            },
        )
    if cv_method == "group":
        cv = {"method": "group", "n_splits": n_splits}
    elif cv_method == "timeseries":
        cv = {
            "method": "time_cutoff",
            "time_column": CONFIG.DATE_COL,
            "cutoffs": list(cutoffs or []),
        }
    else:
        raise ValueError(f"CONFIG.CV_METHOD は group か timeseries です: {cv_method}")
    return ExperimentConfig.model_validate(
        {
            "name": f"x50_regression_016_{model}",
            "task": "regression",
            "data": {
                "train_path": CONFIG.RAW_TRAIN_PATH,
                "test_path": CONFIG.RAW_TEST_PATH,
                "target": CONFIG.TARGET_COL,
                "group_col": CONFIG.GROUP_COL,
                "drop_cols": [CONFIG.DATE_COL, CONFIG.GROUP_COL],
                # Y_cat の列は、スケーリングのグループに使ってから Pipeline の中で除く
                "feature_cols": [CONFIG.CAT_COL, *feature_cols],
            },
            "features": features,
            "cv": cv,
            "model": {"name": model, "params": params, "early_stopping_rounds": 50},
            "metrics": ["mae", "rmse"],
            "test_prediction": test_prediction,
            "explain": {"enabled": True, "max_samples": CONFIG.SHAP_MAX_SAMPLES_PER_CAT},
            "evaluation": {
                "enabled": True,
                "training_history": True,
            },
            "tracking": {"experiment_name": CONFIG.MLFLOW_EXPERIMENT},
        }
    )


def scaled_feature_cols(config: ExperimentConfig) -> list[str]:
    """Y_cat ごとにスケーリングする列（スケーリングしなければ空）。

    実験設定の `GroupScaler` のステップから取り出す。
    """
    for step in config.features:
        if step.class_path == "feature_engineering.scaling.GroupScaler":
            return list(step.params["variables"])
    return []


def category_configs(
    features: dict[str, list[str]], scaling_cols: list[str], categorical_cols: list[str]
) -> dict[str, ExperimentConfig]:
    """Y_cat ごとの実験設定を作る（違うのは特徴量だけ。他は `CONFIG` の値で共通）。

    スケーリングする列・カテゴリの説明変数は、その Y_cat が使う列だけにする
    （スケーリングする列を1つも使わない Y_cat は、スケーリングしない）。

    Args:
        features: Y_cat → 特徴量（`category_features_map` の結果）。
        scaling_cols: スケーリングする列（全 Y_cat 共通の指定）。
        categorical_cols: カテゴリの説明変数（全 Y_cat 共通の指定）。
    """
    configs = {}
    for category, columns in features.items():
        scaled = [c for c in scaling_cols if c in columns]
        configs[category] = make_config(
            CONFIG.MODEL,
            columns,
            CONFIG.CV_METHOD,
            CONFIG.N_SPLITS,
            CONFIG.CUTOFFS,
            CONFIG.TEST_PREDICTION,
            CONFIG.FEATURE_SCALING if scaled else None,
            scaled,
            objective=CONFIG.OBJECTIVE,
            objective_alpha=CONFIG.OBJECTIVE_ALPHA,
            model_params=CONFIG.MODEL_PARAMS,
            categorical_cols=[c for c in categorical_cols if c in columns],
        )
    return configs

### 学習・後処理と確認

In [ ]:
def prepare(config: ExperimentConfig, train: pd.DataFrame, test: pd.DataFrame) -> Dataset:
    """学習用データ一式を作り、リークのおそれがないことを確かめる。

    確かめること:
        - モデルの特徴量に、目的変数・日付・グループ・`eliminate_flg` の列と、`CONFIG` で指定して
          いないラグ・移動平均の列が無い。
        - GroupKFold: 同じ年月が学習と検証の両方に入らず、各年月は1つの fold の検証に入る。
        - 時系列CV: 各 fold の学習期間が、すべて検証期間より前にある。

    Raises:
        LeakageError: 上のいずれかを満たさない場合。
    """
    dataset = prepare_dataset(config, train, test)
    leaked = [
        c
        for c in dataset.X.columns
        if c in (CONFIG.TARGET_COL, CONFIG.DATE_COL, CONFIG.GROUP_COL, CONFIG.ELIMINATE_COL)
        or (
            c.startswith((f"{CONFIG.TARGET_COL}_lag_", f"{CONFIG.TARGET_COL}_ma_"))
            and c not in LagSpec.from_config().names
        )
    ]
    if leaked:
        raise LeakageError(f"モデルの特徴量に使えない列があります: {leaked}")
    groups = train[CONFIG.GROUP_COL].to_numpy()
    dates = train[CONFIG.DATE_COL]
    validated: list[str] = []
    for i, (train_idx, valid_idx) in enumerate(dataset.folds):
        if config.cv.method == "group":
            shared = set(groups[train_idx]) & set(groups[valid_idx])
            if shared:
                raise LeakageError(f"fold {i}: 同じ年月が学習と検証にあります: {sorted(shared)}")
            validated += sorted(set(groups[valid_idx]))
        elif dates.iloc[train_idx].max() >= dates.iloc[valid_idx].min():
            raise LeakageError(f"fold {i}: 学習期間が検証期間より後の日を含みます")
    if config.cv.method == "group" and sorted(validated) != sorted(set(groups)):
        raise LeakageError("検証に入らない年月、または複数の fold の検証に入る年月があります")
    return dataset


def tail_weights(y: np.ndarray, categories: np.ndarray, irregular: np.ndarray) -> np.ndarray | None:
    """目的変数の分布の裾の行に重みを付ける（`CONFIG.TAIL_QUANTILES` が None なら None を返す）。

    カテゴリごとに、渡した行（CV の学習側）の分位点を求め、
    下側の分位点以下・上側の分位点以上の行の重みを
    `CONFIG.TAIL_WEIGHT` にする（それ以外は1）。`irregular_flg` が 1 の行は通常と違う値なので、
    分位点の計算にも重み付けにも使わない（重み1）。

    Raises:
        ValueError: 分位の指定が 0〜1 でない、または下側が上側以上の場合。
    """
    if CONFIG.TAIL_QUANTILES is None:
        return None
    low_q, high_q = CONFIG.TAIL_QUANTILES
    for q in (low_q, high_q):
        if q is not None and not 0 <= q <= 1:
            raise ValueError(
                f"CONFIG.TAIL_QUANTILES は 0〜1 で指定してください: {CONFIG.TAIL_QUANTILES}"
            )
    if low_q is not None and high_q is not None and low_q >= high_q:
        raise ValueError(
            f"CONFIG.TAIL_QUANTILES は (下側, 上側) の順で指定してください: {CONFIG.TAIL_QUANTILES}"
        )
    normal = irregular != 1  # 欠損（NaN）は通常の行として扱う
    frame = pd.DataFrame({"y": y, "cat": categories})
    grouped = frame[normal].groupby("cat")["y"]
    tail = np.zeros(len(y), dtype=bool)
    if low_q is not None:
        tail |= y <= frame["cat"].map(grouped.quantile(low_q)).to_numpy()
    if high_q is not None:
        tail |= y >= frame["cat"].map(grouped.quantile(high_q)).to_numpy()
    return np.where(normal & tail, float(CONFIG.TAIL_WEIGHT), 1.0)


def memory_mb() -> float:
    """このプロセスが使っているメモリ（RSS、MB）。psutil が無ければ NaN。"""
    try:
        import psutil
    except ImportError:
        return float("nan")
    return psutil.Process().memory_info().rss / 2**20


def release_memory() -> None:
    """使い終わったオブジェクトを解放する（Linux では、解放したメモリを OS に返す）。

    Python・NumPy が解放したメモリは、プロセスの中に残ったままになることがある。
    Linux（glibc）では `malloc_trim` で OS に返し、ほかの処理（次の Y_cat の学習）が
    使えるようにする。
    """
    gc.collect()
    if sys.platform.startswith("linux"):
        with contextlib.suppress(OSError, AttributeError):
            ctypes.CDLL("libc.so.6").malloc_trim(0)


class MemoryPeak:
    """学習中のプロセスのメモリ（RSS）の最大を、測った時点の中から記録する。"""

    def __init__(self) -> None:
        self.peak = memory_mb()

    def update(self) -> None:
        """今のメモリを測り、最大を更新する。"""
        current = memory_mb()
        if not np.isnan(current):
            self.peak = current if np.isnan(self.peak) else max(self.peak, current)


def assert_scaled(frame: pd.DataFrame, groups: np.ndarray, method: str, label: str) -> None:
    """Y_cat ごとに、標準化なら平均0・標準偏差1、正規化なら最小値0・最大値1 であることを確かめる。

    ばらつきの無い列は、変換後は全部 0 になる。

    Raises:
        AssertionError: 中心・分母がずれている場合。
    """
    values = frame.astype("float64").replace([np.inf, -np.inf], np.nan)
    grouped = values.groupby(groups)
    if method == "standardize":
        low, high = grouped.mean().to_numpy(), grouped.std(ddof=0).to_numpy()
    else:
        low, high = grouped.min().to_numpy(), grouped.max().to_numpy()
    # 値の無い Y_cat の列（全部欠損）は比べない。ばらつきの無い列は、変換後は全部 0
    observed = ~np.isnan(low)
    spread = observed & (np.abs(high) > 1e-9)
    assert np.allclose(low[observed], 0.0, atol=1e-5), f"{label}: 中心がずれています"
    assert np.allclose(high[spread], 1.0, atol=1e-5), f"{label}: 分母がずれています"


def check_scaling(
    pipeline: Any,
    scaler: TargetScaler,
    X_train: pd.DataFrame,
    y_train: np.ndarray,
    scaled_cols: list[str],
    label: str,
) -> None:
    """1つの fold のスケーリングが、その fold の学習側だけで fit されていることを確かめる。

    - 説明変数: Pipeline の先頭の `GroupScaler` の中心が、学習側から pandas で求めた値と一致し、
      学習側を変換すると Y_cat ごとに平均0・標準偏差1（正規化なら最小値0・最大値1）になる。
    - 目的変数: 学習側を変換すると同じく平均0・標準偏差1（最小値0・最大値1）になり、
      逆変換で元に戻る。

    Raises:
        AssertionError: 上のいずれかを満たさない場合。
    """
    groups = X_train[CONFIG.CAT_COL].to_numpy()
    if scaled_cols:
        step = pipeline.steps[0][1]
        assert isinstance(step, GroupScaler), "Pipeline の先頭が GroupScaler ではありません"
        raw = X_train[scaled_cols].astype("float64").replace([np.inf, -np.inf], np.nan)
        grouped = raw.groupby(groups)
        expected = grouped.mean() if CONFIG.FEATURE_SCALING == "standardize" else grouped.min()
        observed = expected.notna().to_numpy()
        center = step.center_.reindex(index=expected.index, columns=expected.columns).to_numpy()
        assert np.allclose(center[observed], expected.to_numpy()[observed]), (
            f"{label}: 説明変数の統計量が学習側の値と違います"
        )
        assert_scaled(
            step.transform(X_train)[scaled_cols],
            groups,
            CONFIG.FEATURE_SCALING,
            f"{label} 説明変数",
        )
    z = scaler.transform(y_train, groups)
    np.testing.assert_allclose(scaler.inverse(z, groups), y_train, rtol=1e-9, atol=1e-9)
    if scaler.kind in TargetScaler.AFFINE:
        assert_scaled(pd.DataFrame({"y": z}), groups, scaler.kind, f"{label} 目的変数")


def rescale_shap(
    result: ShapResult, scaler: TargetScaler, X: pd.DataFrame, scaled_cols: list[str]
) -> ShapResult:
    """1つの fold の OOF SHAP を、目的変数・説明変数のスケーリングの前の尺度に逆変換する。

    - SHAP 値・期待値: 目的変数の変換が一次式（標準化・正規化）なら、行の Y_cat の (中心, 分母) で
      `SHAP 値 × 分母`、`期待値 × 分母 + 中心` にする（`予測 = 期待値 + SHAP 値の和` なので、
      元の尺度の予測の厳密な分解になる）。対数の変換では戻せないので、モデルの尺度のまま返す。
    - 説明変数の値（beeswarm の色・相関の計算に使う）: スケーリングした列を、スケーリング前の値
      （±inf は、モデルの入力と同じく欠損）に置き換える。

    Args:
        result: 1つの fold の `compute_oof_shap` の結果（モデルの尺度。`rows` は `X` の行の位置）。
        scaler: その fold の目的変数の変換。
        X: その Y_cat の学習データの特徴量。
        scaled_cols: スケーリングした列。
    """
    values, base = result.values, result.base_values
    if scaler.kind in TargetScaler.AFFINE:
        center, scale = scaler.affine(X[CONFIG.CAT_COL].to_numpy()[result.rows])
        values = values * scale[:, None]
        base = base * scale + center
    data = result.data
    columns = [c for c in scaled_cols if c in data.columns]
    if columns:
        raw = X.iloc[result.rows][columns].astype("float64").replace([np.inf, -np.inf], np.nan)
        data = data.assign(**{c: raw[c].to_numpy() for c in columns})
    return dataclasses.replace(result, values=values, base_values=base, data=data)


def concat_shap(parts: list[ShapResult]) -> ShapResult | None:
    """SHAP の結果をつなぐ（fold・Y_cat ごとに特徴量の列が違っても、列の和集合にそろえる）。

    無い列は、SHAP 値を 0（予測に寄与しない）、特徴量の値を欠損にする
    （`compute_oof_shap` と同じ扱い）。
    """
    if not parts:
        return None
    columns = list(dict.fromkeys(c for part in parts for c in part.feature_names))
    values = [
        pd.DataFrame(p.values, columns=p.feature_names)
        .reindex(columns=columns, fill_value=0.0)
        .to_numpy()
        for p in parts
    ]
    return ShapResult(
        values=np.concatenate(values),
        base_values=np.concatenate([p.base_values for p in parts]),
        data=pd.concat([p.data.reindex(columns=columns) for p in parts], ignore_index=True),
        rows=np.concatenate([p.rows for p in parts]),
        explainer_kind="/".join(dict.fromkeys(p.explainer_kind for p in parts)),
    )


def fold_influence(
    pipeline: Any, X: pd.DataFrame, residuals: np.ndarray, rows: np.ndarray
) -> pd.DataFrame | None:
    """1つの fold のモデルの入力（前処理後の検証側）で、Leverage と Cook の距離を求める。

    `modeling.evaluation` の学習時の図と同じ計算（fold ごとに、その fold のモデルの
    前処理の後の値で計算）。
    行数が特徴量数以下などで計算できなければ None。

    Args:
        pipeline: その fold のモデル。
        X: その Y_cat の学習データの特徴量。
        residuals: `rows` の行の残差（実測 − OOF 予測）。
        rows: 計算する検証側の行の位置。
    """
    if rows.size == 0:
        return None
    model_input = np.asarray(pipeline[:-1].transform(X.iloc[rows]), dtype=np.float64)
    try:
        influence = compute_influence(model_input, residuals)
    except (ValueError, np.linalg.LinAlgError):
        return None
    return influence.assign(row=rows)


def coefficient_table(pipeline: Any) -> pd.DataFrame | None:
    """線形モデルの係数の表（絶対値の大きい順。線形モデルでなければ None）。"""
    if get_model_spec(CONFIG.MODEL).explainer_kind != "linear":
        return None
    model = pipeline.named_steps[MODEL_STEP]
    linear = model[-1]
    return (
        pd.DataFrame({"feature": model.feature_names_in_, "coef": linear.coef_})
        .assign(intercept=float(linear.intercept_))
        .assign(abs_coef=lambda d: d["coef"].abs())
        .sort_values("abs_coef", ascending=False, kind="stable")
        .drop(columns="abs_coef")
        .reset_index(drop=True)
    )


def changed_test_features(
    X_test: pd.DataFrame, one_day: np.ndarray, categorical_cols: list[str]
) -> pd.DataFrame:
    """リークの確認用に、テスト期間のある1日の説明変数を書き換えたテストの特徴量。

    説明変数・日付の特徴量は乱数に、カテゴリの説明変数は学習に無いカテゴリにする
    （Y_cat は変えない）。
    """
    rng = np.random.default_rng(1)
    exog = [c for c in X_test.columns if c != CONFIG.CAT_COL and c not in categorical_cols]
    changed = X_test.copy()
    # 整数の列（日付の年・月など）にも小数の乱数を入れられるよう、小数の型にする
    # （ラグ・移動平均の列も書き換えるが、再帰予測の中で計算した値に置き換わる）
    changed[exog] = changed[exog].astype(float)
    changed.loc[one_day, exog] = rng.uniform(1e3, 1e4, size=(int(one_day.sum()), len(exog)))
    # 小数の列は元の型に戻す（型が変わるとラグの丸めが変わり、予測がわずかにずれるため）
    floats = {c: X_test[c].dtype for c in exog if pd.api.types.is_float_dtype(X_test[c])}
    changed = changed.astype(floats)
    if categorical_cols:
        changed.loc[one_day, categorical_cols] = "__changed__"
    return changed


def save_model(pipeline: Any, scaler: TargetScaler, path: Path) -> Path:
    """テスト予測に使ったモデルを joblib で保存する（ノートブックの外でも読めるようにする）。

    保存するのは辞書 `{"pipeline": Pipeline, "target_transform": 目的変数の変換, "lag_features":
    ラグ・移動平均の指定}`。ラグ・移動平均は Pipeline に含まれないので、元の尺度の履歴
    （`irregular_flg` が 1 の日は NaN）から作って渡す。
    Pipeline は `src` のクラスだけでできているので、`src` を import パスに入れれば読める。
    目的変数の変換はこのノートブックのクラスではなく、値だけ
    （`kind`・`offset`・Y_cat ごとの `center`・`scale`）で保存する。
    元の尺度の予測は、`z = pipeline.predict(X)` から次のように戻す。

    - `kind` が None: `z`
    - `"standardize"`・`"normalize"`: `z × scale[Y_cat] + center[Y_cat]`
    - `"log"`: `exp(z) − offset`
    """
    target_transform: dict[str, Any] = {"kind": scaler.kind, "offset": scaler.offset}
    if scaler.kind in TargetScaler.AFFINE:
        target_transform["center"] = {str(k): float(v) for k, v in scaler.center_.items()}
        target_transform["scale"] = {str(k): float(v) for k, v in scaler.scale_.items()}
    lags = {"lags": CONFIG.LAGS, "windows": CONFIG.ROLLING_WINDOWS}
    lags["min_periods"] = CONFIG.ROLLING_MIN_PERIODS
    saved = {"pipeline": pipeline, "target_transform": target_transform, "lag_features": lags}
    joblib.dump(saved, path)
    return path


def fold_hidden(
    history: pd.Series, dates: pd.Series, valid_idx: np.ndarray, cv_method: str
) -> np.ndarray:
    """1つの fold で、実測値を見てはいけない日（検証期間）を返す（`history` の日ごとの真偽値）。

    - 時系列 CV: 検証の最初の日から後の全部の日。
    - GroupKFold: 検証の年月の全部の日。

    検証の行が無い日（`eliminate_flg` が 1 の日など）も、検証期間の中なら含める。

    Args:
        history: その Y_cat の目的変数の系列（1日刻み）。
        dates: その Y_cat の学習データの日付。
        valid_idx: 検証の行の位置。
        cv_method: 実験設定の CV の方法（`group` / `time_cutoff`）。
    """
    valid_dates = dates.iloc[valid_idx]
    if cv_method == "group":
        months = valid_dates.dt.strftime("%Y-%m").unique()
        return np.asarray(history.index.strftime("%Y-%m").isin(months))
    return np.asarray(history.index >= valid_dates.min())


def horizon_days(hidden: np.ndarray, positions: np.ndarray) -> np.ndarray:
    """再帰予測で何日先の予測か（実測値を見てよい最後の日から数えた日数）を返す。"""
    days = np.arange(len(hidden))
    last_known = np.maximum.accumulate(np.where(hidden, -1, days))
    return (days - last_known)[positions]


def fold_features(
    X: pd.DataFrame, dates: pd.Series, known: pd.Series, train_idx: np.ndarray, spec: LagSpec
) -> pd.DataFrame:
    """学習側の行のラグ・移動平均を、検証期間の実測値を消した系列から作り直す。

    GroupKFold では、検証の月の後の学習側の行が、検証の月の実測値をラグに使わないようにするため。
    検証側の行は、実測値のラグ（1期先予測用）のまま。
    """
    values = X[spec.names].to_numpy(dtype=np.float64, copy=True)
    positions = day_positions(dates.iloc[train_idx], known)
    values[train_idx] = spec.compute(known.to_numpy(), positions)
    return with_lags(X, values, spec.names)


def recursive_predict(
    pipeline: Any,
    scaler: TargetScaler,
    X: pd.DataFrame,
    dates: pd.Series,
    irregular: np.ndarray,
    known: pd.Series,
    spec: LagSpec,
) -> tuple[np.ndarray, pd.DataFrame]:
    """日付の順に、予測値を履歴に加えながら予測する（再帰予測）。

    履歴に加えるのは元の尺度の予測値（後処理の前）。`irregular_flg` が 1 の日は、予測値の代わりに
    NaN を加える（後の日のラグ・移動平均に使わない）。`spec.block` 日分は、予測値を待たずに
    まとめて予測できる。

    Args:
        pipeline: 学習済みのモデル。
        scaler: 目的変数の変換。
        X: 予測する行の特徴量（1つの Y_cat。ラグ・移動平均の列の値は使わない）。
        dates: 予測する行の日付。
        irregular: 予測する行の `irregular_flg` が 1 か。
        known: 目的変数の系列（1日刻み。予測する日は NaN）。
        spec: ラグ・移動平均の定義。

    Returns:
        (予測（元の尺度。`X` の順）, 予測に使ったラグ・移動平均を入れた特徴量)。

    Raises:
        LeakageError: 予測する日の実測値が `known` にある場合や、系列より前の日を予測する場合。
        ValueError: 同じ日の行が複数ある場合。
    """
    positions = day_positions(dates, known)
    if len(np.unique(positions)) != len(positions):
        raise ValueError("予測する行に、同じ Y_cat・日付の行が複数あります")
    if positions.min() < 0:
        raise LeakageError("目的変数の系列の最初より前の日は、再帰予測できません")
    values = np.full(max(len(known), int(positions.max()) + 1), np.nan)
    values[: len(known)] = known.to_numpy()
    if np.isfinite(values[positions]).any():
        raise LeakageError("予測する日の実測値が、ラグの履歴に入っています")
    categories = X[CONFIG.CAT_COL].to_numpy()
    order = np.argsort(positions, kind="stable")
    ordered = positions[order]
    pred = np.full(len(X), np.nan)
    lags = np.full((len(X), len(spec.names)), np.nan)
    first = 0
    while first < len(order):
        last = int(np.searchsorted(ordered, ordered[first] + spec.block))
        rows = order[first:last]
        lags[rows] = spec.compute(values, positions[rows])
        features = with_lags(X.iloc[rows], lags[rows], spec.names)
        z = predict(pipeline, features, Task.REGRESSION)
        pred[rows] = scaler.inverse(z, categories[rows])
        # irregular_flg が 1 の日の予測値は NaN に上書きして、後の日のラグ・移動平均に使わない
        values[positions[rows]] = np.where(irregular[rows], np.nan, pred[rows])
        first = last
    # 予測の途中で使ったラグが、最後の履歴から計算し直した値と同じ（後の日の値を使っていない）
    np.testing.assert_array_equal(spec.compute(values, positions), lags)
    assert np.isnan(values[positions[irregular]]).all(), "irregular_flg が 1 の日の値が残っています"
    return pred, with_lags(X, lags, spec.names)


class TestPredictions:
    """テスト予測（再帰予測）を、テストを予測するモデルごとに足し合わせる（モデルを残さないため）。

    リークの確認のため、予測に使った特徴量をまとめて渡した予測のやり直しと、ある1日の説明変数を書き換えた
    ときの、その日までの再帰予測も足し合わせる。

    Args:
        X_test: テストの特徴量（1つの Y_cat）。
        dates: テストの日付。
        history: その Y_cat の目的変数の系列（学習期間の全部）。
        spec: ラグ・移動平均の定義。
        changed: ある1日の説明変数を書き換えたテストの特徴量（None なら確かめない）。
        one_day_date: 書き換えた日。
    """

    def __init__(
        self,
        X_test: pd.DataFrame,
        dates: pd.Series,
        history: pd.Series,
        spec: LagSpec,
        changed: pd.DataFrame | None,
        one_day_date: pd.Timestamp | None,
    ) -> None:
        self.X_test, self.dates, self.history, self.spec = X_test, dates, history, spec
        self.irregular = (X_test[CONFIG.IRREGULAR_COL] == 1).to_numpy()
        self.changed, self.one_day_date = changed, one_day_date
        self.sums: dict[str, np.ndarray] = {}
        self.n_models = 0

    def add(self, pipeline: Any, scaler: TargetScaler) -> None:
        """1つのモデルのテスト予測（元の尺度）を足す。"""
        pred, features = recursive_predict(
            pipeline, scaler, self.X_test, self.dates, self.irregular, self.history, self.spec
        )
        z = predict(pipeline, features, Task.REGRESSION)
        outputs = {
            "pred": pred,
            "again": scaler.inverse(z, self.X_test[CONFIG.CAT_COL].to_numpy()),
            "lags": features[self.spec.names].to_numpy(dtype=np.float64),
        }
        if self.changed is not None and self.one_day_date is not None:
            # 書き換えた日までを再帰予測する（その日より前の予測が変わらないことを確かめる）
            until = (self.dates <= self.one_day_date).to_numpy()
            changed = np.full(len(self.X_test), np.nan)
            changed[until], _ = recursive_predict(
                pipeline,
                scaler,
                self.changed[until],
                self.dates[until],
                self.irregular[until],
                self.history,
                self.spec,
            )
            outputs["changed"] = changed
        for name, values in outputs.items():
            self.sums[name] = self.sums.get(name, 0.0) + values
        self.n_models += 1

    def mean(self, name: str = "pred") -> np.ndarray:
        """足し合わせた値の平均（`fold_mean` なら fold のモデルの平均、`refit_full` なら1つ）。"""
        return self.sums[name] / self.n_models


def train_category(
    config: ExperimentConfig,
    dataset: Dataset,
    dates: pd.Series,
    history: pd.Series,
    params: dict | None = None,
    analyze: bool = True,
    test_dates: pd.Series | None = None,
    one_day_date: pd.Timestamp | None = None,
    model_dir: Path | None = None,
) -> tuple[CVResult, dict[str, Any]]:
    """1つの Y_cat のモデルを CV で学習し、OOF 予測・テスト予測を元の尺度で返す。

    013 の `train_cv` と同じ流れ（目的変数の変換・裾の重み付けを fold ごとに行い、
    予測を元の尺度に逆変換する）。メモリを節約するため、各 fold のモデルは、その fold の分析
    （スケーリングの確認・SHAP・Leverage・学習の推移）とテスト予測を済ませたらすぐに捨てる
    （同時にメモリにあるモデルは最大1つ）。
    返す CV の結果の `models` は空。

    各 fold では、検証期間の実測値を消した系列から学習側のラグを作り、検証側は再帰予測する
    （OOF 予測・SHAP・Leverage は、再帰予測に使ったラグによるもの）。
    early stopping は、検証側の1期先予測（実測値のラグ）の損失で止める。

    Args:
        config: 実験設定。
        dataset: その Y_cat の学習用データ一式（`X` のラグ・移動平均は実測値から作ったもの）。
        dates: 学習データの日付（`dataset.X` の行の順）。
        history: その Y_cat の目的変数の系列（`target_histories` の結果）。
        params: モデルのパラメータ（None なら設定の値。Optuna の試行で上書きする）。
        analyze: True なら分析とテスト予測も行う。Optuna の試行では False にし、
            学習の推移も記録しない。
        test_dates: テストの日付（`dataset.X_test` の行の順。`analyze` のときに必要）。
        one_day_date: リークの確認でテストの説明変数を書き換える日（None なら確かめない）。
        model_dir: テスト予測に使ったモデルの保存先（None なら保存しない）。

    Returns:
        (CV の結果, 分析の結果。キーは `shap`・`influence`・`influence_irregular0`・`n_params`・
        `histories`・`coefficients`・`checks`・`onestep`・`horizon`・`test_lags`)。
    """
    X, y, X_test = dataset.X, dataset.y, dataset.X_test
    categories = X[CONFIG.CAT_COL].to_numpy()
    irregular = X[CONFIG.IRREGULAR_COL].to_numpy()
    # この Y_cat のモデルが使うラグ・移動平均
    spec_lags = LagSpec.from_config().select(list(X.columns))
    positions = day_positions(dates, history)
    kind, offset = CONFIG.TARGET_TRANSFORM, CONFIG.TARGET_LOG_OFFSET
    metrics = [get_metric(name) for name in config.metrics]
    spec = get_model_spec(config.model.name)
    params = dict(config.model.params if params is None else params)
    scaled_cols = scaled_feature_cols(config)
    fold_ids = folds_to_ids(dataset.folds, len(X))
    if not analyze:
        # Optuna の試行では学習の推移（反復ごとの損失）を記録しない（その分のメモリと時間を省く）
        evaluation = config.evaluation.model_copy(update={"training_history": False})
        config = config.model_copy(update={"evaluation": evaluation})
    # SHAP を計算する行は、学習の前に決めておく（モデルを fold ごとに捨てるため）
    shap_limit = config.explain.max_samples
    if spec.shap_max_samples is not None:
        shap_limit = min(shap_limit, spec.shap_max_samples)
    shap_rows = sample_oof_rows(fold_ids, shap_limit, config.seed)
    tests = None
    if analyze:
        assert X_test is not None and test_dates is not None
        # カテゴリの説明変数（Y_cat 以外の文字列の列）は、学習に無いカテゴリに書き換える
        text_cols = [
            c
            for c in X_test.columns
            if c != CONFIG.CAT_COL and not pd.api.types.is_numeric_dtype(X_test[c])
        ]
        changed = None
        if one_day_date is not None:
            one_day = (test_dates == one_day_date).to_numpy()
            changed = changed_test_features(X_test, one_day, text_cols)
        tests = TestPredictions(X_test, test_dates, history, spec_lags, changed, one_day_date)
    extras: dict[str, Any] = {
        "shap": [],
        "influence": [],
        "influence_irregular0": [],
        "ranks": [],
        "histories": [],
        "coefficients": None,
        "checks": {},
        "onestep": np.full(len(X), np.nan),
        "horizon": np.zeros(len(X), dtype=np.int64),
        "test_lags": None,
    }
    oof = np.full(len(X), np.nan)
    fold_scores: dict[str, list[float]] = {m.name: [] for m in metrics}
    best_iterations: list[int | None] = []
    n_folds = len(dataset.folds)
    for fold, (train_idx, valid_idx) in enumerate(dataset.folds):
        cats_tr, cats_va = categories[train_idx], categories[valid_idx]
        # 検証期間の実測値を消した系列から、学習側のラグ・移動平均を作る
        hidden = fold_hidden(history, dates, valid_idx, config.cv.method)
        known = history.mask(hidden)
        X_fold = fold_features(X, dates, known, train_idx, spec_lags)
        scaler = TargetScaler(kind, offset).fit(y[train_idx], cats_tr)
        pipeline = fit_pipeline(
            config,
            X_fold.iloc[train_idx],
            scaler.transform(y[train_idx], cats_tr),
            X_fold.iloc[valid_idx],
            scaler.transform(y[valid_idx], cats_va),
            params=params,
            sample_weight=tail_weights(y[train_idx], cats_tr, irregular[train_idx]),
        )
        # 比較用の1期先予測（前日までの実測値が分かるとしたラグ）
        onestep = predict(pipeline, X_fold.iloc[valid_idx], Task.REGRESSION)
        extras["onestep"][valid_idx] = scaler.inverse(onestep, cats_va)
        extras["horizon"][valid_idx] = horizon_days(hidden, positions[valid_idx])
        # 検証期間の再帰予測（検証期間の実測値は使わない）
        pred, X_valid = recursive_predict(
            pipeline,
            scaler,
            X_fold.iloc[valid_idx],
            dates.iloc[valid_idx],
            irregular[valid_idx] == 1,
            known,
            spec_lags,
        )
        oof[valid_idx] = pred
        for m in metrics:
            fold_scores[m.name].append(m(y[valid_idx], pred))
        best_iterations.append(spec.best_iteration(pipeline.named_steps[MODEL_STEP]))
        if analyze:
            # SHAP・Leverage は、再帰予測に使ったラグ・移動平均で計算する
            lag_values = X_fold[spec_lags.names].to_numpy(dtype=np.float64, copy=True)
            lag_values[valid_idx] = X_valid[spec_lags.names].to_numpy(dtype=np.float64)
            fold_data = dataclasses.replace(
                dataset, X=with_lags(X_fold, lag_values, spec_lags.names)
            )
            analyze_fold(config, fold_data, fold, pipeline, scaler, oof, shap_rows, extras)
            if config.test_prediction == "fold_mean":
                assert tests is not None
                tests.add(pipeline, scaler)
                if model_dir is not None:
                    path = ensure_parent_dir(model_dir / f"fold{fold}.joblib")
                    save_model(pipeline, scaler, path)
                if fold == n_folds - 1:
                    extras["coefficients"] = coefficient_table(pipeline)
        # この fold のモデルは使い終わったので捨てる
        del pipeline
        release_memory()
    test_pred = None
    if analyze:
        assert tests is not None
        if config.test_prediction != "fold_mean":
            # early stopping の fold 平均の木の本数で、その Y_cat の学習データ全体で学習し直す
            # （ラグ・移動平均は、学習期間の全部の実測値から作ったもの）
            iterations = [b for b in best_iterations if b is not None]
            full_params = (
                spec.with_n_iterations(params, int(np.mean(iterations))) if iterations else params
            )
            scaler = TargetScaler(kind, offset).fit(y, categories)
            full = fit_pipeline(
                config,
                X,
                scaler.transform(y, categories),
                params=full_params,
                early_stopping=False,
                sample_weight=tail_weights(y, categories, irregular),
            )
            tests.add(full, scaler)
            extras["coefficients"] = coefficient_table(full)
            if model_dir is not None:
                path = ensure_parent_dir(model_dir / "refit_full.joblib")
                save_model(full, scaler, path)
            del full
            release_memory()
        test_pred = tests.mean()
        extras["checks"] = check_test_predictions(tests)
        extras["test_lags"] = pd.DataFrame(tests.mean("lags"), columns=spec_lags.names)
    predicted = fold_ids >= 0
    result = CVResult(
        oof_pred=oof,
        fold_ids=fold_ids,
        fold_scores=fold_scores,
        oof_scores={m.name: m(y[predicted], oof[predicted]) for m in metrics},
        test_pred=test_pred,
        models=[],
        best_iterations=best_iterations,
        params=params,
    )
    ranks = extras.pop("ranks")
    extras["n_params"] = int(round(float(np.mean(ranks)))) if ranks else 0
    if analyze:
        extras["checks"]["スケーリング"] = (
            f"{n_folds} fold とも、学習側だけで fit している（目的変数: {kind}"
            f" / 説明変数: {CONFIG.FEATURE_SCALING if scaled_cols else None}）"
        )
    return result, extras


def analyze_fold(
    config: ExperimentConfig,
    dataset: Dataset,
    fold: int,
    pipeline: Any,
    scaler: TargetScaler,
    oof: np.ndarray,
    shap_rows: np.ndarray,
    extras: dict[str, Any],
) -> None:
    """1つの fold のモデルを捨てる前に、モデルが必要な分析を行い、小さい結果だけを `extras` に足す。

    1. スケーリングの確認（`check_scaling`）。
    2. SHAP（`shap_rows` のうち、この fold の検証側の行。元の尺度に逆変換する）。
    3. Leverage・Cook の距離（検証側の全行と、`irregular_flg` が 0 の行だけ）。
    4. 学習の推移（記録のあるモデルだけ）。
    """
    X, y = dataset.X, dataset.y
    train_idx, valid_idx = dataset.folds[fold]
    scaled_cols = scaled_feature_cols(config)
    check_scaling(pipeline, scaler, X.iloc[train_idx], y[train_idx], scaled_cols, f"fold {fold}")
    fold_ids = folds_to_ids(dataset.folds, len(X))
    rows = shap_rows[fold_ids[shap_rows] == fold]
    if rows.size:
        # この fold のモデルだけを持つ CV の結果を作り、この fold の行だけを説明する
        only_this_fold = np.full(len(X), -1)
        only_this_fold[rows] = 0
        one_fold = CVResult(
            oof_pred=oof,
            fold_ids=only_this_fold,
            fold_scores={},
            oof_scores={},
            test_pred=None,
            models=[pipeline],
        )
        shap_part = compute_oof_shap(
            config, X, [dataset.folds[fold]], one_fold, max_samples=int(rows.size)
        )
        extras["shap"].append(rescale_shap(shap_part, scaler, X, scaled_cols))
    observed = np.asarray(valid_idx)[np.isfinite(y[valid_idx]) & np.isfinite(oof[valid_idx])]
    normal = (X[CONFIG.IRREGULAR_COL].to_numpy() == 0)[observed]
    for key, part in (("influence", observed), ("influence_irregular0", observed[normal])):
        influence = fold_influence(pipeline, X, y[part] - oof[part], part)
        if influence is not None:
            extras[key].append(influence.assign(fold=fold))
            if key == "influence":
                extras["ranks"].append(float(influence["leverage"].sum()))
    history = get_model_spec(config.model.name).training_history(pipeline.named_steps[MODEL_STEP])
    if history is not None:
        extras["histories"].append(history)


def check_test_predictions(tests: TestPredictions) -> dict[str, str]:
    """テスト予測のリークの確認（モデルを残さないので、テストを予測したときの値で確かめる）。

    1. 再帰予測に使った特徴量をまとめて渡して予測し直した値が、テスト予測と一致する。
    2. テスト期間のある1日の説明変数を書き換えても、それより前の日の予測は変わらない
       （後の日の予測は、ラグを通して変わる）。

    Raises:
        AssertionError: 上のいずれかを満たさない場合。
    """
    base = tests.mean()
    # 1行ずつの予測とまとめた予測では、足し算の順などで最後の桁が変わることがある
    np.testing.assert_allclose(tests.mean("again"), base, rtol=1e-6, atol=1e-6)
    checks = {"予測のやり直し": "テスト予測と一致する"}
    if "changed" in tests.sums:
        before = (tests.dates < tests.one_day_date).to_numpy()
        # RandomForest・ExtraTrees は木の予測を並列に足すため、足す順で最後の桁
        # （1e-14 程度）が変わり、再帰するとその差が後の日に伝わる。
        # 後の日の特徴量が漏れていれば差はずっと大きくなるので、ごく小さい許容差で比べる
        np.testing.assert_allclose(tests.mean("changed")[before], base[before], rtol=1e-9, atol=0)
        checks["後の日を使わない"] = (
            f"{tests.one_day_date:%Y-%m-%d} を書き換えても、前の {int(before.sum())}日は変わらない"
        )
    return checks


def tune_params(
    config: ExperimentConfig, dataset: Dataset, dates: pd.Series, history: pd.Series
) -> tuple[dict, pd.DataFrame]:
    """Optuna で1つの Y_cat のモデルのパラメータを探す（目的関数は再帰予測の CV の MAE）。

    探索空間は `modeling` のモデルごとの既定。各試行は `train_category` と同じ条件（目的変数の変換・
    裾の重み付け・CV の分け方）で評価し、分析・テスト予測はしない（モデル・学習の推移も残さない）。
    `CONFIG.N_TRIALS` 回（`CONFIG.TUNING_TIMEOUT` 秒で打ち切り）試す。

    Returns:
        (最良の試行で使ったパラメータ, 試行の一覧)。
    """
    import optuna

    # 試行ごとのログは画面に出さない（小さいインスタンスで出力量が増えないように）
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    base = dict(config.model.params)
    spec = get_model_spec(config.model.name)

    def objective(trial: optuna.Trial) -> float:
        params = base | spec.search_space(trial, config.task)
        # 探索空間には試行で選ばない固定値もあり、best_params には入らない。
        # 学習し直すときに同じ条件にするため、使ったパラメータを試行に残す
        trial.set_user_attr("params", params)
        cv, _ = train_category(config, dataset, dates, history, params, analyze=False)
        score = float(cv.oof_scores["mae"])
        del cv
        release_memory()
        return score

    study = optuna.create_study(
        direction="minimize", sampler=optuna.samplers.TPESampler(seed=config.seed)
    )
    study.optimize(
        objective, n_trials=CONFIG.N_TRIALS, timeout=CONFIG.TUNING_TIMEOUT, gc_after_trial=True
    )
    trials = study.trials_dataframe(attrs=("number", "value", "params", "state", "duration"))
    return dict(study.best_trial.user_attrs["params"]), trials


def fold_table_for(
    category: str, dataset: Dataset, frame: pd.DataFrame, cv: CVResult
) -> pd.DataFrame:
    """1つの Y_cat の、fold ごとの行数・検証の期間・誤差の表。

    `（flg=0）` の付いた列は、検証側のうち `irregular_flg` が 0 の行だけで計算した値。
    """
    dates = frame[CONFIG.DATE_COL]
    normal = (frame[CONFIG.IRREGULAR_COL] == 0).to_numpy()
    rows = []
    for i, (train_idx, valid_idx) in enumerate(dataset.folds):
        normal_idx = np.asarray(valid_idx)[normal[valid_idx]]
        normal_scores = scores_or_nan(dataset.y[normal_idx], cv.oof_pred[normal_idx])
        rows.append(
            {
                CONFIG.CAT_COL: category,
                "fold": i,
                "学習の行数": len(train_idx),
                "検証の行数": len(valid_idx),
                "検証の行数（flg=0）": len(normal_idx),
                "検証の最初の日": f"{dates.iloc[valid_idx].min():%Y-%m-%d}",
                "検証の最後の日": f"{dates.iloc[valid_idx].max():%Y-%m-%d}",
                "MAE": cv.fold_scores["mae"][i],
                "RMSE": cv.fold_scores["rmse"][i],
                "MAE（flg=0）": normal_scores["mae"],
                "RMSE（flg=0）": normal_scores["rmse"],
                "木の本数": cv.best_iterations[i],
            }
        )
    return pd.DataFrame(rows)


def run_category(
    config: ExperimentConfig,
    category: str,
    frame: pd.DataFrame,
    test: pd.DataFrame,
    rows: np.ndarray,
    test_rows: np.ndarray,
    output_dir: Path,
    one_day_date: pd.Timestamp,
    history: pd.Series,
) -> CategoryResult:
    """1つの Y_cat のモデルを学習し（Optuna・CV・テスト予測・分析）、小さい結果だけを返す。

    学習用データ・モデルはこの関数の中だけで使い、返す前に捨てる。

    Args:
        config: 実験設定。
        category: Y_cat。
        frame: その Y_cat の学習の表（`model_frames` の学習の表のうち、その Y_cat の行）。
        test: その Y_cat のテストの表。
        rows: 全 Y_cat の学習の表の中の、`frame` の行の位置。
        test_rows: 全 Y_cat のテストの表の中の、`test` の行の位置。
        output_dir: 実験の出力ディレクトリ。
        one_day_date: リークの確認で説明変数を書き換える日。
        history: その Y_cat の目的変数の系列（`target_histories` の結果）。
    """
    start = time.perf_counter()
    peak = MemoryPeak()
    dataset = prepare(config, frame, test)
    dates = frame[CONFIG.DATE_COL]
    params, trials = None, None
    tunable = get_model_spec(config.model.name).has_search_space
    if CONFIG.N_TRIALS > 0 and tunable:
        # Optuna でパラメータを探し、最良のパラメータで学習し直す
        params, trials = tune_params(config, dataset, dates, history)
        trials = trials.assign(**{CONFIG.CAT_COL: category})
        peak.update()
    model_dir = output_dir / "models" / category if CONFIG.SAVE_MODELS else None
    cv, extras = train_category(
        config,
        dataset,
        dates,
        history,
        params,
        analyze=True,
        test_dates=test[CONFIG.DATE_COL],
        one_day_date=one_day_date,
        model_dir=model_dir,
    )
    peak.update()
    category_dir = output_dir / "categories" / category
    save_cv_outputs(config, dataset, cv, category_dir)
    if trials is not None:
        trials.to_csv(category_dir / "tuning_trials.csv", index=False)
    # SHAP・Leverage の行の位置を、全 Y_cat の学習の表の中の位置にする
    shap = concat_shap(extras["shap"])
    if shap is not None:
        shap = dataclasses.replace(shap, rows=rows[shap.rows])
    influences = {}
    for key in ("influence", "influence_irregular0"):
        parts = extras[key]
        influences[key] = (
            pd.concat(parts, ignore_index=True).assign(row=lambda d: rows[d["row"].to_numpy()])
            if parts
            else None
        )
    result = CategoryResult(
        category=category,
        rows=rows,
        test_rows=test_rows,
        oof_pred=cv.oof_pred,
        onestep_pred=extras["onestep"],
        horizon=extras["horizon"],
        fold_ids=cv.fold_ids,
        test_pred=np.asarray(cv.test_pred),
        test_lags=extras["test_lags"],
        oof_scores={k: float(v) for k, v in cv.oof_scores.items()},
        params=cv.params,
        fold_table=fold_table_for(category, dataset, frame, cv),
        shap=shap,
        influence=influences["influence"],
        influence_irregular0=influences["influence_irregular0"],
        n_params=extras["n_params"],
        histories=extras["histories"],
        coefficients=extras["coefficients"],
        checks=extras["checks"],
        tuning_trials=trials,
        seconds=time.perf_counter() - start,
        memory_mb=float("nan"),
        peak_memory_mb=float("nan"),
    )
    # 学習用データ・CV の結果は使い終わったので捨てる（次の Y_cat の学習にメモリを空ける）
    del dataset, cv, extras
    release_memory()
    result.memory_mb = memory_mb()
    peak.update()
    result.peak_memory_mb = peak.peak
    return result


def run_training(
    configs: dict[str, ExperimentConfig],
    features: dict[str, list[str]],
    train: pd.DataFrame,
    test: pd.DataFrame,
    categorical_cols: list[str],
    tracker: Tracker,
    output_root: Path | None,
    histories: dict[str, pd.Series],
) -> RegressionRun:
    """Y_cat ごとにモデルを学習・予測し、結果をまとめて出力ディレクトリと MLflow に残す。

    Y_cat を1つずつ順に学習し、終わった Y_cat の学習用データ・モデルは捨てる（メモリにあるのは、
    学習中の Y_cat の1つのモデルと、終わった Y_cat の小さい結果だけ）。
    OOF 予測・テスト予測・SHAP は元の尺度（対数の変換のときだけ SHAP はモデルの尺度）。

    Args:
        configs: Y_cat → 実験設定（違うのは特徴量だけ）。
        features: Y_cat → モデルの特徴量。
        train: 学習の表（全 Y_cat）。
        test: テストの表（全 Y_cat）。
        categorical_cols: カテゴリの説明変数。
        tracker: MLflow に記録するか。
        output_root: 実験の出力のルート。
        histories: Y_cat → 目的変数の系列（`target_histories` の結果）。
    """
    categories = list(dict.fromkeys(train[CONFIG.CAT_COL]))
    # 実験名・CV・モデルのパラメータなど、Y_cat によらない設定の代表
    config = configs[categories[0]]
    output_dir = make_output_dir(config.name, output_root)
    missing = sorted(set(test[CONFIG.CAT_COL]) - set(categories))
    if missing:
        raise ValueError(
            f"学習データに無い Y_cat がテストデータにあります（モデルが無い）: {missing}"
        )
    # リークの確認で説明変数を書き換える、テスト期間のある1日（全 Y_cat で同じ日）
    test_dates = test[CONFIG.DATE_COL].drop_duplicates().sort_values()
    one_day_date = test_dates.iloc[len(test_dates) // 4]
    keys = [CONFIG.DATE_COL, CONFIG.CAT_COL]
    oof = (
        train[keys]
        .reset_index(drop=True)
        .assign(
            **{
                CONFIG.ACTUAL_COL: train[CONFIG.TARGET_COL].to_numpy(dtype=np.float64),
                CONFIG.PRED_COL: np.nan,
                "fold": -1,
                "pred_onestep": np.nan,
                "horizon": 0,
            }
        )
    )
    predictions = test[keys].reset_index(drop=True).assign(**{CONFIG.PRED_COL: np.nan})
    results: dict[str, CategoryResult] = {}
    start = time.perf_counter()
    with tracker.start_run(run_name=config.name, tags={"model_type": "regression_016"}):
        # 特徴量は Y_cat ごとに違うので、代表の設定からは除き、表にして残す
        dumped = config.model_dump(mode="json", by_alias=True)
        dumped["data"].pop("feature_cols", None)
        tracker.log_params({"config": dumped})
        features_path = output_dir / "features_by_category.csv"
        features_table(features).to_csv(features_path, index_label="feature")
        tracker.log_artifact(features_path)
        tracker.log_params({"tail_quantiles": CONFIG.TAIL_QUANTILES})
        tracker.log_params(
            {
                "lags": CONFIG.LAGS,
                "rolling_windows": CONFIG.ROLLING_WINDOWS,
                "rolling_min_periods": CONFIG.ROLLING_MIN_PERIODS,
                "lag_use_eliminated": CONFIG.LAG_USE_ELIMINATED,
            }
        )
        for category in categories:
            rows = np.flatnonzero((train[CONFIG.CAT_COL] == category).to_numpy())
            test_rows = np.flatnonzero((test[CONFIG.CAT_COL] == category).to_numpy())
            # その Y_cat の行だけを取り出す（全 Y_cat の表は複製しない）
            result = run_category(
                configs[category],
                category,
                train.iloc[rows].reset_index(drop=True),
                test.iloc[test_rows].reset_index(drop=True),
                rows,
                test_rows,
                output_dir,
                one_day_date,
                histories[category],
            )
            results[category] = result
            oof.loc[rows, CONFIG.PRED_COL] = result.oof_pred
            oof.loc[rows, "fold"] = result.fold_ids
            oof.loc[rows, "pred_onestep"] = result.onestep_pred
            oof.loc[rows, "horizon"] = result.horizon
            predictions.loc[test_rows, CONFIG.PRED_COL] = result.test_pred
            tracker.log_metrics({f"oof_mae_{category}": result.oof_scores["mae"]})
            logger.info(
                "%s: OOF MAE %.4f（%d行、%.1f秒、メモリ %.0f MB）",
                category,
                result.oof_scores["mae"],
                len(rows),
                result.seconds,
                result.memory_mb,
            )
        predicted = (oof["fold"] >= 0).to_numpy()
        actual, pred = oof[CONFIG.ACTUAL_COL].to_numpy(), oof[CONFIG.PRED_COL].to_numpy()
        tracker.log_metrics(
            {f"oof_{k}": v for k, v in scores_or_nan(actual[predicted], pred[predicted]).items()}
        )
        shap = concat_shap([r.shap for r in results.values() if r.shap is not None])
        if shap is not None:
            for path in save_shap_outputs(
                shap, output_dir / "shap", config.name, None, config.explain
            ):
                tracker.log_artifact(path, artifact_path="shap")
        run_id = tracker.active_run_id()
    logger.info("学習終了（%d種類、%.1f秒）", len(categories), time.perf_counter() - start)
    # テストの再帰予測に使ったラグ・移動平均（行の順はテストの表と同じ）。
    # その Y_cat で使わない列は欠損
    test_lags = pd.concat(
        [r.test_lags.set_axis(r.test_rows, axis=0) for r in results.values()]
    ).sort_index()
    test_lags = pd.concat(
        [
            predictions[keys],
            test[CONFIG.IRREGULAR_COL].reset_index(drop=True),
            predictions[CONFIG.PRED_COL],
            test_lags.reindex(columns=LagSpec.from_config().names),
        ],
        axis=1,
    )
    trials = [r.tuning_trials for r in results.values() if r.tuning_trials is not None]
    return RegressionRun(
        categories=categories,
        frame=train,
        output_dir=output_dir,
        run_id=run_id,
        predictions=predictions,
        oof=oof,
        test_lags=test_lags,
        results=results,
        categorical_cols=categorical_cols,
        features=features,
        shap=shap,
        tuning_trials=pd.concat(trials, ignore_index=True) if trials else None,
    )


def features_table(features: dict[str, list[str]]) -> pd.DataFrame:
    """どの Y_cat がどの特徴量を使うか（使うなら1）の表。行が特徴量、列が Y_cat。"""
    names = list(dict.fromkeys(c for columns in features.values() for c in columns))
    return pd.DataFrame(
        {cat: [int(c in columns) for c in names] for cat, columns in features.items()},
        index=names,
    )


def cv_table(run: RegressionRun) -> pd.DataFrame:
    """全 Y_cat の、fold ごとの結果の表（Y_cat ごとの `fold_table` をつないだもの）。"""
    return pd.concat([r.fold_table for r in run.results.values()], ignore_index=True)


def target_bounds(categories: list[str]) -> pd.DataFrame:
    """カテゴリごとの予測値の上下限の表（列 `Y_cat`, `lower`, `upper`。制限なしは NaN）。

    `CONFIG.TARGET_BOUNDS` に無いカテゴリは `CONFIG.DEFAULT_TARGET_BOUNDS` を使う。

    Raises:
        ValueError: 学習データに無いカテゴリを指定した場合、または下限が上限より大きい場合。
    """
    unknown = sorted(set(CONFIG.TARGET_BOUNDS) - set(categories))
    if unknown:
        raise ValueError(f"CONFIG.TARGET_BOUNDS に、学習データに無いカテゴリがあります: {unknown}")
    rows = []
    for category in categories:
        lower, upper = CONFIG.TARGET_BOUNDS.get(category, CONFIG.DEFAULT_TARGET_BOUNDS)
        if lower is not None and upper is not None and lower > upper:
            raise ValueError(f"{category}: 下限 {lower} が上限 {upper} より大きいです")
        rows.append({CONFIG.CAT_COL: category, "lower": lower, "upper": upper})
    return pd.DataFrame(rows).astype({"lower": float, "upper": float})


@dataclass(frozen=True)
class LinearCorrection:
    """予測値の補正の式 `補正後 = intercept + slope × 予測値`。

    Attributes:
        slope: 傾き。
        intercept: 切片。
        n_rows: 式を求めるのに使った行数。
    """

    slope: float
    intercept: float
    n_rows: int

    def apply(self, pred: np.ndarray) -> np.ndarray:
        """予測値を補正する。"""
        return self.intercept + self.slope * pred


def fit_correction(pred: np.ndarray, actual: np.ndarray) -> LinearCorrection:
    """予測値を説明変数、実績値を目的変数とする線形回帰（最小二乗法）で、補正の式を求める。

    予測値に散らばりが無い（傾きが決まらない）ときは、傾きを 1 にして平均のずれだけを補正する。
    行が無いときは補正しない式（傾き 1・切片 0）を返す。

    Args:
        pred: 予測値（欠損の無い値）。
        actual: 実績値（欠損の無い値）。
    """
    if len(pred) == 0:
        return LinearCorrection(1.0, 0.0, 0)
    pred_mean, actual_mean = float(pred.mean()), float(actual.mean())
    centered = pred - pred_mean
    if np.ptp(pred) <= 1e-9 * max(1.0, abs(pred_mean)):
        slope = 1.0
    else:
        slope = float(centered @ (actual - actual_mean) / (centered @ centered))
    return LinearCorrection(slope, actual_mean - slope * pred_mean, len(pred))


def correct_oof(
    oof: pd.DataFrame, irregular: np.ndarray
) -> tuple[pd.DataFrame, dict[str, LinearCorrection]]:
    """Y_cat ごとに補正の式を求め、OOF 予測を補正する。

    式を求めるのも補正するのも、`irregular_flg` が 1 でない行（欠損の行は含む）だけ。1 の行は通常と
    違う値で、テストでは 0 に置き換えるので、補正せずにそのまま残す。

    Args:
        oof: OOF 予測（列 `Y_cat`, `actual`, `pred`, `fold`）。
        irregular: 各行の `irregular_flg`。

    Returns:
        (`oof` に次の列を加えた表, Y_cat → 補正の式（全部の OOF 予測から求めた、テストに使う式）)。

        - `pred_corrected`: その行の fold **以外** の行で求めた式で補正した値。補正後の MAE は
          この値で計算する（評価する行の実績値を、式を求めるのに使わない）。fold が1つしかなく
          式を求められないときは NaN。
        - `pred_corrected_fit`: テストに使う式（その行も含めて求めた式）で補正した値。参考。
    """
    actual = oof[CONFIG.ACTUAL_COL].to_numpy()
    pred = oof[CONFIG.PRED_COL].to_numpy()
    folds = oof["fold"].to_numpy()
    cats_of_rows = oof[CONFIG.CAT_COL].to_numpy()
    target = (folds >= 0) & (irregular != 1) & np.isfinite(actual) & np.isfinite(pred)
    crossfit, refit = pred.copy(), pred.copy()
    corrections = {}
    for category in dict.fromkeys(cats_of_rows):
        rows = target & (cats_of_rows == category)
        corrections[category] = fit_correction(pred[rows], actual[rows])
        refit[rows] = corrections[category].apply(pred[rows])
        for fold in np.unique(folds[rows]):
            inside, others = rows & (folds == fold), rows & (folds != fold)
            if others.any():
                crossfit[inside] = fit_correction(pred[others], actual[others]).apply(pred[inside])
            else:
                crossfit[inside] = np.nan
    return oof.assign(pred_corrected=crossfit, pred_corrected_fit=refit), corrections


def mae_or_nan(actual: np.ndarray, pred: np.ndarray) -> float:
    """MAE（元の尺度）。行が無いか、計算できない予測（NaN）があれば NaN にする。"""
    return scores_or_nan(actual, pred)["mae"] if np.isfinite(pred).all() else float("nan")


def mae_change_table(oof: pd.DataFrame, categories: list[str]) -> pd.DataFrame:
    """Y_cat ごとの、補正の前と後の MAE（元の尺度。上下限のクリップなどの前）の表。

    `補正後` は、評価する fold 以外の行で求めた式による値。`補正後（参考）` は、評価する行も含めて
    求めた式による値で、楽観的になる。改善量・改善率は正なら改善（MAE が小さくなった）。
    """
    rows = []
    for category in categories:
        part = oof[oof[CONFIG.CAT_COL] == category]
        actual = part[CONFIG.ACTUAL_COL].to_numpy()
        before, after, refit = (
            mae_or_nan(actual, part[column].to_numpy())
            for column in (CONFIG.PRED_COL, "pred_corrected", "pred_corrected_fit")
        )
        rows.append(
            {
                CONFIG.CAT_COL: category,
                "行数": len(part),
                "MAE 補正前": before,
                "MAE 補正後": after,
                "改善量": before - after,
                "改善率（%）": (before - after) / before * 100 if before else np.nan,
                "MAE 補正後（参考）": refit,
            }
        )
    return pd.DataFrame(rows).set_index(CONFIG.CAT_COL)


def corrected_categories(table: pd.DataFrame) -> list[str]:
    """テスト予測を補正する Y_cat（`CONFIG.CORRECTION_CATS`）を決める。

    Args:
        table: 全行の `mae_change_table` の結果（行が Y_cat）。

    Raises:
        ValueError: 学習データに無い Y_cat や、未対応の値を指定した場合。
    """
    categories = list(table.index)
    if CONFIG.CORRECTION_CATS is None:
        return categories
    if CONFIG.CORRECTION_CATS == "improved":
        # 補正後の MAE を計算できない（NaN の）Y_cat は補正しない
        return [c for c in categories if table.loc[c, "改善量"] > 0]
    if not isinstance(CONFIG.CORRECTION_CATS, (list, tuple)):
        raise ValueError(
            'CONFIG.CORRECTION_CATS は None・"improved"・Y_cat のリストのどれかです:'
            f" {CONFIG.CORRECTION_CATS}"
        )
    unknown = sorted(set(CONFIG.CORRECTION_CATS) - set(categories))
    if unknown:
        raise ValueError(f"CONFIG.CORRECTION_CATS に、学習データに無い Y_cat があります: {unknown}")
    return [c for c in categories if c in CONFIG.CORRECTION_CATS]


def mae_change_text(before: float, after: float) -> str:
    """補正の前と後の MAE を「前 → 後（改善量、改善率）」の文にする。"""
    if np.isnan(after):
        return f"MAE {before:.4f} → 計算できません（行が無い、または CV の fold が1つしか無い）"
    diff = before - after
    if diff == 0:
        return f"MAE {before:.4f} → {after:.4f}（変化なし）"
    word = "改善" if diff > 0 else "悪化"
    return f"MAE {before:.4f} → {after:.4f}（{abs(diff):.4f} {word}、{diff / before:+.2%}）"


def postprocess(
    predictions: pd.DataFrame,
    test_rows: pd.DataFrame,
    bounds: pd.DataFrame,
    corrections: dict[str, LinearCorrection],
) -> pd.DataFrame:
    """モデルの予測に後処理をかける（モデルには含めない）。

    1. 補正する Y_cat の予測値を、線形回帰の式で補正する（`irregular_flg` が 1 の行を除く）。
    2. カテゴリごとの上下限で、予測値をクリップする。
    3. テストデータの `irregular_flg` が 1 の行は、予測値を強制的に 0 にする（上下限より優先する）。
       `irregular_flg` が欠損の行は 0 にしない（1 と分からないため）。

    Args:
        predictions: モデルのテスト予測（列 `date`, `Y_cat`, `pred`）。
        test_rows: テストデータ（列 `date`, `Y_cat`, `irregular_flg`）。
        bounds: `target_bounds` の結果。
        corrections: 補正する Y_cat → 補正の式（無い Y_cat は補正しない）。

    Returns:
        列 `date`, `Y_cat`, `irregular_flg`, `pred_model`（モデルの予測）,
        `pred_corrected`（補正の後）, `pred`（後処理の後）,
        `clipped`（上下限でクリップした行）, `zeroed`（0 にした行）の表。

    Raises:
        ValueError: テストデータの行と予測の行が対応しない場合。
    """
    keys = [CONFIG.DATE_COL, CONFIG.CAT_COL]
    table = predictions[[*keys, CONFIG.PRED_COL]].merge(
        test_rows[[*keys, CONFIG.IRREGULAR_COL]],
        on=keys,
        how="outer",
        validate="1:1",
        indicator=True,
    )
    if (table["_merge"] != "both").any():
        raise ValueError("テストデータの行と、予測の行が対応しません")
    table = table.drop(columns="_merge").rename(columns={CONFIG.PRED_COL: "pred_model"})
    lower = table[CONFIG.CAT_COL].map(bounds.set_index(CONFIG.CAT_COL)["lower"]).fillna(-np.inf)
    upper = table[CONFIG.CAT_COL].map(bounds.set_index(CONFIG.CAT_COL)["upper"]).fillna(np.inf)
    zeroed = table[CONFIG.IRREGULAR_COL] == 1
    corrected = table["pred_model"]
    for category, correction in corrections.items():
        rows = (table[CONFIG.CAT_COL] == category) & ~zeroed
        corrected = corrected.mask(rows, correction.apply(table["pred_model"]))
    clipped_values = corrected.clip(lower=lower, upper=upper)
    return table.assign(
        **{
            "pred_corrected": corrected,
            CONFIG.PRED_COL: clipped_values.mask(zeroed, 0.0),
            "clipped": (clipped_values != corrected) & ~zeroed,
            "zeroed": zeroed,
        }
    )


def test_prediction_table(final: pd.DataFrame, categories: list[str]) -> pd.DataFrame:
    """後処理の後のテスト予測を、提出用の並びの表にする。

    列は `ID`（0から始まる連番）, `date`, `Y_cat`, `pred`。行は目的変数のカテゴリ、日付の順に昇順。
    カテゴリは `categories` の順（Y1, Y2, …, Y10。文字列の順では Y10 が Y1 の次になるため）。
    """
    order = {category: i for i, category in enumerate(categories)}
    table = final[[CONFIG.DATE_COL, CONFIG.CAT_COL, CONFIG.PRED_COL]].sort_values(
        [CONFIG.CAT_COL, CONFIG.DATE_COL],
        key=lambda s: s.map(order) if s.name == CONFIG.CAT_COL else s,
        kind="stable",
    )
    return table.reset_index(drop=True).rename_axis(CONFIG.ID_COL).reset_index()


def irregular_zero_rows(run: RegressionRun) -> np.ndarray:
    """学習に使った行のうち、`irregular_flg` が 0 の行（真偽値）。

    行順は `run.frame`・`run.oof` と同じ。

    欠損（NaN）の行は 0 と分からないので含めない。
    """
    return (run.frame[CONFIG.IRREGULAR_COL] == 0).to_numpy()


def scores_or_nan(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    """MAE・RMSE（元の尺度）。行が無ければ NaN にする。"""
    if len(y_true) == 0:
        return {"mae": np.nan, "rmse": np.nan}
    return {name: float(get_metric(name)(y_true, y_pred)) for name in ("mae", "rmse")}


# 何日先の予測かの区切り（日数の上限）。最後の区切りより先は1つにまとめる
HORIZON_EDGES = (1, 7, 28, 90, 180)


def horizon_labels(horizon: pd.Series) -> pd.Series:
    """何日先の予測かを、区切りの名前（`1日` / `2〜7日` / … / `181日〜`）にする。"""
    labels, low = [], 1
    for high in HORIZON_EDGES:
        labels.append(f"{low}日" if low == high else f"{low}〜{high}日")
        low = high + 1
    labels.append(f"{low}日〜")
    return pd.cut(horizon, [0, *HORIZON_EDGES, np.inf], labels=labels)


def recursion_table(oof: pd.DataFrame, by: pd.Series) -> pd.DataFrame:
    """再帰予測と1期先予測の MAE（元の尺度、後処理の前）を、`by` の値ごとに比べる表。

    Args:
        oof: OOF 予測（検証に入った行）。
        by: 分ける値（Y_cat・何日先かの区切りなど）。
    """
    rows = []
    for key, part in oof.groupby(by, sort=False, observed=True):
        actual = part[CONFIG.ACTUAL_COL].to_numpy()
        recursive = mae_or_nan(actual, part[CONFIG.PRED_COL].to_numpy())
        onestep = mae_or_nan(actual, part["pred_onestep"].to_numpy())
        rows.append(
            {
                by.name: key,
                "行数": len(part),
                "MAE 再帰予測": recursive,
                "MAE 1期先予測": onestep,
                "差（再帰 − 1期先）": recursive - onestep,
            }
        )
    return pd.DataFrame(rows).set_index(by.name)

### 図の部品と要約

In [ ]:
def save_figure(fig: plt.Figure, path: Path) -> Path:
    """図を保存して閉じる。"""
    fig.savefig(ensure_parent_dir(path), dpi=150, bbox_inches="tight")
    plt.close(fig)
    logger.info("図を保存: %s", shown(path))
    return path


def plot_shap_overview(table: pd.DataFrame, title: str, top_k: int = 15) -> plt.Figure:
    """SHAP 重要度の割合を、上位の特徴量 × 目的変数のヒートマップで描く。"""
    ensure_japanese_font()
    top = table.loc[table.mean(axis=1).sort_values(ascending=False).index[:top_k]]
    fig, ax = plt.subplots(
        figsize=(1.0 * top.shape[1] + 4, 0.4 * top_k + 2), constrained_layout=True
    )
    image = ax.imshow(top.to_numpy(), cmap="viridis", aspect="auto")
    ax.set_xticks(range(top.shape[1]), top.columns)
    ax.set_yticks(range(top.shape[0]), top.index)
    for i in range(top.shape[0]):
        for j in range(top.shape[1]):
            value = top.iat[i, j]
            ax.text(
                j,
                i,
                f"{value:.2f}",
                ha="center",
                va="center",
                fontsize=8,
                color="white" if value < top.to_numpy().max() * 0.6 else "black",
            )
    fig.colorbar(image, ax=ax, label="重要度の割合（平均|SHAP値|。種類ごとに合計1）")
    ax.set_title(f"{title}: 種類ごとの SHAP 重要度（上位{top_k}特徴量）")
    add_caption(fig, "0.00 には、その種類のモデルで使わない特徴量も含まれる。")
    return fig

In [ ]:
def plot_cv_mae(summary: pd.DataFrame, title: str) -> plt.Figure:
    """種類ごとの CV の相対MAE の棒グラフ。"""
    ensure_japanese_font()
    fig, ax = plt.subplots(figsize=(12, 5), constrained_layout=True)
    positions = np.arange(len(summary))
    ax.bar(positions - 0.2, summary["cv_relative_mae"], 0.4, label="全行")
    ax.bar(
        positions + 0.2,
        summary["cv_relative_mae_irregular0"],
        0.4,
        label=f"{CONFIG.IRREGULAR_COL} = 0 の行",
    )
    ax.set_xticks(positions, summary[CONFIG.CAT_COL])
    ax.set_ylim(bottom=0)  # 棒グラフは0起点
    ax.set_ylabel("相対MAE（MAE ÷ 実測値の平均）")
    ax.set_title(f"{title}: 種類ごとの CV の誤差")
    ax.legend(loc="upper left")
    add_caption(
        fig,
        "種類ごとに水準が違うため、MAE を実測値の平均で割って比べる（元の尺度）。"
        f"右の棒は {CONFIG.IRREGULAR_COL} が 0 の行だけで計算（MAE・実測値の平均とも）。",
    )
    return fig


def plot_training_overview(run: RegressionRun, title: str, ncols: int = 2) -> plt.Figure:
    """学習結果: 系列ごとに、学習期間の実測値と OOF 予測を点で並べて描く。"""
    ensure_japanese_font()
    nrows = int(np.ceil(len(run.categories) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(14, 2.6 * nrows), constrained_layout=True, sharex=True, squeeze=False
    )
    normal = irregular_zero_rows(run)
    for ax, category in zip(axes.flat, run.categories, strict=False):
        in_category = ((run.oof[CONFIG.CAT_COL] == category) & (run.oof["fold"] >= 0)).to_numpy()
        part = run.oof[in_category]
        normal_part = run.oof[in_category & normal]
        ax.scatter(part[CONFIG.DATE_COL], part[CONFIG.ACTUAL_COL], s=3, color="black", label="実測")
        ax.scatter(part[CONFIG.DATE_COL], part[CONFIG.PRED_COL], s=3, color="C3", label="OOF予測")
        score = scores_or_nan(part[CONFIG.ACTUAL_COL].to_numpy(), part[CONFIG.PRED_COL].to_numpy())
        normal_score = scores_or_nan(
            normal_part[CONFIG.ACTUAL_COL].to_numpy(), normal_part[CONFIG.PRED_COL].to_numpy()
        )
        ax.set_title(f"{category}（CV MAE={score['mae']:.2f} / flg=0: {normal_score['mae']:.2f}）")
    for ax in list(axes.flat)[len(run.categories) :]:
        ax.set_visible(False)
    axes.flat[0].legend(loc="upper left")
    fig.suptitle(f"{title}: 実測値と OOF 予測（系列ごと）")
    add_caption(
        fig,
        "予測は、その行を学習に使っていない fold のモデルの再帰予測。検証に入らない行は描かない。"
        f"「flg=0」は {CONFIG.IRREGULAR_COL} が 0 の行だけの CV MAE。",
    )
    return fig


def summary_frame(run: RegressionRun) -> pd.DataFrame:
    """種類ごとの CV の MAE をまとめた表（元の尺度）。

    `_irregular0` の付いた列は、`irregular_flg` が 0 の行だけで計算した値（相対MAE の分母も
    その行の実測値の平均）。
    """
    mae = get_metric("mae")
    actual_col, pred_col = CONFIG.ACTUAL_COL, CONFIG.PRED_COL
    normal = irregular_zero_rows(run)
    rows = []
    for category in run.categories:
        in_category = ((run.oof[CONFIG.CAT_COL] == category) & (run.oof["fold"] >= 0)).to_numpy()
        oof = run.oof[in_category]
        normal_oof = run.oof[in_category & normal]
        cv_mae = mae(oof[actual_col].to_numpy(), oof[pred_col].to_numpy())
        normal_mae = scores_or_nan(
            normal_oof[actual_col].to_numpy(), normal_oof[pred_col].to_numpy()
        )["mae"]
        row: dict[str, object] = {
            CONFIG.CAT_COL: category,
            "cv_oof_mae": cv_mae,
            "cv_relative_mae": cv_mae / float(oof[actual_col].mean()),
            "n_irregular0": len(normal_oof),
            "cv_oof_mae_irregular0": normal_mae,
            "cv_relative_mae_irregular0": normal_mae / float(normal_oof[actual_col].mean())
            if len(normal_oof)
            else np.nan,
        }
        if actual_col in run.predictions.columns:
            test = run.predictions[run.predictions[CONFIG.CAT_COL] == category]
            test_mae = mae(test[actual_col].to_numpy(), test[pred_col].to_numpy())
            row["test_mae"] = test_mae
            row["test_relative_mae"] = test_mae / float(test[actual_col].mean())
        rows.append(row)
    return pd.DataFrame(rows)


def plot_test_overview(run: RegressionRun, title: str, ncols: int = 2) -> plt.Figure:
    """テスト結果: 系列ごとに、テスト期間の予測（正解があれば実測も）を並べて描く。"""
    ensure_japanese_font()
    nrows = int(np.ceil(len(run.categories) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(14, 2.6 * nrows), constrained_layout=True, sharex=True, squeeze=False
    )
    date_col, actual_col, pred_col = CONFIG.DATE_COL, CONFIG.ACTUAL_COL, CONFIG.PRED_COL
    has_answer = actual_col in run.predictions.columns
    mae = get_metric("mae")
    for ax, category in zip(axes.flat, run.categories, strict=False):
        part = run.predictions[run.predictions[CONFIG.CAT_COL] == category]
        label = category
        if has_answer:
            ax.plot(part[date_col], part[actual_col], color="black", linewidth=0.8, label="実測")
            label += f"（MAE={mae(part[actual_col].to_numpy(), part[pred_col].to_numpy()):.2f}）"
        ax.plot(part[date_col], part[pred_col], color="C3", linewidth=1.2, label="予測")
        ax.set_title(label)
    for ax in list(axes.flat)[len(run.categories) :]:
        ax.set_visible(False)
    axes.flat[0].legend(loc="upper left")
    fig.suptitle(f"{title}: テスト期間の再帰予測（系列ごと、後処理の前）")
    return fig


def shap_importance_table(run: RegressionRun) -> pd.DataFrame:
    """系列ごとの SHAP 重要度（平均|SHAP値|の割合）を、特徴量 × 種類の表にする。

    OOF SHAP 値を行の系列（種類）で分けて平均する。系列ID（`Y_cat`）は `系列ID` に名前を変える。
    ワンホットの列（`Y_cat_Y1`・`X3_a` など）は、行ごとに SHAP 値を足し合わせて元の列1つにまとめる
    （SHAP 値は足し算で分解した値なので、足せばその列全体の寄与になる）。
    """
    values = pd.read_parquet(run.output_dir / "shap" / "shap_values.parquet")
    series = run.frame[CONFIG.CAT_COL].to_numpy()[values.pop("row").to_numpy()]
    # 元の列はモデルに渡す前の文字列の列（系列ID・カテゴリの説明変数）。名前の長い方から当てはめる
    # （`X1` と `X1_b` がどちらもカテゴリの列なら、`X1_b_c` は `X1_b` のワンホットの列）
    sources = sorted(run.categorical_cols, key=len, reverse=True)
    onehot: dict[str, list[str]] = {}
    for column in values.columns:
        source = next((s for s in sources if column.startswith(f"{s}_")), None)
        if column not in run.frame.columns and source is not None:
            onehot.setdefault(source, []).append(column)
    for source, columns in onehot.items():
        values = values.drop(columns=columns).assign(**{source: values[columns].sum(axis=1)})
    rename = {c: "系列ID" if c == CONFIG.CAT_COL else c for c in values.columns}
    importance = values.abs().groupby(series).mean().T.rename(index=rename)
    importance = importance[[c for c in run.categories if c in importance.columns]]
    return importance / importance.sum()


def save_outputs(
    run: RegressionRun, model: str, tracker_enabled: bool, log_path: Path | None = None
) -> pd.DataFrame:
    """要約の表・予測・図・ログを実験の出力ディレクトリに保存し、MLflow の同じ run にも記録する。"""
    out = run.output_dir
    title = f"x50_regression_016_{model}"
    summary = summary_frame(run)
    paths = [ensure_parent_dir(out / "summary.csv")]
    summary.to_csv(paths[0], index=False)
    paths.append(out / "test_predictions.csv")
    run.predictions.to_csv(paths[-1], index=False, date_format="%Y-%m-%d")
    paths.append(out / "shap_importance_by_target.csv")
    shap_table = shap_importance_table(run)
    shap_table.to_csv(paths[-1], index_label="feature")
    paths.append(ensure_parent_dir(out / "recursion" / "test_lag_features.csv"))
    run.test_lags.to_csv(paths[-1], index=False, date_format="%Y-%m-%d")
    figures = out / "figures"
    paths.append(save_figure(plot_cv_mae(summary, title), figures / "summary_mae.png"))
    paths.append(save_figure(plot_training_overview(run, title), figures / "training_overview.png"))
    paths.append(save_figure(plot_test_overview(run, title), figures / "test_overview.png"))
    paths.append(save_figure(plot_shap_overview(shap_table, title), figures / "shap_overview.png"))

    message = (
        f"要約: CV 相対MAE の平均 {float(summary['cv_relative_mae'].mean()):.4f}"
        f"（{CONFIG.IRREGULAR_COL} = 0 の行だけ:"
        f" {float(summary['cv_relative_mae_irregular0'].mean()):.4f}）"
    )
    if "test_relative_mae" in summary.columns:
        message += f" / テスト 相対MAE の平均 {float(summary['test_relative_mae'].mean()):.4f}"
    logger.info("%s（%d種類）。表: %s", message, len(run.categories), shown(paths[0]))
    if log_path is not None:
        for handler in logging.getLogger().handlers:
            handler.flush()
        paths.append(Path(shutil.copy(log_path, out / "run.log")))

    # run_experiment のrun（終了済み）に、種類ごとの指標と追加の出力を記録する
    if tracker_enabled and run.run_id is not None:
        import mlflow

        client = mlflow.tracking.MlflowClient()
        metrics = {
            "mean_cv_relative_mae": float(summary["cv_relative_mae"].mean()),
            "mean_cv_relative_mae_irregular0": float(summary["cv_relative_mae_irregular0"].mean()),
        }
        for row in summary.to_dict("records"):
            metrics[f"cv_oof_mae_{row[CONFIG.CAT_COL]}"] = float(row["cv_oof_mae"])
            if "test_mae" in row:
                metrics[f"test_mae_{row[CONFIG.CAT_COL]}"] = float(row["test_mae"])
        if "test_relative_mae" in summary.columns:
            metrics["mean_test_relative_mae"] = float(summary["test_relative_mae"].mean())
        for key, value in metrics.items():
            client.log_metric(run.run_id, key, value)
        for path in paths:
            client.log_artifact(run.run_id, str(path), artifact_path="summary")
        logger.info("MLflow に記録: run_id=%s", run.run_id)
    return summary


def residual_groups(oof: pd.DataFrame) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    """カテゴリ → (実測値, OOF 予測)。"""
    return {
        str(cat): (part[CONFIG.ACTUAL_COL].to_numpy(), part[CONFIG.PRED_COL].to_numpy())
        for cat, part in oof.groupby(CONFIG.CAT_COL, sort=False)
    }


def residual_summary_table(oof: pd.DataFrame) -> pd.DataFrame:
    """全体とカテゴリごとの残差の要約（元の尺度、後処理の前）。"""
    return pd.DataFrame(
        [
            {
                CONFIG.CAT_COL: "全体",
                **residual_summary(oof[CONFIG.ACTUAL_COL], oof[CONFIG.PRED_COL]),
            },
            *[
                {CONFIG.CAT_COL: cat, **residual_summary(*pair)}
                for cat, pair in residual_groups(oof).items()
            ],
        ]
    )


def save_residual_figures(
    run: RegressionRun,
    rows: np.ndarray,
    influence: pd.DataFrame | None,
    n_params: int,
    directory: Path,
    label: str,
    max_points: int,
) -> dict[str, Path]:
    """残差の図（分布・Q-Q・残差プロット・Leverage/Cook の距離）を保存する。

    OOF 予測（元の尺度、後処理の前）の、`rows` の行（学習の表の中の真偽値）で描く。
    Leverage・Cook の距離は、学習の途中で、各 Y_cat の各 fold のモデルの入力（前処理後の検証側）で
    計算した表（`CategoryResult.influence`）をつないで描く。

    Returns:
        図の名前（`residual_distribution` など）→ 保存したパス。行が無ければ空。
    """
    actual = run.oof[CONFIG.ACTUAL_COL].to_numpy()
    pred = run.oof[CONFIG.PRED_COL].to_numpy()
    selected = np.flatnonzero(
        (run.oof["fold"] >= 0).to_numpy() & np.isfinite(actual) & np.isfinite(pred) & rows
    )
    if selected.size == 0:
        logger.warning("%s の行が無いため、残差の図を作りません", label)
        return {}
    note = f"OOF予測（{label}。Y_cat ごとのモデル）n={selected.size:,}"
    y_rows, pred_rows = actual[selected], pred[selected]
    figures = {
        "residual_distribution": lambda: (
            ResidualDistributionDisplay.from_predictions(y_rows, pred_rows).figure_
        ),
        "qq_plot": lambda: QQPlotDisplay.from_predictions(y_rows, pred_rows).figure_,
        "residual_plot": lambda: (
            ResidualPlotDisplay.from_predictions(y_rows, pred_rows, max_points=max_points).figure_
        ),
    }
    paths = {}
    for name, draw in figures.items():
        fig = draw()
        add_caption(fig, f"{run.output_dir.parent.name}: {note}")
        paths[name] = save_figure(fig, directory / f"{name}.png")
    if influence is not None and len(influence):
        fig = InfluenceDisplay(influence, n_params).plot(max_points=max_points).figure_
        add_caption(
            fig,
            f"{run.output_dir.parent.name}: {note}。"
            "Leverageは各 Y_cat の各foldのモデルに入力した特徴量（前処理後）から計算した"
            "線形回帰の診断量で、非線形モデルでは影響の大きいサンプルの目安",
        )
        paths["leverage_cooks_distance"] = save_figure(
            fig, directory / "leverage_cooks_distance.png"
        )
    return paths


def combined_influence(run: RegressionRun, key: str) -> tuple[pd.DataFrame | None, int]:
    """全 Y_cat の Leverage・Cook の距離の表と、ランクの平均（Y_cat の間の平均）。"""
    parts = [getattr(r, key) for r in run.results.values() if getattr(r, key) is not None]
    if not parts:
        return None, 0
    ranks = [r.n_params for r in run.results.values() if r.n_params > 0]
    return pd.concat(parts, ignore_index=True), int(round(float(np.mean(ranks)))) if ranks else 0


def plot_mae_change(tables: dict[str, pd.DataFrame], title: str) -> plt.Figure:
    """Y_cat ごとの、補正の前と後の MAE の棒グラフ（行の範囲ごとに1枚）。

    Args:
        tables: 行の範囲の名前 → `mae_change_table` の結果。
        title: 図の題。
    """
    ensure_japanese_font()
    fig, axes = plt.subplots(
        1, len(tables), figsize=(14, 4), constrained_layout=True, sharey=True, squeeze=False
    )
    for ax, (label, table) in zip(axes.flat, tables.items(), strict=True):
        positions = np.arange(len(table))
        for offset, name in ((-0.2, "補正前"), (0.2, "補正後")):
            ax.bar(positions + offset, table[f"MAE {name}"], 0.4, label=name)
        ax.set_xticks(positions, table.index)
        ax.set_ylim(bottom=0)  # 棒グラフは0起点
        ax.set_xlabel(CONFIG.CAT_COL)
        ax.set_title(label)
    axes.flat[0].set_ylabel("OOF MAE（元の尺度）")
    axes.flat[0].legend(loc="upper left")
    fig.suptitle(f"{title}: 線形回帰による補正の前と後の MAE")
    add_caption(
        fig,
        "補正後は、評価する fold 以外の行で求めた式で補正した値（全 Y_cat を補正した場合）。"
        "上下限のクリップなどの前。",
    )
    return fig


def plot_correction(
    oof: pd.DataFrame,
    irregular: np.ndarray,
    corrections: dict[str, LinearCorrection],
    title: str,
    ncols: int = 5,
) -> plt.Figure:
    """Y_cat ごとの、OOF 予測と実績値の散布図に、補正の式（回帰直線）を重ねて描く。"""
    ensure_japanese_font()
    nrows = int(np.ceil(len(corrections) / ncols))
    fig, axes = plt.subplots(
        nrows, ncols, figsize=(3.2 * ncols, 3.2 * nrows), constrained_layout=True, squeeze=False
    )
    target = oof[((oof["fold"] >= 0) & (irregular != 1)).to_numpy()]
    for ax, (category, correction) in zip(axes.flat, corrections.items(), strict=False):
        part = target[target[CONFIG.CAT_COL] == category]
        ax.scatter(part[CONFIG.PRED_COL], part[CONFIG.ACTUAL_COL], s=3, color="C0", alpha=0.4)
        if len(part):
            ends = np.array([part[CONFIG.PRED_COL].min(), part[CONFIG.PRED_COL].max()])
            ax.plot(
                ends, ends, color="black", linestyle="--", linewidth=1, label="補正なし（y = x）"
            )
            ax.plot(ends, correction.apply(ends), color="C3", linewidth=1.5, label="補正の式")
        ax.set_title(
            f"{category}: {correction.intercept:.2f} + {correction.slope:.3f} × 予測", fontsize=10
        )
    for ax in list(axes.flat)[len(corrections) :]:
        ax.set_visible(False)
    for ax in axes[:, 0]:
        ax.set_ylabel("実績値")
    for ax in axes[-1, :]:
        ax.set_xlabel("OOF 予測")
    axes.flat[0].legend(loc="upper left", fontsize=8)
    fig.suptitle(f"{title}: OOF 予測と実績値の線形回帰（Y_cat ごと）")
    add_caption(
        fig,
        f"n={len(target):,}。{CONFIG.IRREGULAR_COL} が 1 の行は除く。元の尺度、後処理の前。"
        "補正しない Y_cat の式も描く。",
    )
    return fig


def save_correction_outputs(
    run: RegressionRun,
    oof: pd.DataFrame,
    table: pd.DataFrame,
    metrics: dict[str, float],
    figures: dict[str, plt.Figure],
    tracker_enabled: bool,
) -> dict[str, Path]:
    """補正の式・MAE の表、補正後の OOF 予測、図を保存し、MLflow の同じ run にも記録する。

    Args:
        run: 学習の結果。
        oof: `correct_oof` の結果。
        table: 補正の式と MAE の表（行が Y_cat）。
        metrics: MLflow に記録する指標（NaN のものは記録しない）。
        figures: 図の名前 → 図。
        tracker_enabled: MLflow に記録するか。

    Returns:
        図の名前 → 保存したパス。
    """
    directory = run.output_dir / "correction"
    table_path = ensure_parent_dir(directory / "correction_by_category.csv")
    table.to_csv(table_path)
    oof.to_csv(directory / "oof_corrected.csv", index=False, date_format="%Y-%m-%d")
    figure_paths = {
        name: save_figure(fig, directory / f"{name}.png") for name, fig in figures.items()
    }
    if tracker_enabled and run.run_id is not None:
        import mlflow

        client = mlflow.tracking.MlflowClient()
        for key, value in metrics.items():
            if np.isfinite(value):
                client.log_metric(run.run_id, key, value)
        for path in (table_path, *figure_paths.values()):
            client.log_artifact(run.run_id, str(path), artifact_path="correction")
    logger.info("補正の式と MAE を保存: %s", shown(table_path))
    return figure_paths

In [ ]:
# ログをファイル（INFO 以上）と画面（CONFIG.LOG_LEVEL 以上）に出す
stamp = dt.datetime.now().strftime("%Y%m%d_%H%M%S")
log_name = f"regression_x50_016_{CONFIG.MODEL}_{stamp}_notebook.log"
log_path = setup_logging(
    outputs_dir() / "logs" / log_name, console_level=getattr(logging, CONFIG.LOG_LEVEL)
)
output_root = None if CONFIG.OUTPUT_ROOT is None else Path(CONFIG.OUTPUT_ROOT)
print(f"ログ: {shown(log_path)}")

# 前半: 前処理

使う列と行を決めて、日付の特徴量を作る。目的変数のラグ・移動平均も、学習データではここで（前日までの実測値から）作る。

## 1. 読み込みと確認

どれかの Y_cat が使う説明変数（`CONFIG.FEATURE_COLS`・`CONFIG.FEATURE_COLS_BY_CAT`）だけを CSV から読む。特徴量（説明変数・`irregular_flg`）の欠損は埋めず、
そのままモデルに渡す（木のモデルは欠損をそのまま扱い、線形モデルはモデルの Pipeline の中で学習側の中央値で補う。スケーリングも欠損を残したまま行う。±inf は `ToModelInput` がモデルに渡す直前に欠損にする）。フラグが int 型の 0/1 であることを確かめ、学習データの
`eliminate_flg` が 1 の行を除く（以降の前処理・学習のどちらにも入らない。除く前に、ラグ・移動平均の元になる目的変数の系列だけを取り出す）。`irregular_flg` は特徴量として残す。

学習・テストのどちらかで数値として読めない説明変数（または `CONFIG.CATEGORICAL_COLS` の列）は、カテゴリの説明変数として
CSV の文字のまま読み直す。エンコードはモデルの Pipeline の中で、各 fold の学習側だけで fit する。下の2つ目の表は、
カテゴリの説明変数ごとのカテゴリ数・欠損・学習に無いカテゴリ（テストで初めて出る値）の行数。

In [ ]:
train_path = data_dir() / CONFIG.RAW_TRAIN_PATH
test_path = data_dir() / CONFIG.RAW_TEST_PATH
output_dir = data_dir() / CONFIG.FEATURES_DIR
# 出力先が生データ（data/raw・data/external）でないことを確かめる（生データは書き換えない）
for immutable in ("raw", "external"):
    if output_dir.resolve().is_relative_to((data_dir() / immutable).resolve()):
        raise ValueError(f"data/{immutable} には書き込めません: {output_dir}")

# Y_cat ごとの指定から、どれかの Y_cat が使う説明変数を決めて、その列だけを読む
all_cats = list(
    pd.read_csv(train_path, usecols=[CONFIG.CAT_COL], encoding=csv_encoding(train_path))[
        CONFIG.CAT_COL
    ].unique()
)
read_cols = columns_to_read(
    all_cats, [c for c in all_cats if CONFIG.TARGET_CATS is None or c in CONFIG.TARGET_CATS]
)
train_raw = read_columns(train_path, read_cols, train=True)
test_raw = read_columns(test_path, read_cols, train=False)
# 説明変数（irregular_flg 以外）と、モデルの特徴量にする列（irregular_flg + 説明変数）
x_cols = [
    c for c in test_raw.columns if c not in (CONFIG.DATE_COL, CONFIG.CAT_COL, CONFIG.IRREGULAR_COL)
]
feature_cols = [CONFIG.IRREGULAR_COL, *x_cols]
# カテゴリの説明変数は文字のまま読み直す（行を絞り込む前に、CSV の行の順のまま置き換える）
categorical_cols = categorical_columns(train_raw, test_raw, x_cols)
train_raw = read_as_text(train_path, train_raw, categorical_cols)
test_raw = read_as_text(test_path, test_raw, categorical_cols)
numeric_x_cols = [c for c in x_cols if c not in categorical_cols]
if CONFIG.TARGET_CATS is not None:
    train_raw = train_raw[train_raw[CONFIG.CAT_COL].isin(CONFIG.TARGET_CATS)].reset_index(drop=True)
    test_raw = test_raw[test_raw[CONFIG.CAT_COL].isin(CONFIG.TARGET_CATS)].reset_index(drop=True)

# ラグ・移動平均の元になる、Y_cat ごとの目的変数の系列（irregular_flg が 1 の日などは NaN）。
# 日付を詰めずに数えるため、eliminate_flg が 1 の行を除く前の表から作る
lag_spec = LagSpec.from_config()
histories = target_histories(train_raw)
train_used = drop_eliminated(train_raw)
test_used = test_raw.sort_values(CONFIG.DATE_COL, kind="stable").reset_index(drop=True)
# 特徴量の欠損は埋めず、そのままモデルに渡す。
# 学習データで全行が欠損の列はモデルが使えないので知らせる。
# そうした列があると集計で NumPy の警告（All-NaN slice）が出続けるので、その警告は出さない
all_missing = [c for c in feature_cols if train_used[c].isna().all()]
if all_missing:
    print(f"注意: 学習データで全行が欠損の特徴量があります（モデルは使えません）: {all_missing}")
warnings.filterwarnings("ignore", message="All-NaN slice encountered", category=RuntimeWarning)
cats = list(dict.fromkeys(train_raw[CONFIG.CAT_COL]))
if CONFIG.DETAIL_TARGET not in cats:
    raise ValueError(f"CONFIG.DETAIL_TARGET は {cats} から選んでください: {CONFIG.DETAIL_TARGET}")
shown_cols = x_cols[:10]
print(
    f"説明変数（どれかの Y_cat が使う列）: {len(x_cols)}列"
    f" {shown_cols}{' …' if x_cols != shown_cols else ''}"
    f" + {CONFIG.IRREGULAR_COL}"
)
print(f"うちカテゴリの説明変数: {len(categorical_cols)}列 {categorical_cols}")
if categorical_cols:
    categories_table = category_summary(train_used, test_used, categorical_cols)
    display(categories_table.round(3))
    # 数値の列に文字が混ざった列と、カテゴリが多すぎる列を知らせる
    # （ワンホットの列が増えすぎ、学習がとても遅くなる）
    # CONFIG.CATEGORICAL_COLS で指定した列（数値のコードをカテゴリとして扱う列）は知らせない
    suspicious = [
        c
        for c in categories_table.index[categories_table["数値の割合"] >= 0.9]
        if c not in (CONFIG.CATEGORICAL_COLS or [])
    ]
    if suspicious:
        print(
            f"注意: 値のほとんどが数値なのにカテゴリと判定した列があります: {suspicious}。"
            "数値の列なら、CSV の文字の値を欠損（空）に直す"
            "か、CONFIG.FEATURE_COLS で除いてください"
        )
    many = categories_table.index[categories_table["学習のカテゴリ数"] > 100].tolist()
    if many:
        print(
            f"注意: カテゴリが 100 を超える列があります: {many}。"
            "ワンホットでは列がその数だけ増えます（CONFIG.ONEHOT_MAX_CATEGORIES で抑えられる）"
        )
pd.DataFrame(
    {
        "読み込んだ行数": [len(train_raw), len(test_raw)],
        f"{CONFIG.ELIMINATE_COL} = 1 で除いた行数": [int(train_raw[CONFIG.ELIMINATE_COL].sum()), 0],
        "使う行数": [len(train_used), len(test_used)],
        f"うち {CONFIG.IRREGULAR_COL} = 1 の行数": [
            int(train_used[CONFIG.IRREGULAR_COL].sum()),
            int(test_used[CONFIG.IRREGULAR_COL].sum()),
        ],
        "最初の日": [f"{f[CONFIG.DATE_COL].min():%Y-%m-%d}" for f in (train_used, test_used)],
        "最後の日": [f"{f[CONFIG.DATE_COL].max():%Y-%m-%d}" for f in (train_used, test_used)],
    },
    index=["学習", "テスト"],
)

## 2. 前処理の定義

| 対象 | 前処理 | 既定 | fit するデータ |
|---|---|---|---|
| 日付 | 成分の取り出し（`DatetimeFeaturesExtractor`）・周期特徴量（`CyclicalFeaturesEncoder`） | 行う | 不要 |
| 日付 | 経過日数（`ElapsedTimeTransformer`。基準日 = 学習データの初日） | 行う | 学習データ |
| 説明変数 | `CONFIG.FEATURE_SCALING`: Y_cat ごとの標準化・正規化（`GroupScaler`）。対象の列は `CONFIG.FEATURE_SCALING_COLS` | **行わない** | 各 fold の学習側（Pipeline の先頭） |
| カテゴリの説明変数 | `CONFIG.CAT_ENCODING`: 整数化（`PandasOrdinalEncoder`）またはワンホット（`PandasOneHotEncoder`） | 行う | 各 fold の学習側（Pipeline の中） |
| 目的変数のラグ・移動平均 | `CONFIG.LAGS`・`CONFIG.ROLLING_WINDOWS`（`LagSpec`）。前日までの値（元の尺度）から作る。`irregular_flg` が 1 の日は NaN | 行う | 不要（CV では、fold ごとに検証期間の実測値を消して作り直す。`train_category` の中） |
| 目的変数 | `CONFIG.TARGET_TRANSFORM`: Y_cat ごとの標準化・正規化、または対数 | **行わない** | 各 fold の学習側（`eliminate_flg` が 1 の行は除く。`train_cv` の中） |

テストデータのラグ・移動平均は、予測しながら作るので、ここでは欠損のままにする。ラグ・移動平均は元の尺度の値で、
`CONFIG.TARGET_TRANSFORM` を指定しても変換しない（スケーリングするときは `FEATURE_SCALING_COLS` に列名を書く）。
`irregular_flg` はそのまま特徴量にする（`FEATURE_SCALING_COLS` で指定しない限りスケーリングしない）。カテゴリの説明変数もスケーリングしない。説明変数のスケーリングと目的変数の変換は、CV の fold ごとに fit するので、
前処理の CSV の値は変換前のままになる。

In [ ]:
class Preprocessor:
    """日付の特徴量を作る前処理（学習データだけで fit する）。"""

    def fit(self, train: pd.DataFrame) -> "Preprocessor":
        """経過日数の基準日を、学習データで決める。"""
        self.elapsed = ElapsedTimeTransformer(CONFIG.DATE_COL, unit="days").fit(
            self._date_features(train)
        )
        return self

    def transform(self, frame: pd.DataFrame) -> pd.DataFrame:
        """日付の特徴量を加える（各行の日付だけから計算する）。"""
        return self.elapsed.transform(self._date_features(frame))

    @staticmethod
    def _date_features(frame: pd.DataFrame) -> pd.DataFrame:
        """日付の成分と周期特徴量を加える（学習は不要）。"""
        frame = DatetimeFeaturesExtractor(
            CONFIG.DATE_COL, components=CONFIG.DATE_COMPONENTS
        ).fit_transform(frame)
        if "month" in CONFIG.DATE_COMPONENTS:
            frame = CyclicalFeaturesEncoder(f"{CONFIG.DATE_COL}_month", period=12).fit_transform(
                frame
            )
        if "weekday" in CONFIG.DATE_COMPONENTS:
            frame = CyclicalFeaturesEncoder(f"{CONFIG.DATE_COL}_weekday", period=7).fit_transform(
                frame
            )
        return frame

## 3. 前処理の実行と出力

In [ ]:
prep = Preprocessor().fit(train_used)
train_features = add_lag_features(prep.transform(train_used), histories, lag_spec)
# テストのラグ・移動平均は予測しながら作るので、ここでは欠損の列だけを用意する
test_features = prep.transform(test_used).assign(**dict.fromkeys(lag_spec.names, np.nan))
date_feature_cols = [c for c in test_features.columns if c.startswith(f"{CONFIG.DATE_COL}_")]
lag_cols = lag_spec.names
exog_cols = [*date_feature_cols, *feature_cols, *lag_cols]
print(f"日付の特徴量: {date_feature_cols}")
print(f"目的変数のラグ・移動平均: {lag_cols}")
excluded_days = {cat: int(history.isna().sum()) for cat, history in histories.items()}
print(f"ラグ・移動平均に使わない日（NaN）の数: {excluded_days}")
# Y_cat ごとのモデルの特徴量（日付の特徴量・irregular_flg・説明変数・ラグと移動平均から選ぶ）
features_by_cat = category_features_map(all_cats, cats, date_feature_cols, x_cols, lag_cols)
print(
    f"表の特徴量: {len(exog_cols)}列（日付 + {CONFIG.IRREGULAR_COL} + 説明変数 + ラグ・移動平均）"
)
print(f"Y_cat ごとの特徴量の数: { ({cat: len(cols) for cat, cols in features_by_cat.items()}) }")
for filename, frame in (
    ("x50_train_features.csv", train_features),
    ("x50_test_features.csv", test_features),
):
    path = ensure_parent_dir(output_dir / filename)
    frame.to_csv(path, index=False, date_format="%Y-%m-%d")
    logger.info("前処理済みの CSV を保存: %s（%d行 × %d列）", shown(path), *frame.shape)
train_features.head()

## 4. 前処理のリークの確認

1. **`eliminate_flg` が 1 の行は使われない:** その行の説明変数を書き換えても、前処理の結果は変わらない
   （目的変数はラグに使うので書き換えない。`LAG_USE_ELIMINATED = False` のときは目的変数も書き換える）。
2. **テストデータは学習側の結果に影響しない:** テストデータを変えて前処理をやり直しても、学習データの結果は変わらない。
3. **`irregular_flg` が 1 の日はラグに使われない:** その日の目的変数を書き換えても、ラグ・移動平均は変わらない。
4. **ラグ・移動平均の値:** 読み込んだデータから pandas の `shift`・`rolling` で別に計算した値と一致する（日付で数え、
   前日までの値だけを使い、`irregular_flg` が 1 の日は NaN）。

In [ ]:
# 1. eliminate_flg が 1 の行を書き換えても、前処理の結果は同じ
changed_raw = train_raw.copy()
eliminated = changed_raw[CONFIG.ELIMINATE_COL] == 1
# 数値の説明変数は大きな値に、カテゴリの説明変数は学習に無いカテゴリに書き換える
if numeric_x_cols:
    changed_raw[numeric_x_cols] = changed_raw[numeric_x_cols].astype(float)
    changed_raw.loc[eliminated, numeric_x_cols] = 1e6
if categorical_cols:
    changed_raw.loc[eliminated, categorical_cols] = "__changed__"
if not CONFIG.LAG_USE_ELIMINATED:
    changed_raw.loc[eliminated, CONFIG.TARGET_COL] = 1e6
changed_used = drop_eliminated(changed_raw)
pd.testing.assert_frame_equal(
    add_lag_features(
        Preprocessor().fit(changed_used).transform(changed_used),
        target_histories(changed_raw),
        lag_spec,
    ),
    train_features,
)
rewritten_part = "説明変数" if CONFIG.LAG_USE_ELIMINATED else "説明変数・目的変数"
print(
    f"OK 1: eliminate_flg が 1 の {int(eliminated.sum())}行の{rewritten_part}を書き換えても、"
    "前処理の結果は同じ"
)

# 2. 前処理の fit はテストデータを見ない
base_cols = [c for c in train_features.columns if c not in lag_cols]
pd.testing.assert_frame_equal(
    Preprocessor().fit(train_used).transform(train_used), train_features[base_cols]
)
assert prep.transform(test_used).equals(test_features.drop(columns=lag_cols))
assert test_features[lag_cols].isna().all().all(), "テストのラグ・移動平均が欠損ではありません"
print("OK 2: テストデータは学習データの前処理結果に影響しない")

# 3. irregular_flg が 1 の日の目的変数を書き換えても、ラグ・移動平均は同じ
irregular_rows = train_raw[CONFIG.IRREGULAR_COL] == 1
changed_raw = train_raw.assign(
    **{CONFIG.TARGET_COL: train_raw[CONFIG.TARGET_COL].mask(irregular_rows, 1e6)}
)
pd.testing.assert_frame_equal(
    add_lag_features(train_used, target_histories(changed_raw), lag_spec)[lag_cols],
    train_features[lag_cols],
)
print(
    f"OK 3: {CONFIG.IRREGULAR_COL} が 1 の {int(irregular_rows.sum())}行の目的変数を書き換えても、"
    "ラグ・移動平均は同じ"
)

# 4. pandas の shift・rolling で別に計算した値と一致する
for cat in cats:
    raw_cat = train_raw[train_raw[CONFIG.CAT_COL] == cat].set_index(CONFIG.DATE_COL)
    usable = raw_cat[CONFIG.IRREGULAR_COL] != 1
    if not CONFIG.LAG_USE_ELIMINATED:
        usable &= raw_cat[CONFIG.ELIMINATE_COL] != 1
    daily = raw_cat[CONFIG.TARGET_COL].astype(float).where(usable).sort_index().asfreq("D")
    expected = pd.DataFrame(
        {f"{CONFIG.TARGET_COL}_lag_{k}": daily.shift(k) for k in CONFIG.LAGS}
        | {
            f"{CONFIG.TARGET_COL}_ma_{w}": daily.shift(1)
            .rolling(w, min_periods=min(CONFIG.ROLLING_MIN_PERIODS, w))
            .mean()
            for w in CONFIG.ROLLING_WINDOWS
        },
        index=daily.index,
    )
    part = train_features[train_features[CONFIG.CAT_COL] == cat]
    np.testing.assert_allclose(
        part[lag_cols].to_numpy(dtype=float),
        expected.reindex(part[CONFIG.DATE_COL])[lag_cols].to_numpy(),
        rtol=1e-9,
        equal_nan=True,
    )
print("OK 4: ラグ・移動平均は、pandas で別に計算した値と一致する（全 Y_cat）")
# 確認に使った表は消す（メモリの節約）
del changed_raw, changed_used

# 後半: 学習・予測（Y_cat ごとの個別モデル）と後処理

## 5. 読み込みとリークの確認

学習・テストの表（`model_frames`）を作り、前処理の途中の表は消す（メモリの節約）。後で使う列
（`eliminate_flg` が 1 の行の日付と Y_cat、テストの `irregular_flg`）だけを小さい表に残す。

In [ ]:
memory_before = memory_mb()
train_model, test_model = model_frames(train_features, test_features, exog_cols)
n_missing_target = len(train_features) - len(train_model)
# 前処理の途中の表は使い終わったので消す（メモリの節約）。後で使う列だけを小さい表に残す
eliminated_keys = train_raw.loc[
    train_raw[CONFIG.ELIMINATE_COL] == 1, [CONFIG.DATE_COL, CONFIG.CAT_COL]
].reset_index(drop=True)
test_flags = test_raw[[CONFIG.DATE_COL, CONFIG.CAT_COL, CONFIG.IRREGULAR_COL]].copy()
n_test_rows = len(test_raw)
del train_raw, test_raw, train_used, test_used, train_features, test_features
release_memory()
table_mb = sum(f.memory_usage(deep=True).sum() for f in (train_model, test_model)) / 2**20
print(
    f"学習・テストの表: {table_mb:.1f} MB"
    f"（説明変数の型: {CONFIG.FEATURE_DTYPE or '読み込んだまま'}）"
    f" / プロセスのメモリ: {memory_before:.0f} MB → {memory_mb():.0f} MB"
)
print(f"学習に使う行: {len(train_model)}行（目的変数が欠損の {n_missing_target}行を除く）")
print(f"予測する行: {len(test_model)}行")
# 種類の列（Y_cat）は目的変数の値ではないので除く。ラグ・移動平均の列は、予測するまで全部欠損
leaked = [
    c
    for c in test_model.columns
    if c not in (CONFIG.CAT_COL, *lag_cols)
    and (c == CONFIG.TARGET_COL or c.startswith(f"{CONFIG.TARGET_COL}_"))
]
leaked += [c for c in lag_cols if test_model[c].notna().any()]
if leaked:
    raise LeakageError(f"テストデータに目的変数（またはその特徴量）があります: {leaked}")
print(f"目的変数の種類: {cats}")
train_model.groupby(CONFIG.CAT_COL, sort=False)[CONFIG.TARGET_COL].describe()[
    ["count", "mean", "std", "min", "max"]
].round(2)

## 6. 実験設定と Y_cat ごとの特徴量

Y_cat ごとの実験設定を作る（違うのは特徴量だけで、他は共通）。下の表は、Y_cat ごとの行数と、使う特徴量の数・
使わない特徴量（どれかの Y_cat が使う特徴量のうち、その Y_cat では使わないもの）。学習用データ一式（CV の分割を含む）は、メモリを節約するため、
「7. 学習とテスト予測」で Y_cat ごとに作って、その Y_cat が終わったら捨てる（CV の分割のリークの確認
（`prepare`）も、そのときに Y_cat ごとに行う）。

In [ ]:
# スケーリングする列の指定は全 Y_cat 共通。全部の特徴量を使う設定を作って、指定を確かめる
config = make_config(
    CONFIG.MODEL,
    exog_cols,
    CONFIG.CV_METHOD,
    CONFIG.N_SPLITS,
    CONFIG.CUTOFFS,
    CONFIG.TEST_PREDICTION,
    CONFIG.FEATURE_SCALING,
    CONFIG.FEATURE_SCALING_COLS or numeric_x_cols,
    objective=CONFIG.OBJECTIVE,
    objective_alpha=CONFIG.OBJECTIVE_ALPHA,
    model_params=CONFIG.MODEL_PARAMS,
    categorical_cols=categorical_cols,
)
scaling_cols = scaled_feature_cols(config)
# 実際に学習に使う、Y_cat ごとの設定（その Y_cat が使う特徴量だけ）
configs = category_configs(features_by_cat, scaling_cols, categorical_cols)
print(f"実験名: {config.name} / タスク: {config.task} / モデル: {config.model.name}（Y_cat ごと）")
print(f"CV: {CONFIG.CV_METHOD}（Y_cat ごとに分割）")
print(
    f"目的変数の変換: {CONFIG.TARGET_TRANSFORM}"
    f" / 説明変数のスケーリング（Y_cat ごと）: {CONFIG.FEATURE_SCALING}"
)
if scaling_cols:
    print(
        f"スケーリングする列: {len(scaling_cols)}列 {scaling_cols[:10]}"
        f"{' …' if len(scaling_cols) > 10 else ''}"
    )
if any(len(cols) < len(exog_cols) for cols in features_by_cat.values()):
    print("特徴量: Y_cat ごとに選択（下の表。Y_cat は使わない）")
else:
    print(f"特徴量: 全 Y_cat で {len(exog_cols)}列（Y_cat は使わない）")
print(f"目的関数: {CONFIG.OBJECTIVE}（alpha={CONFIG.OBJECTIVE_ALPHA}）")
print(f"カテゴリの説明変数のエンコーディング: {category_encoding(CONFIG.MODEL)}")
print(f"モデルのパラメータ: {config.model.params}")
print(
    f"裾の重み付け: {CONFIG.TAIL_QUANTILES}（重み {CONFIG.TAIL_WEIGHT}）"
    f" / Optuna の試行回数: {CONFIG.N_TRIALS}（Y_cat ごと）"
)
pd.DataFrame(
    {
        "学習の行数": train_model[CONFIG.CAT_COL].value_counts(sort=False),
        "テストの行数": test_model[CONFIG.CAT_COL].value_counts(sort=False),
        "特徴量の数": {cat: len(cols) for cat, cols in features_by_cat.items()},
        "使わない特徴量": {
            cat: ", ".join(c for c in exog_cols if c not in cols) or "なし"
            for cat, cols in features_by_cat.items()
        },
    }
).loc[cats]

## 7. 学習とテスト予測（Y_cat ごと）

Y_cat を1つずつ順に、その Y_cat の行だけで CV・学習・テスト予測を行う。

- **CV:** fold ごとに、検証期間の実測値を消した系列から学習側のラグ・移動平均を作って学習し、検証期間を再帰予測する
  （OOF 予測）。
- **テスト予測:** 学習期間の全部の実測値を履歴にして、テスト期間を1日ずつ再帰予測する。`TEST_PREDICTION = "fold_mean"` では、
  fold のモデルごとに（自分の予測値をラグに使って）再帰予測し、その平均をとる。
- **`irregular_flg` が 1 の日:** 予測はするが、履歴には予測値の代わりに NaN を加える（最終的な予測は後処理で 0 にする）。

メモリを節約するため、次のようにする。

| 方法 | 効果 |
|---|---|
| Y_cat を1つずつ学習し、終わった Y_cat の学習用データ・モデルはすぐに捨てる | メモリにあるのは、学習中の1つの Y_cat の分と、終わった Y_cat の小さい結果（予測・SHAP の一部など）だけ |
| 各 fold のモデルは、分析（スケーリングの確認・SHAP・Leverage・学習の推移）とテスト予測を済ませたら捨てる | 同時にメモリにあるモデルは最大1つ（013 は fold の数 + 1 のモデルを残していた） |
| `fold_mean` のテスト予測は、fold ごとに足し合わせる | テスト予測のために fold のモデルを残さない |
| Optuna の試行ではモデル・学習の推移を残さない | 試行の数だけモデルが溜まらない |
| SHAP の行は Y_cat ごとに `SHAP_MAX_SAMPLES_PER_CAT` 行まで | SHAP の値・特徴量の表が大きくならない |
| 説明変数を `float32` で持つ（`FEATURE_DTYPE`） | 学習・テストの表のメモリが半分 |
| 使い終わったら `gc.collect()`（Linux では `malloc_trim` で OS に返す） | 解放したメモリを、次の Y_cat の学習で使える |

モデルを後で使うときは `SAVE_MODELS = True` にして、ファイル（joblib）に保存する。表の「メモリ」は、その Y_cat を終えて
モデルを捨てた後のプロセスのメモリ（psutil が無い環境では NaN）。

In [ ]:
tracker = make_tracker(CONFIG.USE_TRACKING)
run = run_training(
    configs,
    features_by_cat,
    train_model,
    test_model,
    categorical_cols,
    tracker,
    output_root,
    histories,
)
summary = save_outputs(run, CONFIG.MODEL, CONFIG.USE_TRACKING, log_path)
out = run.output_dir
print(f"出力: {shown(out)}")
pd.DataFrame(
    [
        {
            CONFIG.CAT_COL: r.category,
            "学習の行数": len(r.rows),
            "テストの行数": len(r.test_rows),
            "特徴量の数": len(run.features[r.category]),
            "OOF MAE": r.oof_scores["mae"],
            "秒": r.seconds,
            "メモリ（MB）": r.memory_mb,
            "学習中の最大（MB、測った時点）": r.peak_memory_mb,
        }
        for r in run.results.values()
    ]
).round(2)

## 予測のリークの確認

モデルを残さないので、1・2 はテストを予測したとき（学習の途中）に、Y_cat ごとに確かめている（確かめられなければ
学習がエラーで止まる）。ここでは、その結果を表示する。

1. **予測のやり直し:** 再帰予測に使った特徴量（ラグ・移動平均を含む）をまとめて渡して予測し直した値が、テスト予測と一致する。
2. **後の日を使わない:** テスト期間のある1日の説明変数を書き換えても、それより前の日の予測は変わらない（後の日の予測は、
   ラグを通して変わる）。
3. **`eliminate_flg`:** 1 の行が学習に入っていない。テストデータは全行を予測している。

In [ ]:
check_table = pd.DataFrame({cat: r.checks for cat, r in run.results.items()}).T
display(check_table)
assert check_table.notna().all().all(), "確認していない Y_cat があります"
print(f"OK 1・2: 全 {len(run.results)} 種類の Y_cat のモデルで確かめた")

# eliminate_flg が 1 の行は学習に入っていない。テストデータは全行を予測している
assert (
    run.frame[[CONFIG.DATE_COL, CONFIG.CAT_COL]]
    .merge(eliminated_keys, on=[CONFIG.DATE_COL, CONFIG.CAT_COL])
    .empty
)
assert len(run.predictions) == n_test_rows
assert run.predictions[CONFIG.PRED_COL].notna().all(), "予測の無いテストの行があります"
print(
    f"OK 3: eliminate_flg が 1 の {len(eliminated_keys)}行は学習に使っていない。"
    f"テストは全 {n_test_rows}行を予測"
)

## 再帰予測の確認

1. **テストのラグ:** テストの再帰予測に使ったラグ・移動平均が、「学習期間の実測値 + テスト期間の予測値（どちらも
   `irregular_flg` が 1 の日は NaN）」から pandas で別に計算した値と一致する（`TEST_PREDICTION = "refit_full"` のとき）。
2. **`irregular_flg` が 1 の日:** その翌日の `Y_lag_1` など、1 の日を指すラグが全部 NaN になっている。
3. **CV のラグ:** 検証期間の実測値を書き換えても、学習側のラグ・移動平均も、検証側の再帰予測に渡す履歴も変わらない。

このほか、再帰予測のたびに、予測する日の実測値が履歴に無いことと、予測の途中で使ったラグが最後の履歴から計算し直した値と
同じことを確かめている（`recursive_predict`）。

表と図は、`CONFIG.DETAIL_TARGET` のテスト期間の最初の部分。`irregular_flg` が 1 の日の予測値（灰色）は、ラグには使わない。

In [ ]:
# 1. テストの再帰予測に使ったラグ・移動平均を、学習期間の実測値と予測値から計算し直す
test_history = run.test_lags[CONFIG.PRED_COL].mask(run.test_lags[CONFIG.IRREGULAR_COL] == 1)
n_irregular_lags = 0
for cat in run.categories:
    part = run.test_lags[run.test_lags[CONFIG.CAT_COL] == cat]
    daily = pd.concat(
        [histories[cat], pd.Series(test_history[part.index].to_numpy(), part[CONFIG.DATE_COL])]
    ).asfreq("D")
    used_cols = [c for c in lag_cols if c in run.features[cat]]
    lag_days = {f"{CONFIG.TARGET_COL}_lag_{k}": k for k in CONFIG.LAGS}
    if CONFIG.TEST_PREDICTION == "refit_full":
        for column in used_cols:
            if column in lag_days:
                values = daily.shift(lag_days[column])
            else:
                window = int(column.rsplit("_", 1)[1])
                values = (
                    daily.shift(1)
                    .rolling(window, min_periods=min(CONFIG.ROLLING_MIN_PERIODS, window))
                    .mean()
                )
            # 特徴量は CONFIG.FEATURE_DTYPE の型（float32）なので、その精度で比べる
            np.testing.assert_allclose(
                part[column].to_numpy(),
                values.reindex(part[CONFIG.DATE_COL]).to_numpy(),
                rtol=1e-5,
                equal_nan=True,
            )
    # 2. irregular_flg が 1 の日を指すラグは NaN
    irregular_dates = part.loc[part[CONFIG.IRREGULAR_COL] == 1, CONFIG.DATE_COL]
    for column in used_cols:
        if column in lag_days:
            shifted = irregular_dates + pd.Timedelta(days=lag_days[column])
            pointed = part.loc[part[CONFIG.DATE_COL].isin(shifted), column]
            assert pointed.isna().all(), f"{cat}: {column} に {CONFIG.IRREGULAR_COL} = 1 の日の値"
            n_irregular_lags += len(pointed)
if CONFIG.TEST_PREDICTION == "refit_full":
    print("OK 1: テストのラグ・移動平均は、実測値と予測値から計算し直した値と一致する（全 Y_cat）")
else:
    print("1. fold_mean では fold ごとに予測値（ラグ）が違うので、計算し直しません")
print(f"OK 2: テストの {CONFIG.IRREGULAR_COL} = 1 の日を指すラグ {n_irregular_lags}個は、全部 NaN")

# 3. 検証期間の実測値を書き換えても、学習側のラグも、再帰予測に渡す履歴も変わらない
for cat in run.categories:
    rows = run.results[cat].rows
    cat_dates = run.frame[CONFIG.DATE_COL].iloc[rows].reset_index(drop=True)
    history = histories[cat]
    cat_folds = run.results[cat].fold_ids
    for fold in np.unique(cat_folds[cat_folds >= 0]):
        valid_idx = np.flatnonzero(cat_folds == fold)
        hidden = fold_hidden(history, cat_dates, valid_idx, config.cv.method)
        valid_days = np.isin(history.index, cat_dates.iloc[valid_idx])
        assert hidden[valid_days].all(), f"{cat} fold {fold}: 検証の日が履歴に残っています"
        pd.testing.assert_series_equal(
            history.mask(valid_days, 1e6).mask(hidden), history.mask(hidden)
        )
print("OK 3: 検証期間の実測値を書き換えても、CV のラグの履歴は変わらない（全 Y_cat・全 fold）")

# CONFIG.DETAIL_TARGET のテスト期間の最初の部分
ensure_japanese_font()
detail_lags = run.test_lags[run.test_lags[CONFIG.CAT_COL] == CONFIG.DETAIL_TARGET].sort_values(
    CONFIG.DATE_COL, kind="stable"
)
recent = histories[CONFIG.DETAIL_TARGET].iloc[-60:]
head = detail_lags.head(120)
is_irregular = head[CONFIG.IRREGULAR_COL] == 1
fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
ax.plot(
    recent.index, recent, "o-", color="black", markersize=3, linewidth=0.8, label="実測値（履歴）"
)
ax.plot(
    head[CONFIG.DATE_COL],
    head[CONFIG.PRED_COL].mask(is_irregular),
    "o-",
    color="C3",
    markersize=3,
    linewidth=0.8,
    label="再帰予測（ラグに使う）",
)
ax.scatter(
    head.loc[is_irregular, CONFIG.DATE_COL],
    head.loc[is_irregular, CONFIG.PRED_COL],
    color="gray",
    marker="x",
    label=f"{CONFIG.IRREGULAR_COL} = 1 の予測（ラグでは NaN）",
)
ax.set_xlabel("日付")
ax.set_ylabel(f"{CONFIG.TARGET_COL}（元の尺度、後処理の前）")
ax.set_title(
    f"{CONFIG.DETAIL_TARGET}: 学習期間の終わりと、テスト期間の最初の {len(head)}日の再帰予測"
)
ax.legend(loc="best", fontsize=10)
nan_days = f"{CONFIG.IRREGULAR_COL} = 1、目的変数が欠損"
if not CONFIG.LAG_USE_ELIMINATED:
    nan_days += f"、{CONFIG.ELIMINATE_COL} = 1"
add_caption(fig, f"線の途切れは NaN の日（{nan_days}）。")
plt.show()
plt.close(fig)
# irregular_flg が 1 の日の前後（ラグが NaN になる様子）
first_irregular = detail_lags.loc[detail_lags[CONFIG.IRREGULAR_COL] == 1, CONFIG.DATE_COL].min()
if pd.isna(first_irregular):
    around = detail_lags.head(6)
else:
    gap = (detail_lags[CONFIG.DATE_COL] - first_irregular).dt.days
    around = detail_lags[gap.between(-2, 4)]
around.set_index(CONFIG.DATE_COL).round(3)

## スケーリングと逆変換の確認

1・2 は、各 Y_cat の各 fold のモデルを捨てる前に確かめている（確かめられなければ学習がエラーで止まる）。

1. **説明変数のスケーリング:** 各 fold のモデルの `GroupScaler` の統計量が、その fold の学習側から pandas で求めた
   値と一致し、学習側を変換すると標準化なら平均 0・標準偏差 1、正規化なら最小値 0・最大値 1 になる。
2. **目的変数のスケーリング:** 各 fold の変換で学習側を変換すると、同じく平均 0・標準偏差 1（最小値 0・最大値 1）
   になり、逆変換で元の値に戻る。学習の表に `eliminate_flg` が 1 の行が無い。
3. **SHAP の逆変換:** 元の尺度の `期待値 + SHAP 値の和` が、元の尺度の OOF 予測と一致する（目的変数の変換が
   一次式で、`OBJECTIVE` が `"poisson"` でないとき）。スケーリングした列の説明変数の値が、スケーリング前の値に
   戻っている。

In [ ]:
# Y_cat ごとに特徴量を選ぶと、スケーリングする列の無い Y_cat があり、結果の文が分かれる
for text in dict.fromkeys(r.checks["スケーリング"] for r in run.results.values()):
    checked = [cat for cat, r in run.results.items() if r.checks["スケーリング"] == text]
    print(f"OK 1・2: {text}（{'全 Y_cat' if len(checked) == len(run.results) else checked}）")
# 学習に使った行（中心・分母を求める行）に eliminate_flg が 1 の行が無い
assert (
    run.frame[[CONFIG.DATE_COL, CONFIG.CAT_COL]]
    .merge(eliminated_keys, on=[CONFIG.DATE_COL, CONFIG.CAT_COL])
    .empty
), "学習に使った行に eliminate_flg が 1 の行があります"
print("OK 2: 中心・分母を求めた学習側に、eliminate_flg が 1 の行は無い")

# 3. SHAP の逆変換
shap_result = run.shap
assert shap_result is not None
oof_pred_all = run.oof[CONFIG.PRED_COL].to_numpy()
if CONFIG.TARGET_TRANSFORM == "log" or CONFIG.OBJECTIVE == "poisson":
    print("3. 対数の尺度の SHAP 値なので、元の尺度の予測との一致は確かめません")
else:
    reconstructed = shap_result.base_values + shap_result.values.sum(axis=1)
    oof_at_rows = oof_pred_all[shap_result.rows]
    # SHAP の計算（木の分解など）は単精度で行うモデルもあるので、
    # 目的変数の尺度に対して小さい差を許す
    tolerance = 1e-4 * float(np.nanmax(np.abs(run.oof[CONFIG.ACTUAL_COL])))
    np.testing.assert_allclose(reconstructed, oof_at_rows, rtol=1e-4, atol=tolerance)
    print(
        "OK 3: 元の尺度の「期待値 + SHAP 値の和」が OOF 予測と一致する"
        f"（{len(shap_result.rows)}行、"
        f"最大の差 {np.max(np.abs(reconstructed - oof_at_rows)):.2e}）"
    )
# ラグ・移動平均は fold ごと（再帰予測）の値なので、学習の表とは比べない
shown_cols = [c for c in scaling_cols if c in shap_result.data.columns and c not in lag_cols]
if shown_cols:
    raw_values = (
        run.frame.iloc[shap_result.rows][shown_cols]
        .astype("float64")
        .replace([np.inf, -np.inf], np.nan)
        .to_numpy()
    )
    # その Y_cat のモデルが使わない特徴量の値は、SHAP の表では欠損になる
    row_cats = run.frame[CONFIG.CAT_COL].to_numpy()[shap_result.rows]
    used = np.column_stack(
        [np.isin(row_cats, [k for k, v in run.features.items() if c in v]) for c in shown_cols]
    )
    raw_values = np.where(used, raw_values, np.nan)
    np.testing.assert_allclose(
        shap_result.data[shown_cols].to_numpy(dtype=np.float64), raw_values, equal_nan=True
    )
    print(f"OK 3: SHAP の説明変数の値（{len(shown_cols)}列）は、スケーリング前の値に戻っている")

## 8. CV の結果

Y_cat ごとのモデルの fold ごとの MAE・RMSE、OOF 全体のスコア、カテゴリごとの誤差（いずれも、検証期間の再帰予測を、スケーリングを
逆変換した元の尺度で、上下限のクリップなどの後処理をかける前のモデルの予測）。fold は Y_cat ごとの CV の fold
（Y_cat が違えば、同じ番号の fold でも検証の期間が違うことがある）。
線形回帰による補正の効果は「12. 後処理1」で見る。
相対MAE は MAE を実測値の平均で割った値。

全行の結果に加えて、`irregular_flg` が 0 の行だけの結果も並べて表示する（表の `（flg=0）`・`_irregular0` の列、
図の右側）。`irregular_flg` が 1 の行はテストで 0 に置き換えるので、モデルの当てはまりは 0 の行だけの誤差で見る。
`irregular_flg` が欠損の行は、0 の行には含めない。

In [ ]:
fold_table = cv_table(run)
normal = irregular_zero_rows(run)
oof_rows = (run.oof["fold"] >= 0).to_numpy()
actual_all = run.oof[CONFIG.ACTUAL_COL].to_numpy()
scores = scores_or_nan(actual_all[oof_rows], oof_pred_all[oof_rows])
normal_scores = scores_or_nan(actual_all[oof_rows & normal], oof_pred_all[oof_rows & normal])
print(
    f"CV: {CONFIG.CV_METHOD}（Y_cat ごと） / 目的変数の変換: {CONFIG.TARGET_TRANSFORM}"
    f" / 説明変数のスケーリング（Y_cat ごと）: {CONFIG.FEATURE_SCALING}"
)
print(f"OOF 全体（{int(oof_rows.sum())}行）: MAE {scores['mae']:.4f} / RMSE {scores['rmse']:.4f}")
print(
    f"OOF {CONFIG.IRREGULAR_COL} = 0 の行だけ（{int((oof_rows & normal).sum())}行）:"
    f" MAE {normal_scores['mae']:.4f} / RMSE {normal_scores['rmse']:.4f}"
)
# Y_cat ごとの fold の誤差の平均 ± 標準偏差
fold_summary = fold_table.groupby(CONFIG.CAT_COL, sort=False)[
    ["MAE", "RMSE", "MAE（flg=0）", "RMSE（flg=0）"]
].agg(["mean", lambda s: s.std(ddof=0)])
fold_summary.columns = [
    f"{name}の{'平均' if stat == 'mean' else '標準偏差'}" for name, stat in fold_summary.columns
]
display(fold_summary.round(4))
display(fold_table.round(4))
ensure_japanese_font()
fig, axes = plt.subplots(1, 2, figsize=(14, 4), constrained_layout=True, sharey=True)
positions = np.arange(len(fold_summary))
for ax, (label, suffix) in zip(
    axes, (("全行", ""), (f"{CONFIG.IRREGULAR_COL} = 0 の行だけ", "（flg=0）")), strict=True
):
    for offset, name in ((-0.2, "MAE"), (0.2, "RMSE")):
        ax.bar(
            positions + offset,
            fold_summary[f"{name}{suffix}の平均"],
            0.4,
            yerr=fold_summary[f"{name}{suffix}の標準偏差"],
            capsize=3,
            label=name,
        )
    ax.set_xticks(positions, fold_summary.index)
    ax.set_ylim(bottom=0)  # 棒グラフは0起点
    ax.set_xlabel(CONFIG.CAT_COL)
    ax.set_title(label)
    ax.legend(loc="upper left")
axes[0].set_ylabel("誤差（元の尺度）")
fig.suptitle(f"Y_cat ごとのモデルの、fold の誤差の平均 ± 標準偏差（{CONFIG.CV_METHOD}）")
plt.show()
plt.close(fig)
print(f"種類ごとの CV の誤差（_irregular0 の列は {CONFIG.IRREGULAR_COL} = 0 の行だけ）:")
summary.round(4)

### 予測の広がり（中央に寄っていないか）

カテゴリごとに、実測と OOF 予測の標準偏差と分位点を比べる（`irregular_flg` が 1 の行は除く）。
**広がりの比**（予測の標準偏差 ÷ 実測の標準偏差）が 1 より小さいほど、予測が中央に寄っている。
`下側の外れ`・`上側の外れ` は、予測が実測の 10% 点を下回る・90% 点を上回る行の割合（中央に寄るほど 0 に近い。
実測と同じ広がりなら 10% 前後）。

In [ ]:
normal_rows = run.frame[CONFIG.IRREGULAR_COL].to_numpy() != 1
spread_source = run.oof[(run.oof["fold"] >= 0).to_numpy() & normal_rows]
rows = []
for category, part in spread_source.groupby(CONFIG.CAT_COL, sort=False):
    actual, pred = part[CONFIG.ACTUAL_COL], part[CONFIG.PRED_COL]
    low, high = actual.quantile(0.1), actual.quantile(0.9)
    rows.append(
        {
            CONFIG.CAT_COL: category,
            "実測の標準偏差": actual.std(),
            "予測の標準偏差": pred.std(),
            "広がりの比": pred.std() / actual.std(),
            "下側の外れ": (pred < low).mean(),
            "上側の外れ": (pred > high).mean(),
        }
    )
spread = pd.DataFrame(rows)
print(
    f"広がりの比の平均: {spread['広がりの比'].mean():.3f}（1 に近いほど、予測が実測と同じ広がり）"
)
spread.round(3)

### 再帰予測と1期先予測の誤差

OOF の再帰予測と、1期先予測（前日までの実測値が分かるとしたラグ）の MAE を比べる。再帰予測の MAE が大きく上回るときは、予測値を
ラグに使うことで誤差が積み重なっている。テストでは実測値が分からないので、テストの誤差の目安になるのは再帰予測の MAE。

何日先の予測かごとの表・図は、`irregular_flg` が 0 の行だけで計算する（何日先かは、履歴の実測値の最後の日から数える。
GroupKFold では、検証の月（続いた月）の最初の日が1日先）。

In [ ]:
evaluated = run.oof[oof_rows]
recursion_all = recursion_table(evaluated, evaluated[CONFIG.CAT_COL])
normal_oof = run.oof[oof_rows & normal]
recursion_normal = recursion_table(normal_oof, normal_oof[CONFIG.CAT_COL])
for label, part in (("全行", evaluated), (f"{CONFIG.IRREGULAR_COL} = 0 の行だけ", normal_oof)):
    actual = part[CONFIG.ACTUAL_COL].to_numpy()
    print(
        f"{label}（{len(part)}行）:"
        f" 再帰予測 MAE {mae_or_nan(actual, part[CONFIG.PRED_COL].to_numpy()):.4f}"
        f" / 1期先予測 MAE {mae_or_nan(actual, part['pred_onestep'].to_numpy()):.4f}"
    )
print("Y_cat ごと（全行）:")
display(recursion_all.round(4))
print(f"Y_cat ごと（{CONFIG.IRREGULAR_COL} = 0 の行だけ）:")
display(recursion_normal.round(4))

# 何日先の予測かごとの MAE
bands = horizon_labels(normal_oof["horizon"]).rename("何日先")
by_horizon = recursion_table(normal_oof, bands)
# 近い順に並べる（行の無い区切りは除く）
by_horizon = by_horizon.loc[[c for c in bands.cat.categories if c in by_horizon.index]]
recursion_dir = out / "recursion"
for name, table in (
    ("mae_by_category", recursion_all),
    ("mae_by_category_irregular0", recursion_normal),
    ("mae_by_horizon_irregular0", by_horizon),
):
    table.to_csv(ensure_parent_dir(recursion_dir / f"{name}.csv"))
ensure_japanese_font()
fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
positions = np.arange(len(by_horizon))
for offset, name in ((-0.2, "MAE 再帰予測"), (0.2, "MAE 1期先予測")):
    ax.bar(positions + offset, by_horizon[name], 0.4, label=name)
ax.set_xticks(positions, [f"{i}\nn={n:,}" for i, n in by_horizon["行数"].items()])
ax.set_ylim(bottom=0)  # 棒グラフは0起点
ax.set_xlabel("何日先の予測か")
ax.set_ylabel("OOF MAE（元の尺度）")
ax.set_title(f"何日先かごとの誤差（{CONFIG.CV_METHOD}、{CONFIG.IRREGULAR_COL} = 0 の行）")
ax.legend(loc="center left", bbox_to_anchor=(1.0, 0.5))
add_caption(fig, "全 Y_cat をまとめた MAE。後処理の前。")
fig.savefig(recursion_dir / "mae_by_horizon.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
if CONFIG.USE_TRACKING and run.run_id is not None:
    import mlflow

    client = mlflow.tracking.MlflowClient()
    onestep_mae = mae_or_nan(
        evaluated[CONFIG.ACTUAL_COL].to_numpy(), evaluated["pred_onestep"].to_numpy()
    )
    client.log_metric(run.run_id, "onestep_oof_mae", onestep_mae)
    for path in sorted(recursion_dir.glob("mae_by_*")):
        client.log_artifact(run.run_id, str(path), artifact_path="recursion")
by_horizon.round(4)

### パラメータの最適化（Optuna）

`CONFIG.N_TRIALS` が 1 以上のときの、Y_cat ごとの最良の CV の MAE（再帰予測、元の尺度）と、`CONFIG.DETAIL_TARGET` の
試行ごとの推移。最適化に使った CV と同じ CV で評価しているので、上の CV の誤差は楽観的になりやすい。

In [ ]:
if run.tuning_trials is None:
    print("Optuna による最適化は行っていません（CONFIG.N_TRIALS = 0）")
else:
    trials = run.tuning_trials
    best = trials.groupby(CONFIG.CAT_COL, sort=False)["value"].agg(
        試行回数="size", 最良の_CV_MAE="min"
    )
    display(best.round(4))
    detail_trials = trials[trials[CONFIG.CAT_COL] == CONFIG.DETAIL_TARGET]
    print(f"{CONFIG.DETAIL_TARGET} のパラメータ: {run.results[CONFIG.DETAIL_TARGET].params}")
    ensure_japanese_font()
    fig, ax = plt.subplots(figsize=(10, 4), constrained_layout=True)
    ax.plot(detail_trials["number"], detail_trials["value"], "o", color="C0", label="各試行")
    ax.plot(
        detail_trials["number"],
        detail_trials["value"].cummin(),
        color="C3",
        label="それまでの最良",
    )
    ax.set_xlabel("試行")
    ax.set_ylabel("CV MAE（元の尺度）")
    ax.set_title(f"{CONFIG.DETAIL_TARGET}: Optuna の最適化の推移")
    ax.legend(loc="upper right")
    plt.show()
    plt.close(fig)

- **カテゴリごとの誤差:** カテゴリごとの CV の相対MAE（左: 全行、右: `irregular_flg` が 0 の行だけ）。
- **学習結果:** カテゴリごとの実測値（黒）と OOF 予測（赤）。`irregular_flg` が 1 の行は、下に外れた点になる。
  各図の題の `flg=0` は、`irregular_flg` が 0 の行だけの CV MAE。
- **テスト結果:** カテゴリごとの 2025年のモデルの再帰予測（後処理の前。正解が無いので実測は無い）。

In [ ]:
figures = out / "figures"
display(Image(filename=str(figures / "summary_mae.png"), width=950))
display(Image(filename=str(figures / "training_overview.png"), width=1000))
display(Image(filename=str(figures / "test_overview.png"), width=1000))

## 9. SHAP によるモデル解釈

Y_cat ごとのモデルを、各 fold の検証側のデータ（ラグ・移動平均は再帰予測に使った値）で説明した SHAP 値（OOF SHAP。Y_cat ごとに
`SHAP_MAX_SAMPLES_PER_CAT` 行まで）を、全 Y_cat でつないだもの。Y_cat は特徴量に使わないので表示されない。
Y_cat ごとに特徴量を選んだときは、その Y_cat で使わない特徴量の SHAP 値を 0 として全体の図に入れる。SHAP 値は、Y_cat ごとのスケーリングを逆変換した
**元の尺度**（`SHAP 値 × 分母`。対数の変換のときだけモデルの尺度）。beeswarm の色（説明変数の値）も、
スケーリング前の値。元の尺度では尺度の大きい Y_cat の SHAP 値が大きく出るので、全体の重要度はその Y_cat の
影響を強く受ける（系列ごとの表は Y_cat ごとの割合なので、尺度に左右されない）。`Y_cat` は系列ID、`irregular_flg` はフラグの特徴量。線形モデルでは系列ID・カテゴリの説明変数がワンホットの列
（`Y_cat_Y1` など）に分かれて表示される（系列ごとの表では1つにまとめる）。線形モデルの SHAP 値は係数から厳密に
計算した値で、`係数 ×（標準化した値 − 学習側の平均）`。

In [ ]:
shap_dir = out / "shap"
display(Image(filename=str(shap_dir / "shap_importance_bar.png"), width=700))
display(Image(filename=str(shap_dir / "shap_beeswarm.png"), width=800))
display(Image(filename=str(figures / "shap_overview.png"), width=1000))

## 10. 残差解析

OOF 予測（再帰予測）の残差（実測 − 予測。元の尺度、後処理の前）を調べる。`irregular_flg` が 1 の行は通常と違う値なので、残差が
大きく出る。そのため、全行の結果に加えて、`irregular_flg` が 0 の行だけの結果も並べて表示する
（残差の要約・誤差分布・影響の大きい行。全行の図・表は `evaluation/`、0 の行だけの図・表は
`evaluation/irregular_flg_0/` に保存する）。

### 残差の要約

全体とカテゴリごとの、残差の平均（偏り）・標準偏差など。

In [ ]:
evaluation_dir = out / EVALUATION_DIR
normal_evaluation_dir = evaluation_dir / "irregular_flg_0"
normal_evaluation_dir.mkdir(parents=True, exist_ok=True)
predicted = run.oof[oof_rows]
normal_predicted = run.oof[oof_rows & normal]
groups = residual_groups(predicted)
normal_groups = residual_groups(normal_predicted)

residual_table = residual_summary_table(predicted)
residual_table.to_csv(evaluation_dir / "residual_summary_by_category.csv", index=False)
normal_residual_table = residual_summary_table(normal_predicted)
normal_residual_table.to_csv(
    normal_evaluation_dir / "residual_summary_by_category.csv", index=False
)
print(f"全行（{len(predicted)}行）:")
display(residual_table.round(4))
print(f"{CONFIG.IRREGULAR_COL} = 0 の行だけ（{len(normal_predicted)}行）:")
display(normal_residual_table.round(4))

# 誤差分布・影響の大きい行の図を、全行と irregular_flg = 0 の行だけで作る
max_points = config.evaluation.max_points
all_figures = save_residual_figures(
    run,
    np.ones(len(run.oof), dtype=bool),
    *combined_influence(run, "influence"),
    evaluation_dir,
    "全行",
    max_points,
)
normal_figures = save_residual_figures(
    run,
    normal,
    *combined_influence(run, "influence_irregular0"),
    normal_evaluation_dir,
    f"{CONFIG.IRREGULAR_COL} = 0 の行だけ",
    max_points,
)

### 線形モデルの係数

線形モデルのときだけ、`CONFIG.DETAIL_TARGET` のモデル（`TEST_PREDICTION = "refit_full"` なら、その Y_cat の全学習データで
学習したモデル。`"fold_mean"` なら最後の fold のモデル）の係数を、絶対値の大きい順に表示する。係数は標準化した後の値に
対するもので、「その説明変数が 1 標準偏差増えると、予測がいくつ変わるか」（モデルの尺度。目的変数・説明変数を
スケーリングした場合は、その後の値に対する係数）。全 Y_cat の係数は `evaluation/linear_coefficients.csv` に保存し、
Y_cat ごとの 0 になった係数の数も表示する。

In [ ]:
# 線形モデルの係数（木のモデルでは表示しない）
if get_model_spec(CONFIG.MODEL).explainer_kind == "linear":
    coef_tables = {
        cat: r.coefficients for cat, r in run.results.items() if r.coefficients is not None
    }
    coef_all = pd.concat(
        [t.assign(**{CONFIG.CAT_COL: cat}) for cat, t in coef_tables.items()], ignore_index=True
    )
    coef_all.to_csv(evaluation_dir / "linear_coefficients.csv", index=False)
    display(
        coef_all.groupby(CONFIG.CAT_COL, sort=False).agg(
            切片=("intercept", "first"),
            係数が0の特徴量=("coef", lambda s: int((s == 0).sum())),
        )
    )
    detail_coef = coef_tables[CONFIG.DETAIL_TARGET]
    print(f"{CONFIG.DETAIL_TARGET} のモデルの係数（上位20）:")
    display(detail_coef.drop(columns="intercept").head(20).round(4))
else:
    print(f"{CONFIG.MODEL} は線形モデルではないため、係数はありません")

### 誤差曲線

`CONFIG.DETAIL_TARGET` のモデルの、各 fold の反復（木の本数）ごとの学習・検証の損失。検証の損失が下がらなくなる所で
early stopping が止まる。学習と検証の差が大きいと過学習。モデルの尺度の値。全 Y_cat の図は
`evaluation/training_history/` に保存する。RandomForest・ExtraTrees・線形モデルには反復ごとの損失が無いので、
学習の推移は表示しない（学習曲線は、このノートブックでは作らない）。

In [ ]:
# 学習の推移は、反復ごとの損失を記録するモデル（LightGBM・XGBoost・CatBoost）だけにある
history_dir = evaluation_dir / "training_history"
for cat, r in run.results.items():
    if r.histories:
        fig = TrainingHistoryDisplay.from_histories(r.histories).figure_
        fig.suptitle(f"{cat}: 学習の推移（fold ごと）")
        add_caption(fig, f"{config.name}: {cat} のモデルの反復ごとの損失（モデルの尺度）")
        save_figure(fig, history_dir / f"training_history_{cat}.png")
detail_history = history_dir / f"training_history_{CONFIG.DETAIL_TARGET}.png"
if detail_history.is_file():
    display(Image(filename=str(detail_history), width=800))
else:
    print(f"{CONFIG.MODEL} には学習の推移（反復ごとの損失）がありません")

### 誤差分布

- **残差の分布:** 山が0からずれていると偏りがある。全体の分布はカテゴリをまとめたものなので、尺度の違いで裾が重く見える。
- **Q-Q プロット:** 点が直線から離れるほど、正規分布から外れている。
- **残差プロット:** 予測値が大きいほど残差が広がるなら、誤差が水準に比例している（対数の変換を試す目安）。
- **カテゴリごとの分布:** 尺度の違いの影響を除いて、カテゴリごとの分布の形を見る。

それぞれ、全行の図の次に `irregular_flg` が 0 の行だけの図を表示する。

In [ ]:
# 全行の図の次に、irregular_flg = 0 の行だけの図を表示する
for name, width in (("residual_distribution", 800), ("qq_plot", 650), ("residual_plot", 800)):
    for label, paths in (
        ("全行", all_figures),
        (f"{CONFIG.IRREGULAR_COL} = 0 の行だけ", normal_figures),
    ):
        if name in paths:
            print(f"{name}: {label}")
            display(Image(filename=str(paths[name]), width=width))
for label, residual_dir, residual_parts in (
    ("全行", evaluation_dir, groups),
    (f"{CONFIG.IRREGULAR_COL} = 0 の行だけ", normal_evaluation_dir, normal_groups),
):
    if not residual_parts:
        continue
    print(f"カテゴリごとの分布: {label}")
    fig, _ = plot_residuals_by_group("distribution", residual_parts)
    fig.savefig(
        residual_dir / "residual_distribution_by_category.png", dpi=150, bbox_inches="tight"
    )
    plt.show()
    plt.close(fig)

### 影響の大きい行

Leverage と Cook's distance。各 Y_cat の各 fold のモデルの入力で計算し、全 Y_cat をつないで描く。
線形回帰の診断量なので、木のモデルでは影響の大きい行の目安として見る。
全行の図の次に、`irregular_flg` が 0 の行だけで計算し直した図を表示する。

In [ ]:
for label, paths in (
    ("全行", all_figures),
    (f"{CONFIG.IRREGULAR_COL} = 0 の行だけ", normal_figures),
):
    if "leverage_cooks_distance" in paths:
        print(label)
        display(Image(filename=str(paths["leverage_cooks_distance"]), width=950))

## 11. 1つのカテゴリの詳細（`CONFIG.DETAIL_TARGET`）

残差の推移と自己相関（ACF / PACF）。残差に自己相関が残っていれば、モデルが取りこぼした時間の構造がある。

In [ ]:
ensure_japanese_font()
detail = predicted[predicted[CONFIG.CAT_COL] == CONFIG.DETAIL_TARGET].sort_values(
    CONFIG.DATE_COL, kind="stable"
)
residuals = detail[CONFIG.ACTUAL_COL] - detail[CONFIG.PRED_COL]
fig, ax = plt.subplots(figsize=(12, 4), constrained_layout=True)
ax.scatter(detail[CONFIG.DATE_COL], residuals, s=6, color="C0")
ax.axhline(0, color="gray", linewidth=0.8)
ax.set_title(f"{CONFIG.DETAIL_TARGET}: 残差の推移（実測 − OOF 予測）")
ax.set_xlabel("日付")
ax.set_ylabel(f"残差（{CONFIG.TARGET_COL} の元の尺度）")
add_caption(fig, f"n={len(detail):,}。eliminate_flg が 1 の行と目的変数が欠損の行は除く。")
plt.show()
plt.close(fig)
correlogram = ResidualCorrelogramDisplay.from_residuals(
    {CONFIG.DETAIL_TARGET: residuals.to_numpy()}
)
plt.show()
plt.close(correlogram.figure_)
run.predictions[run.predictions[CONFIG.CAT_COL] == CONFIG.DETAIL_TARGET].head()

## 12. 後処理1: Y_cat ごとの線形回帰による予測値の補正

Y_cat ごとに、OOF の再帰予測（元の尺度）を説明変数、実績値を目的変数とする線形回帰（最小二乗法）で
`実績値 = 切片 + 傾き × 予測値` を求め、予測値を `切片 + 傾き × 予測値` に補正する。傾きが 1 より大きければ、
予測が中央に寄っていて、補正で広がる（1 より小さければ縮む）。

- **対象の行:** `irregular_flg` が 1 でない行（欠損の行は含む）。1 の行は通常と違う値で、テストでは 0 に置き換えるので、
  式を求めるのにも補正にも使わない（全行の MAE では、補正の前の予測のまま計算する）。
- **ラグとの関係:** 補正は、再帰予測がすべて終わった後の予測値にかける。補正した値はラグ・移動平均に使わない
  （補正すると、以降の予測そのものが変わってしまうため）。
- **テストに使う式:** その Y_cat の全部の OOF 予測から求める（表の「傾き」「切片」）。
- **補正後の MAE:** その行の fold **以外** の行で求めた式で補正した値で計算する。評価する行の実績値を式を求めるのに
  使わないので、補正の効果を公平に見られる。「補正後（参考）」は、評価する行も含めて求めた式による値で、楽観的になる。
- **補正する Y_cat:** `CONFIG.CORRECTION_CATS`。最小二乗法は二乗誤差を小さくするので、MAE は小さくならないことがある。
  既定の `"improved"` では、補正後の MAE（全行）が補正前より小さくなる Y_cat だけを補正し、他の Y_cat は
  モデルの予測のまま使う（下では「※補正しない」と表示する）。全部を補正するときは `None`、Y_cat を選ぶときは
  リストを指定する。

In [ ]:
irregular_values = run.frame[CONFIG.IRREGULAR_COL].to_numpy()
corrected_oof, corrections = correct_oof(run.oof, irregular_values)
# MAE の表: 全行（検証に入った行）と、irregular_flg が 0 の行だけ
mae_all = mae_change_table(corrected_oof[oof_rows], run.categories)
mae_normal = mae_change_table(corrected_oof[oof_rows & normal], run.categories)
applied_cats = corrected_categories(mae_all)
applied_corrections = {cat: corrections[cat] for cat in applied_cats}

# 全体の MAE は、補正する Y_cat だけを補正した値で計算する
applied_rows = corrected_oof[CONFIG.CAT_COL].isin(applied_cats)
corrected_pred = (
    corrected_oof["pred_corrected"].where(applied_rows, corrected_oof[CONFIG.PRED_COL]).to_numpy()
)
print(f"補正する Y_cat（CORRECTION_CATS = {CONFIG.CORRECTION_CATS!r}）: {applied_cats or 'なし'}")
total_mae = {}
for label, key, part in (
    ("全行", "", oof_rows),
    (f"{CONFIG.IRREGULAR_COL} = 0 の行だけ", "_irregular0", oof_rows & normal),
):
    before = mae_or_nan(actual_all[part], oof_pred_all[part])
    after = mae_or_nan(actual_all[part], corrected_pred[part])
    total_mae[f"oof_mae_before_correction{key}"] = before
    total_mae[f"oof_mae_after_correction{key}"] = after
    print(f"全体・{label}（{int(part.sum())}行）: {mae_change_text(before, after)}")
for label, table in (("全行", mae_all), (f"{CONFIG.IRREGULAR_COL} = 0 の行だけ", mae_normal)):
    print(f"Y_cat ごと・{label}:")
    for cat, row in table.iterrows():
        note = "" if cat in applied_cats else " ※補正しない（補正した場合の値）"
        print(f"  {cat}: {mae_change_text(row['MAE 補正前'], row['MAE 補正後'])}{note}")
worse = [cat for cat in applied_cats if mae_all.loc[cat, "改善量"] <= 0]
if worse:
    print(
        f"注意: 補正しても CV の MAE が小さくならない Y_cat を補正しています: {worse}"
        '（除くときは CONFIG.CORRECTION_CATS = "improved"）'
    )
unknown_effect = [cat for cat in run.categories if np.isnan(mae_all.loc[cat, "MAE 補正後"])]
if unknown_effect:
    print(f"注意: 補正の効果を確かめられない Y_cat があります: {unknown_effect}")
negative = [cat for cat in applied_cats if corrections[cat].slope <= 0]
if negative:
    print(f"注意: 傾きが 0 以下の Y_cat があります（予測が実績と無関係か逆向き）: {negative}")

equations = pd.DataFrame(
    {
        "傾き": {cat: corrections[cat].slope for cat in run.categories},
        "切片": {cat: corrections[cat].intercept for cat in run.categories},
        "式を求めた行数": {cat: corrections[cat].n_rows for cat in run.categories},
        "補正する": {cat: cat in applied_cats for cat in run.categories},
    }
).rename_axis(CONFIG.CAT_COL)
print("補正の式と、全行の MAE:")
display(equations.join(mae_all).round(4))
print(f"{CONFIG.IRREGULAR_COL} = 0 の行だけの MAE:")
display(mae_normal.round(4))

In [ ]:
# 補正の確認
# 1. 補正後の値は、その fold の実績値を使っていない（fold 0 の実績値を書き換えても変わらない）
first_fold = (run.oof["fold"] == 0).to_numpy()
rewritten = run.oof.assign(**{CONFIG.ACTUAL_COL: run.oof[CONFIG.ACTUAL_COL].mask(first_fold, 1e6)})
rewritten_oof, _ = correct_oof(rewritten, irregular_values)
np.testing.assert_array_equal(
    rewritten_oof.loc[first_fold, "pred_corrected"].to_numpy(),
    corrected_oof.loc[first_fold, "pred_corrected"].to_numpy(),
)
print(f"OK 1: fold 0 の実績値を書き換えても、fold 0（{int(first_fold.sum())}行）の補正後の値は同じ")

# 2. テストに使う式は最小二乗法の解（NumPy の polyfit と同じ予測になる）
correctable = oof_rows & (irregular_values != 1)
for cat in run.categories:
    part = run.oof[correctable & (run.oof[CONFIG.CAT_COL] == cat).to_numpy()]
    x, y = part[CONFIG.PRED_COL].to_numpy(), part[CONFIG.ACTUAL_COL].to_numpy()
    if len(x) < 2 or np.ptp(x) <= 1e-6 * max(1.0, abs(x.mean())):
        continue  # 傾きが決まらない Y_cat は平均のずれだけを補正している
    np.testing.assert_allclose(
        corrections[cat].apply(x), np.polyval(np.polyfit(x, y, 1), x), rtol=1e-6, atol=1e-6
    )
print("OK 2: 補正の式は、最小二乗法の解と一致する")

# 3. irregular_flg が 1 の行は補正していない
untouched = irregular_values == 1
pd.testing.assert_series_equal(
    corrected_oof.loc[untouched, "pred_corrected"],
    corrected_oof.loc[untouched, CONFIG.PRED_COL],
    check_names=False,
)
print(f"OK 3: {CONFIG.IRREGULAR_COL} = 1 の {int(untouched.sum())}行は補正していない")
del rewritten, rewritten_oof

# 補正の式・MAE・図を保存して表示する
correction_table = equations.join(mae_all).join(
    mae_normal.drop(columns="MAE 補正後（参考）").add_suffix("（flg=0）")
)
correction_figures = save_correction_outputs(
    run,
    corrected_oof,
    correction_table,
    total_mae,
    {
        "mae_before_after": plot_mae_change(
            {"全行": mae_all, f"{CONFIG.IRREGULAR_COL} = 0 の行だけ": mae_normal}, config.name
        ),
        "correction_lines": plot_correction(run.oof, irregular_values, corrections, config.name),
    },
    CONFIG.USE_TRACKING,
)
display(Image(filename=str(correction_figures["mae_before_after"]), width=1000))
display(Image(filename=str(correction_figures["correction_lines"]), width=1100))

## 13. 後処理2: 上下限のクリップと、`irregular_flg` の行を0にする処理

再帰予測がすべて終わった後に、次の順で後処理をかける（モデル・CV のスコアには含めない。ラグにも使わない）。

1. 「12. 後処理1」の式で、補正する Y_cat の予測値を補正する（`irregular_flg` が 1 の行を除く）。
2. カテゴリごとの上下限（`CONFIG.TARGET_BOUNDS`。無いカテゴリは `CONFIG.DEFAULT_TARGET_BOUNDS`）で、予測値をクリップする。
3. テストデータの `irregular_flg` が 1 の行は、予測値を強制的に 0 にする（上下限より優先する）。

下の表は、カテゴリごとの上下限と、補正した行数・クリップした行数・0 にした行数、テスト予測の平均（`irregular_flg` が
1 の行を除く。補正の前と後）。

In [ ]:
bounds = target_bounds(cats)
final = postprocess(run.predictions, test_flags, bounds, applied_corrections)
corrected_rows = final[CONFIG.CAT_COL].isin(applied_cats) & ~final["zeroed"]
counts = (
    final.assign(corrected=corrected_rows)
    .groupby(CONFIG.CAT_COL, sort=False)
    .agg(
        **{
            "行数": (CONFIG.PRED_COL, "size"),
            "補正した行数": ("corrected", "sum"),
            "クリップした行数": ("clipped", "sum"),
            "0にした行数": ("zeroed", "sum"),
        }
    )
)
test_means = (
    final[~final["zeroed"]]
    .groupby(CONFIG.CAT_COL, sort=False)[["pred_model", "pred_corrected"]]
    .mean()
    .set_axis(["予測の平均（補正前）", "予測の平均（補正後）"], axis=1)
)
display(bounds.set_index(CONFIG.CAT_COL).join(counts).join(test_means).loc[cats].round(4))
# 後処理が決まりどおりか確かめる
for cat, correction in corrections.items():
    in_cat = (final[CONFIG.CAT_COL] == cat) & ~final["zeroed"]
    model_values = final.loc[in_cat, "pred_model"].to_numpy()
    np.testing.assert_allclose(
        final.loc[in_cat, "pred_corrected"].to_numpy(),
        correction.apply(model_values) if cat in applied_cats else model_values,
    )
assert (final.loc[final["zeroed"], CONFIG.PRED_COL] == 0).all()
inside = final.loc[~final["zeroed"]].merge(bounds, on=CONFIG.CAT_COL)
assert (inside[CONFIG.PRED_COL] >= inside["lower"].fillna(-np.inf)).all()
assert (inside[CONFIG.PRED_COL] <= inside["upper"].fillna(np.inf)).all()
print(
    f"OK: 補正した {int(corrected_rows.sum())}行は式のとおり、"
    f"{CONFIG.IRREGULAR_COL} = 1 の {int(final['zeroed'].sum())}行は 0、"
    "それ以外は上下限の範囲内"
)

## 14. テストデータの予測の出力

後処理（補正・クリップなど）の後のテスト予測を `outputs/predictions/` に CSV で出力する。列は `ID`（0から始まる連番）、`date`（日付）、
`Y_cat`（目的変数のカテゴリ）、`pred`（予測値。元の尺度・後処理の後）の順で、行は目的変数のカテゴリ、日付の順に昇順。
カテゴリは番号の順（Y1, Y2, …, Y10）に並べる。テストデータの全行を含む。

In [ ]:
prediction_table = test_prediction_table(final, cats)
assert len(prediction_table) == n_test_rows, "テストデータの全行がありません"
assert not prediction_table.duplicated([CONFIG.DATE_COL, CONFIG.CAT_COL]).any(), (
    "予測の行が重複しています"
)
prediction_path = ensure_parent_dir(
    outputs_dir() / CONFIG.PREDICTIONS_DIR / f"{config.name}_{stamp}.csv"
)
prediction_table.to_csv(prediction_path, index=False, date_format="%Y-%m-%d")
print(f"テスト予測: {shown(prediction_path)}（{len(prediction_table)}行）")
display(prediction_table.head())
prediction_table.tail()

## まとめ

- 目的変数のラグ・移動平均を特徴量に加え、テスト期間を再帰予測（1日ずつ予測し、予測値を次の日以降のラグに使う）した。
- `irregular_flg` が 1 の日は、学習期間の実測値も予測期間の予測値も NaN に上書きしてからラグ・移動平均を作った
  （「4. 前処理のリークの確認」「再帰予測の確認」で確かめた）。
- CV でも、検証期間の実測値を使わずに再帰予測して評価した（既定は時系列 CV）。前日までの実測値が分かるとした
  1期先予測の誤差も並べて、再帰による誤差の積み重なりを見られるようにした。
- 015 と同じく、Y_cat ごとの個別モデル・特徴量の選択・スケーリングと逆変換・線形回帰による補正・上下限のクリップ・
  `irregular_flg` が 1 の行を 0 にする後処理を行い、`outputs/predictions/` に CSV で出力した。

**再帰予測の注意点**

- ラグを加えると必ず良くなるわけではない。目的変数に自己相関が無ければ、ラグは役に立たない（ダミーデータは、自己相関が
  ほとんど無いように作ってある）。015 と比べるときは、`CV_METHOD`（と `CUTOFFS`）をそろえる。
- 再帰予測では、先の日ほど予測値だけからラグを作るので、誤差が積み重なる。学習では実測値のラグを使うので、
  モデルがラグに頼りすぎると、予測の誤差がそのまま次の日に伝わる。
- 既定の時系列 CV は、検証に入らない行（最初の境目より前）があるので、015 の既定（GroupKFold）の CV の誤差とは
  比べられない。検証の長さは `CUTOFFS` によるので、テスト期間の長さに近い検証期間を入れる。
- GroupKFold では、検証の月の後の学習側の行で、検証の月を指すラグが NaN になる（検証の実測値を学習に使わないため）。
- early stopping は、検証期間の1期先予測の損失で止める。再帰予測の誤差が最小になる木の本数とは違うことがある。
- `eliminate_flg` が 1 の行は学習には使わないが、目的変数の実測値はラグ・移動平均に使う（`LAG_USE_ELIMINATED`）。
- 検証期間のうち、`eliminate_flg` が 1 の日と目的変数が欠損の日は、CV では予測しないので履歴も NaN になる
  （テストでは、`irregular_flg` が 1 の日を除く全部の日の予測値をラグに使う）。
- ラグ・移動平均は日付で数える（1行 = 1日）。休業日などで行の無い日は NaN になるので、営業日ごとのデータでは、
  ラグの日数の意味（7日前 = 同じ曜日）に注意する。
- `irregular_flg` が欠損の日は、1 と分からないので、値をそのままラグに使う。
- ラグ・移動平均は元の尺度の値。線形モデルでは、標準化と欠損の補完（学習側の中央値）がモデルの Pipeline の中で行われる。
- 履歴に加える予測値は、補正・上下限のクリップの前の値。予測が大きく外れ続ける（発散する）ときは、ラグを減らすか除く。
- 1日ずつ予測するので、015 より時間がかかる（モデルの予測の呼び出しが、Y_cat・fold ごとに日数分だけ増える）。
  Optuna を使うときは、試行ごとに再帰予測するので特に時間がかかる。

**015 と同じ注意点**

- 補正の式は最小二乗法（二乗誤差を小さくする）なので、MAE は小さくならないことがある。特に外れ値のある Y_cat では、
  式が外れ値に引かれる。そこで、既定（`"improved"`）では悪化する Y_cat を補正しない。ただし、同じ CV の MAE で
  補正する Y_cat を選ぶので、選んだ後の全体の MAE は少し楽観的になる。
- 補正の式は CV の fold のモデルの予測から求め、テストでは `TEST_PREDICTION` のモデル（`refit_full` なら学習データ全体で
  学習し直したモデル）の予測に使う。モデルが少し違うので、テストでの効果は CV での効果と同じとは限らない。
- 補正後の MAE は、ある fold を他の fold（後の期間を含む）の行で求めた式で補正した値。他の fold の OOF 予測のモデルは、
  評価する fold の行も学習に使っていることがあるので、わずかに楽観的になることがある。また、テストは学習データより
  後の期間なので、実績値の水準や傾向が変わると、学習期間で求めた式が合わなくなることがある。
- CV の結果・SHAP・残差解析（8〜11）は、補正の前のモデルの予測で計算している。補正後の MAE は、上下限のクリップと
  0 にする処理の前の値。
- `SAVE_MODELS` で保存するモデルに補正・再帰予測は含まれない。補正の式は `correction/correction_by_category.csv`、ラグの
  指定は保存した辞書の `lag_features` にある。
- テストの正解が無いので、テストの誤差は計算していない。評価は CV だけで行っている。
- 1つのモデルが学習する行は、その Y_cat の行だけ。行の少ない Y_cat ではモデルが不安定になりやすく、全 Y_cat に
  共通の傾向も使えない。
- `fold` の番号は Y_cat ごとの CV の fold。Y_cat が違えば、同じ番号の fold でも検証の期間が違うことがある。
- モデルはメモリに残さない。後でモデルを使う（別のデータを予測する）ときは `SAVE_MODELS = True` にして保存する。
- `FEATURE_DTYPE = "float32"` では説明変数・ラグ・移動平均の有効桁が約7桁になる。桁の大きい値・細かい差が意味を持つ
  ときは、`None`（float64）にする。
- 表の「メモリ」は psutil で測ったプロセスのメモリ（RSS）で、psutil が無い環境では NaN。「学習中の最大」は、
  Y_cat の区切りなどで測った時点の最大で、本当の最大（fold の学習の最中）より小さいことがある。
- 学習の推移（誤差曲線）と線形モデルの係数は、モデルの尺度（スケーリングの後）。残差の表・図と SHAP は元の尺度
  （対数の変換のときだけ SHAP もモデルの尺度）。